# Analysis pipeline for the reported study

This notebook implements the downstream analyses reported in the manuscript for the fixed-k=50 TCGA-BRCA study.

## Scope

The analysis is organised around the seven evidence levels used in the paper:

1. predictive performance across all 15 non-empty subsets of four omics layers and three classifier families;
2. layer contribution by leave-one-omics-out ablation and exact performance Shapley values;
3. panel sufficiency using a prespecified non-inferiority margin of 0.03;
4. model-relative predictive complementarity using the normalised index \(R_{\mathrm{pred}}\);
5. prediction-level partial information decomposition using BROJA and Williams–Beer \(I_{\min}\);
6. gene-level biological plausibility using attribution rankings and PAM50 overlap;
7. external reproducibility in METABRIC, including importance concordance and direct transfer under two expression-processing arms.

The reported predictive analysis uses 677 TCGA-BRCA patients, 5-fold stratified cross-validation repeated 10 times, shared partitions across panels and classifiers, fixed a priori classifier hyperparameters, and no hyperparameter search or nested cross-validation.

The performance, LOO, Shapley, panel-sufficiency and integration-gain summaries are derived from the recorded fold-level results without refitting models. Prediction-level PID is the explicit exception: the four single-layer models are refitted to reconstruct patient-level out-of-fold predictions, and those reconstructions are accepted only after reproducing the recorded per-fold metrics within the prespecified tolerance.

The notebook may contain optional compatibility code for alternative dimensional regimes, but only the fixed-k=50 analysis is part of the reported study.


**Environment.** Sets the data directory; no analysis is performed here.

In [ ]:
import os
os.environ["REGIMES_TO_ANALYZE"] = "k50"

## 1. Predictive performance, LOO contribution and performance Shapley


**Levels 1-3.** Reads the recorded fold-level results and derives, without refitting any model: panel performance, leave-one-omics-out contributions, exact performance Shapley values, integration gains, and the minimal panel satisfying the non-inferiority criterion.

In [ ]:

"""
===============================================================================
ANALYSIS PIPELINE FOR THE FIXED-PARAMETER RESULTS
===============================================================================

No preprocessing is rerun for the recorded performance summaries, and no hyperparameter search or nested cross-validation is part of the reported analysis.

The script reads directly :
    results/model_evaluation/panel_fold_results.csv

Default directories :
    k50  -> runs/2026-07-26/

It produces :
- performance of the 15 panels ;
- results fold et repetition ;
- IC95 % ;
- multi-omics gains against the best constituent single-omics panel ;
- paired per-repetition tests and BH-FDR ;
- rankings and cross-classifier stability ;
- leave-one-omics-out ;
- exact performance Shapley values ;
- layer frequency among the best panels ;
- minimal non-inferior panel ;
- report.json.

Analyses requiring patient-level predictions (Rpred/PID) are
run only if a compatible out-of-fold prediction file is found.
The script does not create predictions and does not refit models.

USAGE
    Set TCGA_BRCA_DATA_DIR (or K50_RUN_DIR) to the directory holding the
    recorded run, then execute the cell. Only the fixed-k=50 regime is
    analysed.
"""

import os
import json
import math
import hashlib
import warnings
from pathlib import Path
from itertools import combinations
from collections import Counter
from typing import Dict, List, Tuple, Iterable, Any, Optional

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# =============================================================================
# 0. CONFIGURATION AND PORTABLE PATHS
# =============================================================================

try:
    from google.colab import drive
    if not os.path.exists(str(DATA_DIR)):
        drive.mount("/content/drive")
    DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

REGIME_PATHS = {
    "k50": Path(
        os.environ.get(
            "K50_RUN_DIR",
            str(DATA_DIR / "runs" / "2026-07-26"),
        )
    ),
}

requested = os.environ.get("REGIMES_TO_ANALYZE", "k50")
REGIMES_TO_ANALYZE = [
    item.strip() for item in requested.split(",") if item.strip()
]

unknown = [name for name in REGIMES_TO_ANALYZE if name not in REGIME_PATHS]
if unknown:
    raise ValueError(f"Unknown regimes:  {unknown}")

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {
    "mutations": "Mutations",
    "cnv": "CNV",
    "mrna": "mRNA",
    "rppa": "RPPA",
}
CLASSIFIERS = ["RF", "XGB", "SVM"]
PRIMARY_METRIC = "balanced_accuracy"
SEED = 42

CONFIG = {
    "primary_metric": PRIMARY_METRIC,
    "bootstrap_replicates": 10000,
    "fdr_alpha": 0.05,
    "noninferiority_margin": 0.03,
    "top_k_panels_for_presence": 3,
}

# =============================================================================
# 1. OUTILS
# =============================================================================

def log(message: str = "") -> None:
    print(message)


def stable_seed(*parts: Any) -> int:
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def canonical_panel(panel: Iterable[str]) -> Tuple[str, ...]:
    order = {layer: idx for idx, layer in enumerate(LAYERS)}
    return tuple(sorted(tuple(panel), key=lambda x: order[x]))


PANELS: List[Tuple[str, ...]] = []
for size in range(1, len(LAYERS) + 1):
    PANELS.extend(
        canonical_panel(panel)
        for panel in combinations(LAYERS, size)
    )

SINGLES = [panel for panel in PANELS if len(panel) == 1]
PAIRS = [panel for panel in PANELS if len(panel) == 2]
TRIPLETS = [panel for panel in PANELS if len(panel) == 3]
FULL_PANEL = canonical_panel(LAYERS)


def panel_key(panel: Tuple[str, ...]) -> str:
    return "+".join(panel)


def panel_label(panel: Tuple[str, ...]) -> str:
    return "+".join(SHORT[layer] for layer in panel)


PANEL_BY_KEY = {panel_key(panel): panel for panel in PANELS}
EXPECTED_PANEL_KEYS = set(PANEL_BY_KEY)


def bootstrap_mean_ci(
    values: np.ndarray,
    n_boot: int,
    seed: int,
) -> Tuple[float, float]:
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    indices = rng.integers(
        0,
        len(values),
        size=(n_boot, len(values)),
    )
    means = values[indices].mean(axis=1)

    return (
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )


def exact_sign_flip_pvalue(differences: np.ndarray) -> float:
    """
    Exact two-sided test on the paired per-repetition differences.
    Pour n=10, 2^10=1024 permutations : calcul exhaustif possible.
    """
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]

    if len(differences) == 0:
        return np.nan

    observed = abs(float(differences.mean()))
    n = len(differences)

    null_means = np.empty(2 ** n, dtype=float)

    for mask in range(2 ** n):
        signs = np.array([
            1.0 if ((mask >> bit) & 1) else -1.0
            for bit in range(n)
        ])
        null_means[mask] = abs(float(np.mean(signs * differences)))

    return float(
        np.mean(null_means >= observed - 1e-12)
    )


def normalize_panel_name(value: Any) -> str:
    text = str(value).strip()

    aliases = {
        "mutation": "mutations",
        "mut": "mutations",
        "muts": "mutations",
        "copy_number": "cnv",
        "copy-number": "cnv",
        "rna": "mrna",
        "mrna_expression": "mrna",
        "protein": "rppa",
        "proteomics": "rppa",
        "baseline": "EMPTY",
        "empty": "EMPTY",
    }

    if text.upper() == "EMPTY":
        return "EMPTY"

    separators = ["|", ",", ";", "__", " / ", "/"]
    for separator in separators:
        text = text.replace(separator, "+")

    tokens = [
        token.strip().lower()
        for token in text.split("+")
        if token.strip()
    ]
    tokens = [aliases.get(token, token) for token in tokens]

    if not tokens:
        return "EMPTY"

    unknown_tokens = [token for token in tokens if token not in LAYERS]
    if unknown_tokens:
        raise ValueError(
            f"Panel non reconnu : {value!r}; tokens inconnus={unknown_tokens}"
        )

    return panel_key(canonical_panel(tokens))


def normalize_classifier(value: Any) -> str:
    text = str(value).strip().upper()
    aliases = {
        "RANDOMFOREST": "RF",
        "RANDOM_FOREST": "RF",
        "RANDOM FOREST": "RF",
        "XGBOOST": "XGB",
        "SVC": "SVM",
    }
    return aliases.get(text, text)


def find_first_existing(paths: List[Path]) -> Optional[Path]:
    for path in paths:
        if path.exists():
            return path
    return None


def detect_column(
    frame: pd.DataFrame,
    candidates: List[str],
    required: bool = True,
) -> Optional[str]:
    lower_map = {column.lower(): column for column in frame.columns}

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    if required:
        raise RuntimeError(
            f"No column found among {candidates}. "
            f"Colonnes disponibles : {list(frame.columns)}"
        )

    return None


# =============================================================================
# 2. CHARGEMENT ET NORMALISATION DE panel_fold_results.csv
# =============================================================================

def load_fixed_results(regime: str, run_dir: Path) -> pd.DataFrame:
    candidates = [
        run_dir / "results" / "model_evaluation" / "panel_fold_results.csv",
        run_dir / "model_evaluation" / "panel_fold_results.csv",
        run_dir / "results" / "panel_fold_results.csv",
    ]

    path = find_first_existing(candidates)

    if path is None:
        raise FileNotFoundError(
            f"panel_fold_results.csv introuvable pour {regime}. "
            f"Paths tried:  {[str(x) for x in candidates]}"
        )

    raw = pd.read_csv(path)

    classifier_col = detect_column(
        raw,
        ["classifier", "model", "classifier_name"],
    )
    split_col = detect_column(
        raw,
        ["split", "outer_split", "split_index"],
    )
    repeat_col = detect_column(
        raw,
        ["repeat", "repeat_index", "cv_repeat"],
    )
    fold_col = detect_column(
        raw,
        ["fold", "fold_index", "cv_fold"],
    )
    panel_col = detect_column(
        raw,
        ["panel", "panel_key", "combination"],
    )
    panel_label_col = detect_column(
        raw,
        ["panel_label", "panel_name", "combination_label"],
        required=False,
    )
    panel_size_col = detect_column(
        raw,
        ["panel_size", "n_omics", "subset_size"],
        required=False,
    )
    ba_col = detect_column(
        raw,
        ["balanced_accuracy", "balanced_acc", "ba"],
    )
    f1_col = detect_column(
        raw,
        ["macro_f1", "f1_macro", "macro-f1"],
    )
    accuracy_col = detect_column(
        raw,
        ["accuracy", "acc"],
    )

    normalized = pd.DataFrame({
        "regime": regime,
        "classifier": raw[classifier_col].map(normalize_classifier),
        "split": pd.to_numeric(raw[split_col], errors="raise").astype(int),
        "repeat": pd.to_numeric(raw[repeat_col], errors="raise").astype(int),
        "fold": pd.to_numeric(raw[fold_col], errors="raise").astype(int),
        "panel": raw[panel_col].map(normalize_panel_name),
        "balanced_accuracy": pd.to_numeric(raw[ba_col], errors="raise"),
        "macro_f1": pd.to_numeric(raw[f1_col], errors="raise"),
        "accuracy": pd.to_numeric(raw[accuracy_col], errors="raise"),
    })

    if panel_label_col is not None:
        normalized["panel_label_source"] = raw[panel_label_col].astype(str)

    normalized["panel_size"] = normalized["panel"].map(
        lambda key: 0 if key == "EMPTY" else len(PANEL_BY_KEY[key])
    )
    normalized["panel_label"] = normalized["panel"].map(
        lambda key: (
            "Baseline"
            if key == "EMPTY"
            else panel_label(PANEL_BY_KEY[key])
        )
    )

    if panel_size_col is not None:
        source_sizes = pd.to_numeric(
            raw[panel_size_col],
            errors="coerce",
        )
        inconsistent = (
            source_sizes.notna()
            & (source_sizes.astype("Int64") != normalized["panel_size"])
        )
        if inconsistent.any():
            raise RuntimeError(
                f"{regime}: inconsistency entre panel_size source et panel."
            )

    allowed_classifiers = set(CLASSIFIERS)
    invalid_classifiers = (
        set(normalized["classifier"]) - allowed_classifiers
    )
    if invalid_classifiers:
        raise RuntimeError(
            f"{regime}: classifieurs inconnus : {invalid_classifiers}"
        )

    # Baseline allowed in the file but excluded from the 15 panels.
    model_rows = normalized[normalized["panel"] != "EMPTY"].copy()

    if model_rows.duplicated(["split", "classifier", "panel"]).any():
        duplicates = model_rows[
            model_rows.duplicated(
                ["split", "classifier", "panel"],
                keep=False,
            )
        ].head()
        raise RuntimeError(
            f"{regime}: units duplicated.\n{duplicates}"
        )

    splits = sorted(model_rows["split"].unique())
    expected_units = len(splits) * len(CLASSIFIERS) * len(PANELS)

    if len(model_rows) != expected_units:
        raise RuntimeError(
            f"{regime}: results incomplets : "
            f"{len(model_rows)}/{expected_units} rows pour les 15 panels."
        )

    for split_index in splits:
        for classifier in CLASSIFIERS:
            actual = set(
                model_rows.loc[
                    (model_rows["split"] == split_index)
                    & (model_rows["classifier"] == classifier),
                    "panel",
                ]
            )
            if actual != EXPECTED_PANEL_KEYS:
                raise RuntimeError(
                    f"{regime}: panels incomplets pour "
                    f"{classifier}, split={split_index}. "
                    f"Manquants={sorted(EXPECTED_PANEL_KEYS - actual)}"
                )

    if model_rows["repeat"].nunique() != 10:
        raise RuntimeError(
            f"{regime}: 10 repetitions expected,  "
            f"obtenu={model_rows['repeat'].nunique()}"
        )

    log(
        f"{regime}: {path} loaded — "
        f"{len(model_rows)} rows, {len(splits)} splits."
    )

    return normalized


regime_frames = []

for regime in REGIMES_TO_ANALYZE:
    regime_frames.append(
        load_fixed_results(regime, REGIME_PATHS[regime])
    )

all_fold_results = pd.concat(regime_frames, ignore_index=True)

COMPARISON_DIR = DATA_DIR / "runs" / "comparison_k50_fixed_models"
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

atomic_csv(
    all_fold_results,
    COMPARISON_DIR / "normalized_panel_fold_results.csv",
)

# =============================================================================
# 3. AGGREGATION FOLD -> REPETITION -> SUMMARY
# =============================================================================

model_fold_results = all_fold_results[
    all_fold_results["panel"] != "EMPTY"
].copy()

repeat_results = (
    model_fold_results
    .groupby(
        [
            "regime",
            "classifier",
            "repeat",
            "panel",
            "panel_label",
            "panel_size",
        ],
        as_index=False,
    )[["balanced_accuracy", "macro_f1", "accuracy"]]
    .mean()
)

atomic_csv(
    repeat_results,
    COMPARISON_DIR / "performance_repeat.csv",
)

summary_rows = []

for keys, group in repeat_results.groupby(
    [
        "regime",
        "classifier",
        "panel",
        "panel_label",
        "panel_size",
    ]
):
    ba = group["balanced_accuracy"].to_numpy(float)
    ci_low, ci_high = bootstrap_mean_ci(
        ba,
        CONFIG["bootstrap_replicates"],
        stable_seed("summary", *keys),
    )

    summary_rows.append({
        "regime": keys[0],
        "classifier": keys[1],
        "panel": keys[2],
        "panel_label": keys[3],
        "panel_size": int(keys[4]),
        "balanced_accuracy_mean": float(ba.mean()),
        "balanced_accuracy_sd": float(ba.std(ddof=1)),
        "balanced_accuracy_ci95_low": ci_low,
        "balanced_accuracy_ci95_high": ci_high,
        "macro_f1_mean": float(group["macro_f1"].mean()),
        "macro_f1_sd": float(group["macro_f1"].std(ddof=1)),
        "accuracy_mean": float(group["accuracy"].mean()),
        "accuracy_sd": float(group["accuracy"].std(ddof=1)),
    })

performance_summary = pd.DataFrame(summary_rows)

atomic_csv(
    performance_summary,
    COMPARISON_DIR / "performance_summary.csv",
)

# =============================================================================
# 4. Q1 — GAIN VS MEILLEURE SINGLE-OMIC CONSTITUANTE
# =============================================================================

gain_rows = []

for (regime, classifier), clf in repeat_results.groupby(
    ["regime", "classifier"]
):
    for panel in PANELS:
        if len(panel) == 1:
            continue

        pkey = panel_key(panel)

        panel_scores = (
            clf[clf["panel"] == pkey]
            .set_index("repeat")["balanced_accuracy"]
        )

        constituent_scores = []

        for layer in panel:
            single_key = panel_key((layer,))
            constituent_scores.append(
                clf[clf["panel"] == single_key]
                .set_index("repeat")["balanced_accuracy"]
                .rename(layer)
            )

        single_table = pd.concat(
            constituent_scores,
            axis=1,
            join="inner",
        )

        best_constituent = single_table.max(axis=1)

        aligned = pd.concat(
            [
                panel_scores.rename("panel_score"),
                best_constituent.rename("best_single_score"),
            ],
            axis=1,
            join="inner",
        ).dropna()

        differences = (
            aligned["panel_score"] - aligned["best_single_score"]
        ).to_numpy(float)

        ci_low, ci_high = bootstrap_mean_ci(
            differences,
            CONFIG["bootstrap_replicates"],
            stable_seed("gain", regime, classifier, pkey),
        )

        gain_rows.append({
            "regime": regime,
            "classifier": classifier,
            "panel": pkey,
            "panel_label": panel_label(panel),
            "panel_size": len(panel),
            "mean_gain": float(differences.mean()),
            "sd_gain": float(differences.std(ddof=1)),
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "positive_gain_frequency": float(
                np.mean(differences > 0)
            ),
            "nonnegative_gain_frequency": float(
                np.mean(differences >= 0)
            ),
            "exact_sign_flip_p": exact_sign_flip_pvalue(
                differences
            ),
            "n_repeats": len(differences),
        })

gains = pd.DataFrame(gain_rows)
gains["q_bh"] = np.nan

for (regime, classifier), group in gains.groupby(
    ["regime", "classifier"]
):
    idx = group.index
    gains.loc[idx, "q_bh"] = multipletests(
        group["exact_sign_flip_p"].to_numpy(float),
        method="fdr_bh",
    )[1]

atomic_csv(
    gains,
    COMPARISON_DIR / "q1_multiomics_gains.csv",
)

# =============================================================================
# 5. CLASSEMENTS ET ROBUSTESSE INTER-CLASSIFIEUR
# =============================================================================

rankings = performance_summary.copy()
rankings["rank"] = (
    rankings
    .groupby(["regime", "classifier"])["balanced_accuracy_mean"]
    .rank(method="average", ascending=False)
)

atomic_csv(
    rankings,
    COMPARISON_DIR / "panel_rankings.csv",
)

rank_correlation_rows = []

for regime, group in rankings.groupby("regime"):
    pivot = group.pivot(
        index="panel",
        columns="classifier",
        values="rank",
    )

    for clf_a, clf_b in combinations(CLASSIFIERS, 2):
        rho, p_value = spearmanr(
            pivot[clf_a],
            pivot[clf_b],
        )
        rank_correlation_rows.append({
            "regime": regime,
            "classifier_a": clf_a,
            "classifier_b": clf_b,
            "spearman_rho": float(rho),
            "p_value": float(p_value),
        })

rank_correlations = pd.DataFrame(rank_correlation_rows)
atomic_csv(
    rank_correlations,
    COMPARISON_DIR / "rank_stability_between_classifiers.csv",
)

panel_robustness = (
    rankings
    .groupby(
        ["regime", "panel", "panel_label", "panel_size"],
        as_index=False,
    )
    .agg(
        mean_balanced_accuracy=(
            "balanced_accuracy_mean",
            "mean",
        ),
        sd_balanced_accuracy=(
            "balanced_accuracy_mean",
            "std",
        ),
        mean_rank=("rank", "mean"),
        worst_rank=("rank", "max"),
        top3_frequency=(
            "rank",
            lambda values: float(np.mean(values <= 3)),
        ),
    )
    .sort_values(
        ["regime", "mean_rank", "mean_balanced_accuracy"],
        ascending=[True, True, False],
    )
)

atomic_csv(
    panel_robustness,
    COMPARISON_DIR / "panel_classifier_robustness.csv",
)

# =============================================================================
# 6. CONTRIBUTION: LOO, SHAPLEY, TOP-3 PRESENCE
# =============================================================================

loo_rows = []
full_key = panel_key(FULL_PANEL)

for (regime, classifier), clf in repeat_results.groupby(
    ["regime", "classifier"]
):
    full_scores = (
        clf[clf["panel"] == full_key]
        .set_index("repeat")["balanced_accuracy"]
    )

    for layer in LAYERS:
        reduced_panel = canonical_panel(
            x for x in LAYERS if x != layer
        )
        reduced_key = panel_key(reduced_panel)

        reduced_scores = (
            clf[clf["panel"] == reduced_key]
            .set_index("repeat")["balanced_accuracy"]
        )

        aligned = pd.concat(
            [
                full_scores.rename("full"),
                reduced_scores.rename("without_layer"),
            ],
            axis=1,
            join="inner",
        ).dropna()

        differences = (
            aligned["full"] - aligned["without_layer"]
        ).to_numpy(float)

        ci_low, ci_high = bootstrap_mean_ci(
            differences,
            CONFIG["bootstrap_replicates"],
            stable_seed("loo", regime, classifier, layer),
        )

        loo_rows.append({
            "regime": regime,
            "classifier": classifier,
            "layer": layer,
            "layer_label": SHORT[layer],
            "mean_loo_contribution": float(differences.mean()),
            "sd": float(differences.std(ddof=1)),
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "positive_frequency": float(
                np.mean(differences > 0)
            ),
            "exact_sign_flip_p": exact_sign_flip_pvalue(
                differences
            ),
        })

loo = pd.DataFrame(loo_rows)
loo["q_bh"] = np.nan

for (regime, classifier), group in loo.groupby(
    ["regime", "classifier"]
):
    idx = group.index
    loo.loc[idx, "q_bh"] = multipletests(
        group["exact_sign_flip_p"].to_numpy(float),
        method="fdr_bh",
        alpha=CONFIG["fdr_alpha"],
    )[1]

loo["significant_positive"] = (
    (loo["mean_loo_contribution"] > 0)
    & (loo["ci95_low"] > 0)
    & (loo["q_bh"] < CONFIG["fdr_alpha"])
)

atomic_csv(
    loo,
    COMPARISON_DIR / "contribution_leave_one_omics_out.csv",
)


def baseline_value(
    regime: str,
    classifier: str,
    repeat_index: int,
) -> float:
    baseline_rows = all_fold_results[
        (all_fold_results["regime"] == regime)
        & (all_fold_results["classifier"] == classifier)
        & (all_fold_results["repeat"] == repeat_index)
        & (all_fold_results["panel"] == "EMPTY")
    ]

    if len(baseline_rows):
        return float(
            baseline_rows["balanced_accuracy"].mean()
        )

    # Balanced accuracy of a constant predictor in a 5-class problem.
    # Used only when the baseline was not recorded.
    return 0.2


def exact_shapley(
    regime: str,
    classifier: str,
    repeat_index: int,
) -> Dict[str, float]:
    subset_frame = repeat_results[
        (repeat_results["regime"] == regime)
        & (repeat_results["classifier"] == classifier)
        & (repeat_results["repeat"] == repeat_index)
    ]

    values = {
        tuple(): baseline_value(
            regime,
            classifier,
            repeat_index,
        )
    }

    for panel in PANELS:
        key = panel_key(panel)
        row = subset_frame[subset_frame["panel"] == key]

        if len(row) != 1:
            raise RuntimeError(
                f"Score manquant pour Shapley : "
                f"{regime}, {classifier}, repeat={repeat_index}, panel={key}"
            )

        values[panel] = float(
            row.iloc[0]["balanced_accuracy"]
        )

    n = len(LAYERS)
    factorial = math.factorial
    output = {}

    for layer in LAYERS:
        other_layers = [x for x in LAYERS if x != layer]
        contribution = 0.0

        for size in range(len(other_layers) + 1):
            for coalition in combinations(other_layers, size):
                coalition = canonical_panel(coalition)
                coalition_plus = canonical_panel(
                    (*coalition, layer)
                )

                weight = (
                    factorial(size)
                    * factorial(n - size - 1)
                    / factorial(n)
                )

                contribution += weight * (
                    values[coalition_plus] - values[coalition]
                )

        output[layer] = float(contribution)

    expected_total = values[FULL_PANEL] - values[tuple()]
    actual_total = sum(output.values())

    if not np.isclose(actual_total, expected_total, atol=1e-10):
        raise RuntimeError(
            "Shapley efficiency property failed."
        )

    return output


shapley_rows = []

for regime in REGIMES_TO_ANALYZE:
    for classifier in CLASSIFIERS:
        repeats = sorted(
            repeat_results.loc[
                (repeat_results["regime"] == regime)
                & (repeat_results["classifier"] == classifier),
                "repeat",
            ].unique()
        )

        for repeat_index in repeats:
            contributions = exact_shapley(
                regime,
                classifier,
                int(repeat_index),
            )

            for layer, value in contributions.items():
                shapley_rows.append({
                    "regime": regime,
                    "classifier": classifier,
                    "repeat": int(repeat_index),
                    "layer": layer,
                    "layer_label": SHORT[layer],
                    "performance_shapley": value,
                })

shapley_repeat = pd.DataFrame(shapley_rows)
atomic_csv(
    shapley_repeat,
    COMPARISON_DIR / "contribution_shapley_repeat.csv",
)

shapley_summary_rows = []

for keys, group in shapley_repeat.groupby(
    ["regime", "classifier", "layer", "layer_label"]
):
    values = group["performance_shapley"].to_numpy(float)
    ci_low, ci_high = bootstrap_mean_ci(
        values,
        CONFIG["bootstrap_replicates"],
        stable_seed("shapley", *keys),
    )

    shapley_summary_rows.append({
        "regime": keys[0],
        "classifier": keys[1],
        "layer": keys[2],
        "layer_label": keys[3],
        "mean_performance_shapley": float(values.mean()),
        "sd": float(values.std(ddof=1)),
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "positive_frequency": float(np.mean(values > 0)),
        "exact_sign_flip_p": exact_sign_flip_pvalue(values),
    })

shapley_summary = pd.DataFrame(shapley_summary_rows)
shapley_summary["q_bh"] = np.nan
for (regime, classifier), group in shapley_summary.groupby(
    ["regime", "classifier"]
):
    idx = group.index
    shapley_summary.loc[idx, "q_bh"] = multipletests(
        group["exact_sign_flip_p"].to_numpy(float),
        method="fdr_bh",
        alpha=CONFIG["fdr_alpha"],
    )[1]
shapley_summary["significant_positive"] = (
    (shapley_summary["mean_performance_shapley"] > 0)
    & (shapley_summary["ci95_low"] > 0)
    & (shapley_summary["q_bh"] < CONFIG["fdr_alpha"])
)
atomic_csv(
    shapley_summary,
    COMPARISON_DIR / "contribution_shapley_summary.csv",
)

presence_rows = []

for regime, regime_frame in repeat_results.groupby("regime"):
    counts = Counter()
    denominator = 0

    for (_, _), group in regime_frame.groupby(
        ["classifier", "repeat"]
    ):
        top = group.sort_values(
            "balanced_accuracy",
            ascending=False,
        ).head(CONFIG["top_k_panels_for_presence"])

        for key in top["panel"]:
            for layer in PANEL_BY_KEY[key]:
                counts[layer] += 1

        denominator += len(top)

    for layer in LAYERS:
        presence_rows.append({
            "regime": regime,
            "layer": layer,
            "layer_label": SHORT[layer],
            "top_panel_presence_count": int(counts[layer]),
            "top_panel_presence_frequency": float(
                counts[layer] / denominator
            ),
            "denominator_top_panels": int(denominator),
        })

presence = pd.DataFrame(presence_rows)
atomic_csv(
    presence,
    COMPARISON_DIR / "contribution_top3_presence.csv",
)

# =============================================================================
# 7. MINIMAL PANEL BY NON-INFERIORITY
# =============================================================================

noninferiority_rows = []

for (regime, classifier), clf in repeat_results.groupby(
    ["regime", "classifier"]
):
    mean_scores = (
        clf.groupby("panel")["balanced_accuracy"]
        .mean()
        .sort_values(ascending=False)
    )
    best_panel = str(mean_scores.index[0])

    best_scores = (
        clf[clf["panel"] == best_panel]
        .set_index("repeat")["balanced_accuracy"]
    )

    for panel in PANELS:
        key = panel_key(panel)

        candidate_scores = (
            clf[clf["panel"] == key]
            .set_index("repeat")["balanced_accuracy"]
        )

        aligned = pd.concat(
            [
                candidate_scores.rename("candidate"),
                best_scores.rename("best"),
            ],
            axis=1,
            join="inner",
        ).dropna()

        differences = (
            aligned["candidate"] - aligned["best"]
        ).to_numpy(float)

        ci_low, ci_high = bootstrap_mean_ci(
            differences,
            CONFIG["bootstrap_replicates"],
            stable_seed(
                "noninferiority",
                regime,
                classifier,
                key,
            ),
        )

        noninferiority_rows.append({
            "regime": regime,
            "classifier": classifier,
            "best_panel": best_panel,
            "candidate_panel": key,
            "candidate_panel_label": panel_label(panel),
            "candidate_panel_size": len(panel),
            "mean_difference": float(differences.mean()),
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "margin": CONFIG["noninferiority_margin"],
            "noninferior": bool(
                ci_low > -CONFIG["noninferiority_margin"]
            ),
        })

noninferiority = pd.DataFrame(noninferiority_rows)
atomic_csv(
    noninferiority,
    COMPARISON_DIR / "minimal_panel_noninferiority.csv",
)

minimal_rows = []

for (regime, classifier), group in noninferiority.groupby(
    ["regime", "classifier"]
):
    eligible = group[group["noninferior"]].sort_values(
        ["candidate_panel_size", "mean_difference"],
        ascending=[True, False],
    )

    if len(eligible):
        selected = eligible.iloc[0]
        minimal_rows.append({
            "regime": regime,
            "classifier": classifier,
            "minimal_panel": selected["candidate_panel"],
            "minimal_panel_label": selected[
                "candidate_panel_label"
            ],
            "panel_size": int(
                selected["candidate_panel_size"]
            ),
            "mean_difference_from_best": float(
                selected["mean_difference"]
            ),
            "ci95_low": float(selected["ci95_low"]),
            "ci95_high": float(selected["ci95_high"]),
        })

minimal_panels = pd.DataFrame(minimal_rows)
atomic_csv(
    minimal_panels,
    COMPARISON_DIR / "minimal_sufficient_panel.csv",
)

# =============================================================================
# 8. AUTOMATED REPORT
# =============================================================================

def best_panel_for_regime(
    regime: str,
    panel_size: Optional[int] = None,
) -> Dict[str, Any]:
    subset = panel_robustness[
        panel_robustness["regime"] == regime
    ]

    if panel_size is not None:
        subset = subset[
            subset["panel_size"] == panel_size
        ]

    subset = subset.sort_values(
        ["mean_rank", "mean_balanced_accuracy"],
        ascending=[True, False],
    )

    row = subset.iloc[0]

    return {
        "panel": str(row["panel"]),
        "panel_label": str(row["panel_label"]),
        "panel_size": int(row["panel_size"]),
        "mean_balanced_accuracy_across_classifiers": float(
            row["mean_balanced_accuracy"]
        ),
        "mean_rank": float(row["mean_rank"]),
        "worst_rank": float(row["worst_rank"]),
        "top3_frequency": float(row["top3_frequency"]),
    }


regime_reports = {}

for regime in REGIMES_TO_ANALYZE:
    significant_positive = gains[
        (gains["regime"] == regime)
        & (gains["mean_gain"] > 0)
        & (gains["q_bh"] < CONFIG["fdr_alpha"])
    ]

    dominance = (
        shapley_summary[
            shapley_summary["regime"] == regime
        ]
        .groupby(["layer", "layer_label"], as_index=False)
        ["mean_performance_shapley"]
        .mean()
        .sort_values(
            "mean_performance_shapley",
            ascending=False,
        )
    )

    dominant = dominance.iloc[0]

    minimal_for_regime = minimal_panels[
        minimal_panels["regime"] == regime
    ]

    minimal_consensus = None
    if len(minimal_for_regime):
        counts = minimal_for_regime[
            "minimal_panel"
        ].value_counts()
        minimal_consensus = {
            "panel": str(counts.index[0]),
            "agreement_count": int(counts.iloc[0]),
            "n_classifiers": int(
                len(minimal_for_regime)
            ),
        }

    regime_reports[regime] = {
        "does_multiomics_improve_prediction": {
            "answer": bool(len(significant_positive) > 0),
            "n_classifier_specific_significant_positive_gains": int(
                len(significant_positive)
            ),
            "criterion": (
                "positive mean gain and BH-adjusted exact "
                "sign-flip q < 0.05"
            ),
        },
        "best_single_omic": best_panel_for_regime(
            regime,
            panel_size=1,
        ),
        "best_pair": best_panel_for_regime(
            regime,
            panel_size=2,
        ),
        "best_triplet": best_panel_for_regime(
            regime,
            panel_size=3,
        ),
        "best_overall_panel": best_panel_for_regime(
            regime
        ),
        "minimal_sufficient_panel": minimal_consensus,
        "most_dominant_layer": {
            "layer": str(dominant["layer"]),
            "layer_label": str(dominant["layer_label"]),
            "mean_performance_shapley_across_classifiers": float(
                dominant["mean_performance_shapley"]
            ),
        },
    }




report = {
    "analysis": "fixed_hyperparameter_post_preprocessing",
    "regime_paths": {
        key: str(value)
        for key, value in REGIME_PATHS.items()
        if key in REGIMES_TO_ANALYZE
    },
    "primary_question": (
        "Does integrating multiple omics layers provide reproducible "
        "predictive gains beyond the best constituent single-omics layer?"
    ),
    "regime_results": regime_reports,
    "prediction_level_analyses": {
        "rpred_status": (
            "not_computed_from_panel_fold_results_only"
        ),
        "pid_status": (
            "not_computed_from_panel_fold_results_only"
        ),
        "reason": (
            "Rpred patient-level bootstrap and prediction-level PID "
            "require out-of-fold predictions for every patient, panel, "
            "classifier and repetition. panel_fold_results.csv contains "
            "aggregate fold metrics only."
        ),
    },
    "methodological_notes": [
        "No preprocessing is rerun.",
        "No nested cross-validation is part of the reported analysis.",
        "All inference uses ten repeat-level paired observations rather than fifty overlapping folds.",
        "The reported analysis is the fixed-k=50 regime with the same a priori classifier hyperparameters across panels and repetitions.",
        "The The reported analysis uses the fixed-k=50 regime; any alternative dimensional-regime comparison is outside the reported study.",
        "The reported study uses the fixed-k=50, fixed-hyperparameter analysis; nested cross-validation is not part of the reported analysis.",
    ],
}

atomic_json(
    report,
    COMPARISON_DIR / "report.json",
)

log("")
log("=" * 80)
log("PIPELINE COMPLETE")
log(f"Results : {COMPARISON_DIR}")
log("=" * 80)


## 2. Predictive complementarity with $R_{\mathrm{pred}}$


**Level 4.** Computes the model-relative predictive complementarity index at the repetition level for the six layer pairs and three classifiers, tests each pair against the null of no gain over its stronger constituent, and assesses agreement of the pair ordering between classifiers.

In [ ]:
"""
 - FULL RPRED FROM THE FIXED RESULTS

No preprocessing, nested cross-validation or model retraining is performed.

Lancement :
    %run rpred_fixed_models_complete.py

"""

from __future__ import annotations

import hashlib
import json
import os
import warnings
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# =============================================================================
# 0. CONFIGURATION
# =============================================================================

try:
    from google.colab import drive

    if not os.path.exists(str(DATA_DIR)):
        drive.mount("/content/drive")
    DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

REGIME_PATHS = {
    "k50": Path(
        os.environ.get(
            "K50_RUN_DIR",
            str(DATA_DIR / "runs" / "2026-07-26"),
        )
    ),
}

REGIMES_TO_ANALYZE = [
    item.strip()
    for item in os.environ.get("REGIMES_TO_ANALYZE", "k50").split(",")
    if item.strip()
]

unknown = [name for name in REGIMES_TO_ANALYZE if name not in REGIME_PATHS]
if unknown:
    raise ValueError(f"Unknown regimes:  {unknown}")

LAYERS = ["mutations", "cnv", "mrna", "rppa"]
SHORT = {
    "mutations": "Mutations",
    "cnv": "CNV",
    "mrna": "mRNA",
    "rppa": "RPPA",
}
CLASSIFIERS = ["RF", "XGB", "SVM"]
PAIRS = list(combinations(LAYERS, 2))
FULL_PANEL = "+".join(LAYERS)

SEED = int(os.environ.get("SEED", "42"))
N_BOOTSTRAP = int(os.environ.get("RPRED_BOOTSTRAP_REPLICATES", "10000"))
FDR_ALPHA = float(os.environ.get("RPRED_FDR_ALPHA", "0.05"))
DENOMINATOR_EPSILON = float(
    os.environ.get("RPRED_DENOMINATOR_EPSILON", "1e-8")
)

OUT_DIR = DATA_DIR / "runs" / "comparison_rpred_fixed_models"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "regimes": REGIMES_TO_ANALYZE,
    "regime_paths": {k: str(REGIME_PATHS[k]) for k in REGIMES_TO_ANALYZE},
    "layers": LAYERS,
    "classifiers": CLASSIFIERS,
    "bootstrap_replicates": N_BOOTSTRAP,
    "fdr_alpha": FDR_ALPHA,
    "denominator_epsilon": DENOMINATOR_EPSILON,
    "formula": (
        "Rpred = 1 - [BA(pair)-max(BA(single_A),BA(single_B))] "
        "/ [BA(full)-BA(baseline)]"
    ),
}

# =============================================================================
# 1. UTILITAIRES
# =============================================================================


def log(message: str = "") -> None:
    print(message)


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(payload: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def stable_seed(*parts: Any) -> int:
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def find_existing(candidates: list[Path], required: bool = True) -> Optional[Path]:
    for path in candidates:
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(
            "No file found among: \n" + "\n".join(map(str, candidates))
        )
    return None


def detect_column(
    frame: pd.DataFrame,
    candidates: Iterable[str],
    required: bool = True,
) -> Optional[str]:
    lower_map = {str(column).lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    if required:
        raise RuntimeError(
            f"Column not found among {list(candidates)}. "
            f"Colonnes disponibles : {list(frame.columns)}"
        )
    return None


def normalize_classifier(value: Any) -> str:
    text = str(value).strip().upper()
    aliases = {
        "RANDOMFOREST": "RF",
        "RANDOM_FOREST": "RF",
        "RANDOM FOREST": "RF",
        "XGBOOST": "XGB",
        "SVC": "SVM",
    }
    return aliases.get(text, text)


def normalize_panel(value: Any) -> str:
    text = str(value).strip().lower()
    aliases = {
        "mutation": "mutations",
        "mut": "mutations",
        "muts": "mutations",
        "copy_number": "cnv",
        "copy-number": "cnv",
        "rna": "mrna",
        "protein": "rppa",
        "proteomics": "rppa",
        "baseline": "EMPTY",
        "dummy": "EMPTY",
        "empty": "EMPTY",
    }
    if text.upper() == "EMPTY":
        return "EMPTY"
    for separator in ("|", ",", ";", "__", "/", " "):
        text = text.replace(separator, "+")
    tokens = [
        aliases.get(token.strip(), token.strip())
        for token in text.split("+")
        if token.strip()
    ]
    if not tokens:
        return "EMPTY"
    order = {layer: idx for idx, layer in enumerate(LAYERS)}
    unknown_tokens = [token for token in tokens if token not in order]
    if unknown_tokens:
        raise ValueError(
            f"Panel non reconnu : {value!r}; tokens inconnus={unknown_tokens}"
        )
    return "+".join(sorted(set(tokens), key=lambda token: order[token]))


def bootstrap_mean_ci(
    values: np.ndarray,
    n_bootstrap: int,
    seed: int,
) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(n_bootstrap, len(values)))
    means = values[indices].mean(axis=1)
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


def exact_sign_flip_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan
    observed = abs(float(differences.mean()))
    n_values = len(differences)
    null_means = np.empty(2**n_values, dtype=float)
    for mask in range(2**n_values):
        signs = np.asarray(
            [1.0 if ((mask >> bit) & 1) else -1.0 for bit in range(n_values)]
        )
        null_means[mask] = abs(float(np.mean(signs * differences)))
    return float(np.mean(null_means >= observed - 1e-12))


def load_n_classes(run_dir: Path) -> int:
    cohort_path = find_existing(
        [
            run_dir / "results" / "cohort_manifest.csv",
            run_dir / "results" / "cohort.csv",
        ],
        required=False,
    )
    if cohort_path is None:
        return 5
    cohort = pd.read_csv(cohort_path)
    label_col = detect_column(
        cohort,
        ["label_encoded", "y", "target_encoded", "pam50_encoded"],
        required=False,
    )
    if label_col is None:
        return 5
    labels = pd.to_numeric(cohort[label_col], errors="coerce").dropna()
    return int(labels.nunique()) if len(labels) else 5


def load_fold_results(regime: str, run_dir: Path) -> tuple[pd.DataFrame, int]:
    path = find_existing(
        [
            run_dir
            / "results"
            / "model_evaluation"
            / "panel_fold_results.csv",
            run_dir / "model_evaluation" / "panel_fold_results.csv",
            run_dir / "results" / "panel_fold_results.csv",
        ]
    )
    raw = pd.read_csv(path)

    classifier_col = detect_column(raw, ["classifier", "model", "classifier_name"])
    split_col = detect_column(raw, ["split", "outer_split", "split_index"])
    repeat_col = detect_column(raw, ["repeat", "repeat_index", "cv_repeat"])
    fold_col = detect_column(raw, ["fold", "fold_index", "cv_fold"])
    panel_col = detect_column(raw, ["panel", "panel_key", "combination"])
    ba_col = detect_column(raw, ["balanced_accuracy", "balanced_acc", "ba"])
    f1_col = detect_column(
        raw, ["macro_f1", "f1_macro", "macro-f1"], required=False
    )
    accuracy_col = detect_column(raw, ["accuracy", "acc"], required=False)

    normalized = pd.DataFrame(
        {
            "regime": regime,
            "classifier": raw[classifier_col].map(normalize_classifier),
            "split": pd.to_numeric(raw[split_col], errors="raise").astype(int),
            "repeat": pd.to_numeric(raw[repeat_col], errors="raise").astype(int),
            "fold": pd.to_numeric(raw[fold_col], errors="raise").astype(int),
            "panel": raw[panel_col].map(normalize_panel),
            "balanced_accuracy": pd.to_numeric(raw[ba_col], errors="raise"),
        }
    )
    if f1_col is not None:
        normalized["macro_f1"] = pd.to_numeric(raw[f1_col], errors="coerce")
    if accuracy_col is not None:
        normalized["accuracy"] = pd.to_numeric(
            raw[accuracy_col], errors="coerce"
        )

    invalid_classifiers = set(normalized["classifier"]) - set(CLASSIFIERS)
    if invalid_classifiers:
        raise RuntimeError(
            f"{regime}: classifieurs inconnus : {invalid_classifiers}"
        )
    if normalized.duplicated(["split", "classifier", "panel"]).any():
        raise RuntimeError(f"{regime}: units duplicated dans {path}")

    log(f"{regime}: {path} loaded ({len(normalized)} rows).")
    return normalized, load_n_classes(run_dir)


# =============================================================================
# 2. CHARGEMENT ET VALIDATION
# =============================================================================

frames: list[pd.DataFrame] = []
N_CLASSES_BY_REGIME: dict[str, int] = {}

for regime in REGIMES_TO_ANALYZE:
    frame, n_classes = load_fold_results(regime, REGIME_PATHS[regime])
    frames.append(frame)
    N_CLASSES_BY_REGIME[regime] = n_classes

fold_results = pd.concat(frames, ignore_index=True)
atomic_csv(fold_results, OUT_DIR / "normalized_panel_fold_results.csv")

expected_panels = {
    "+".join(panel)
    for size in range(1, len(LAYERS) + 1)
    for panel in combinations(LAYERS, size)
}

for regime in REGIMES_TO_ANALYZE:
    regime_frame = fold_results[
        (fold_results["regime"] == regime) & (fold_results["panel"] != "EMPTY")
    ]
    if regime_frame["split"].nunique() != 50:
        raise RuntimeError(f"{regime}: 50 splits attendus.")
    if regime_frame["repeat"].nunique() != 10:
        raise RuntimeError(f"{regime}: 10 repetitions attendues.")
    for split_index in sorted(regime_frame["split"].unique()):
        for classifier in CLASSIFIERS:
            actual = set(
                regime_frame.loc[
                    (regime_frame["split"] == split_index)
                    & (regime_frame["classifier"] == classifier),
                    "panel",
                ]
            )
            if actual != expected_panels:
                raise RuntimeError(
                    f"{regime}: panels incomplets pour {classifier}, "
                    f"split={split_index}. Manquants={sorted(expected_panels - actual)}"
                )

metric_columns = [
    column
    for column in ["balanced_accuracy", "macro_f1", "accuracy"]
    if column in fold_results.columns
]

repeat_metrics = (
    fold_results.groupby(
        ["regime", "classifier", "repeat", "panel"], as_index=False
    )[metric_columns].mean()
)
atomic_csv(repeat_metrics, OUT_DIR / "repeat_level_metrics.csv")


def get_repeat_scores(regime: str, classifier: str, panel: str) -> pd.Series:
    return (
        repeat_metrics[
            (repeat_metrics["regime"] == regime)
            & (repeat_metrics["classifier"] == classifier)
            & (repeat_metrics["panel"] == panel)
        ]
        .set_index("repeat")["balanced_accuracy"]
    )


# =============================================================================
# 3. RPRED PAR REPETITION
# =============================================================================

repeat_rows: list[dict[str, Any]] = []

for regime in REGIMES_TO_ANALYZE:
    baseline_fallback = 1.0 / N_CLASSES_BY_REGIME[regime]

    for classifier in CLASSIFIERS:
        full_scores = get_repeat_scores(regime, classifier, FULL_PANEL)
        baseline_scores = get_repeat_scores(regime, classifier, "EMPTY")

        for layer_a, layer_b in PAIRS:
            pair = f"{layer_a}+{layer_b}"
            pair_scores = get_repeat_scores(regime, classifier, pair)
            single_a_scores = get_repeat_scores(regime, classifier, layer_a)
            single_b_scores = get_repeat_scores(regime, classifier, layer_b)

            shared_repeats = sorted(
                set(pair_scores.index)
                & set(single_a_scores.index)
                & set(single_b_scores.index)
                & set(full_scores.index)
            )

            for repeat_index in shared_repeats:
                score_a = float(single_a_scores.loc[repeat_index])
                score_b = float(single_b_scores.loc[repeat_index])
                best_single_score = max(score_a, score_b)
                best_single_layer = layer_a if score_a >= score_b else layer_b
                pair_score = float(pair_scores.loc[repeat_index])
                full_score = float(full_scores.loc[repeat_index])

                if repeat_index in baseline_scores.index:
                    baseline_score = float(baseline_scores.loc[repeat_index])
                    baseline_source = "recorded"
                else:
                    baseline_score = float(baseline_fallback)
                    baseline_source = "fallback_1_over_n_classes"

                raw_gain = pair_score - best_single_score
                denominator = full_score - baseline_score
                denominator_valid = bool(
                    np.isfinite(denominator) and denominator > DENOMINATOR_EPSILON
                )
                rpred = (
                    float(1.0 - raw_gain / denominator)
                    if denominator_valid
                    else np.nan
                )

                repeat_rows.append(
                    {
                        "regime": regime,
                        "classifier": classifier,
                        "repeat": int(repeat_index),
                        "pair": pair,
                        "pair_label": f"{SHORT[layer_a]}+{SHORT[layer_b]}",
                        "layer_a": layer_a,
                        "layer_b": layer_b,
                        "single_a_score": score_a,
                        "single_b_score": score_b,
                        "best_single_layer": best_single_layer,
                        "best_single_score": best_single_score,
                        "pair_score": pair_score,
                        "raw_gain": raw_gain,
                        "full_score": full_score,
                        "baseline_score": baseline_score,
                        "baseline_source": baseline_source,
                        "normalizing_denominator": denominator,
                        "denominator_valid": denominator_valid,
                        "rpred": rpred,
                    }
                )

rpred_repeat = pd.DataFrame(repeat_rows)
atomic_csv(rpred_repeat, OUT_DIR / "rpred_repeat_level.csv")

# =============================================================================
# 4. SUMMARY ET TESTS
# =============================================================================

valid_rpred = rpred_repeat[rpred_repeat["denominator_valid"].astype(bool)].copy()
summary_rows: list[dict[str, Any]] = []

for keys, group in valid_rpred.groupby(
    ["regime", "classifier", "pair", "pair_label"]
):
    rpred_values = group["rpred"].to_numpy(float)
    gain_values = group["raw_gain"].to_numpy(float)
    denominator_values = group["normalizing_denominator"].to_numpy(float)

    rpred_ci_low, rpred_ci_high = bootstrap_mean_ci(
        rpred_values,
        N_BOOTSTRAP,
        stable_seed("rpred", *keys),
    )
    gain_ci_low, gain_ci_high = bootstrap_mean_ci(
        gain_values,
        N_BOOTSTRAP,
        stable_seed("gain", *keys),
    )

    summary_rows.append(
        {
            "regime": keys[0],
            "classifier": keys[1],
            "pair": keys[2],
            "pair_label": keys[3],
            "n_valid_repeats": int(len(rpred_values)),
            "rpred_mean": float(rpred_values.mean()),
            "rpred_sd": float(rpred_values.std(ddof=1)),
            "rpred_median": float(np.median(rpred_values)),
            "rpred_min": float(rpred_values.min()),
            "rpred_max": float(rpred_values.max()),
            "rpred_ci95_low": rpred_ci_low,
            "rpred_ci95_high": rpred_ci_high,
            "frequency_rpred_below_1": float(np.mean(rpred_values < 1.0)),
            "frequency_rpred_above_1": float(np.mean(rpred_values > 1.0)),
            "raw_gain_mean": float(gain_values.mean()),
            "raw_gain_sd": float(gain_values.std(ddof=1)),
            "raw_gain_ci95_low": gain_ci_low,
            "raw_gain_ci95_high": gain_ci_high,
            "positive_gain_frequency": float(np.mean(gain_values > 0.0)),
            "denominator_mean": float(denominator_values.mean()),
            "denominator_min": float(denominator_values.min()),
            "exact_sign_flip_p_vs_1": exact_sign_flip_pvalue(
                rpred_values - 1.0
            ),
            "ci_excludes_1": bool(
                rpred_ci_high < 1.0 or rpred_ci_low > 1.0
            ),
        }
    )

rpred_summary = pd.DataFrame(summary_rows)
rpred_summary["q_bh"] = np.nan

for (_, _), group in rpred_summary.groupby(["regime", "classifier"]):
    idx = group.index
    rpred_summary.loc[idx, "q_bh"] = multipletests(
        group["exact_sign_flip_p_vs_1"].to_numpy(float),
        method="fdr_bh",
        alpha=FDR_ALPHA,
    )[1]


def interpret_row(row: pd.Series) -> str:
    if (
        row["raw_gain_mean"] > 0
        and row["rpred_ci95_high"] < 1.0
        and row["q_bh"] < FDR_ALPHA
    ):
        return "robust_predictive_complementarity"
    if (
        row["raw_gain_mean"] < 0
        and row["rpred_ci95_low"] > 1.0
        and row["q_bh"] < FDR_ALPHA
    ):
        return "robust_predictive_interference"
    if row["rpred_ci95_low"] <= 1.0 <= row["rpred_ci95_high"]:
        return "compatible_with_predictive_redundancy"
    return "inconclusive"


rpred_summary["interpretation"] = rpred_summary.apply(interpret_row, axis=1)
atomic_csv(rpred_summary, OUT_DIR / "rpred_summary.csv")

# =============================================================================
# 5. CLASSEMENTS ET ROBUSTESSE ENTRE CLASSIFIEURS
# =============================================================================

rankings = rpred_summary.copy()
rankings["complementarity_rank"] = rankings.groupby(
    ["regime", "classifier"]
)["rpred_mean"].rank(ascending=True, method="average")
rankings["gain_rank"] = rankings.groupby(["regime", "classifier"])[
    "raw_gain_mean"
].rank(ascending=False, method="average")
atomic_csv(rankings, OUT_DIR / "rpred_pair_rankings.csv")

rank_stability_rows: list[dict[str, Any]] = []
for regime, group in rankings.groupby("regime"):
    pivot = group.pivot(
        index="pair", columns="classifier", values="complementarity_rank"
    )
    for classifier_a, classifier_b in combinations(CLASSIFIERS, 2):
        rho, p_value = spearmanr(pivot[classifier_a], pivot[classifier_b])
        rank_stability_rows.append(
            {
                "regime": regime,
                "classifier_a": classifier_a,
                "classifier_b": classifier_b,
                "spearman_rho": float(rho),
                "p_value": float(p_value),
                "n_pairs": int(len(pivot)),
            }
        )

rank_stability = pd.DataFrame(rank_stability_rows)
atomic_csv(
    rank_stability,
    OUT_DIR / "rpred_rank_stability_between_classifiers.csv",
)

robustness_rows: list[dict[str, Any]] = []
for (regime, pair, pair_label), group in rankings.groupby(
    ["regime", "pair", "pair_label"]
):
    counts = group["interpretation"].value_counts().to_dict()
    robustness_rows.append(
        {
            "regime": regime,
            "pair": pair,
            "pair_label": pair_label,
            "mean_rpred_across_classifiers": float(group["rpred_mean"].mean()),
            "sd_rpred_across_classifiers": float(
                group["rpred_mean"].std(ddof=1)
            ),
            "mean_raw_gain_across_classifiers": float(
                group["raw_gain_mean"].mean()
            ),
            "mean_complementarity_rank": float(
                group["complementarity_rank"].mean()
            ),
            "worst_complementarity_rank": float(
                group["complementarity_rank"].max()
            ),
            "n_classifiers_robust_complementarity": int(
                counts.get("robust_predictive_complementarity", 0)
            ),
            "n_classifiers_robust_interference": int(
                counts.get("robust_predictive_interference", 0)
            ),
            "n_classifiers_compatible_redundancy": int(
                counts.get("compatible_with_predictive_redundancy", 0)
            ),
        }
    )

rpred_robustness = pd.DataFrame(robustness_rows).sort_values(
    ["regime", "mean_complementarity_rank", "mean_rpred_across_classifiers"],
    ascending=[True, True, True],
)
atomic_csv(rpred_robustness, OUT_DIR / "rpred_classifier_robustness.csv")

# =============================================================================
# 6. OPTIONAL DIMENSIONAL SENSITIVITY CHECK — not part of the reported analysis
# =============================================================================

k_comparison = pd.DataFrame()
k_rank_comparison = pd.DataFrame()

def most_complementary_pair(regime: str) -> dict[str, Any]:
    row = rpred_robustness[rpred_robustness["regime"] == regime].sort_values(
        ["mean_complementarity_rank", "mean_rpred_across_classifiers"],
        ascending=[True, True],
    ).iloc[0]
    return {
        "pair": str(row["pair"]),
        "pair_label": str(row["pair_label"]),
        "mean_rpred_across_classifiers": float(
            row["mean_rpred_across_classifiers"]
        ),
        "mean_raw_gain_across_classifiers": float(
            row["mean_raw_gain_across_classifiers"]
        ),
        "mean_complementarity_rank": float(row["mean_complementarity_rank"]),
        "n_classifiers_robust_complementarity": int(
            row["n_classifiers_robust_complementarity"]
        ),
    }


def pair_closest_to_redundancy(regime: str) -> dict[str, Any]:
    subset = rpred_robustness[rpred_robustness["regime"] == regime].copy()
    subset["distance_to_one"] = (
        subset["mean_rpred_across_classifiers"] - 1.0
    ).abs()
    row = subset.sort_values("distance_to_one").iloc[0]
    return {
        "pair": str(row["pair"]),
        "pair_label": str(row["pair_label"]),
        "mean_rpred_across_classifiers": float(
            row["mean_rpred_across_classifiers"]
        ),
    }


regime_reports: dict[str, Any] = {}
for regime in REGIMES_TO_ANALYZE:
    regime_reports[regime] = {
        "most_complementary_pair": most_complementary_pair(regime),
        "pair_closest_to_predictive_redundancy": pair_closest_to_redundancy(
            regime
        ),
        "n_classifier_specific_robust_complementarity_results": int(
            (
                (rpred_summary["regime"] == regime)
                & (
                    rpred_summary["interpretation"]
                    == "robust_predictive_complementarity"
                )
            ).sum()
        ),
        "n_classifier_specific_robust_interference_results": int(
            (
                (rpred_summary["regime"] == regime)
                & (
                    rpred_summary["interpretation"]
                    == "robust_predictive_interference"
                )
            ).sum()
        ),
    }


report = {
    "analysis": "Rpred fixed-hyperparameter post-preprocessing analysis",
    "config": CONFIG,
    "definition": {
        "raw_gain": "BA(pair)-max[BA(single_A),BA(single_B)]",
        "Rpred": "1-raw_gain/[BA(full)-BA(baseline)]",
        "interpretation": {
            "Rpred_below_1": "predictive complementarity",
            "Rpred_compatible_with_1": "compatible with predictive redundancy",
            "Rpred_above_1": "predictive interference",
        },
    },
    "primary_answers": regime_reports,
    "methodological_notes": [
        "No preprocessing, nested CV or model retraining is performed.",
        "Five folds are averaged inside each repetition before inference.",
        "The inferential unit is the repeated-CV repetition (n=10).",
        "Rpred is always reported with the raw BA gain and denominator.",
        "Non-positive or near-zero denominators are marked invalid.",
        "Classifier-specific conclusions remain separate.",
        "Rpred measures performance-level complementarity, not PID synergy.",
    ],
}

atomic_json(CONFIG, OUT_DIR / "config.json")
atomic_json(report, OUT_DIR / "report.json")

log("")
log("=" * 80)
log("CALCUL RPRED COMPLETE")
log(f"Results : {OUT_DIR}")
for regime in REGIMES_TO_ANALYZE:
    best = regime_reports[regime]["most_complementary_pair"]
    log(
        f"{regime} — most complementary pair:  "
        f"{best['pair_label']} (Rpred moyen={best['mean_rpred_across_classifiers']:.4f})"
    )
log("=" * 80)


## 3. Prediction-level partial information decomposition


**Dependency.** Installs the ECOS solver required by BROJA-2PID.

In [ ]:
!pip install -q ecos
!pip install -q git+https://github.com/Abzinger/BROJA_2PID.git@master

**Dependency.** Installs BROJA-2PID from its source repository.

In [ ]:
import os
os.environ["RUN_DIR"] = str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'runs' / '2026-07-26')

**Level 5.** Reconstructs the out-of-fold predictions of the four single-layer models, accepting them only after they reproduce the recorded per-fold metrics within tolerance, then computes the prediction-level partial information decomposition under both estimators with a patient-clustered bootstrap, validates the estimators on canonical gates, and tests the concordance between predictive complementarity and the information atoms.

In [ ]:

"""
===============================================================================
 - FULL PREDICTION-LEVEL PID: WILLIAMS-BEER I_min + BROJA-2PID
===============================================================================

NO PREPROCESSING IS RERUN.
NO NESTED CROSS-VALIDATION IN THE REPORTED ANALYSIS.

This script uses the representations already produced and the models with
fixed hyperparameters. It performs:

A. Controlled reconstruction of the out-of-fold predictions of the four single-omics models
   (at most 600 fits, with fine-grained resume).
B. Audit : les metrics reconstruites doivent reproduire
   panel_fold_results.csv.
C. PID Williams–Beer I_min true.
D. PID BROJA-2PID true avec solveur ECOS.
E. Estimations par repetition, estimations pooled-repeat et IC95 %.
F. Bootstrap patient stratified par grappes :
   a patient is resampled together with its predictions from all ten repetitions.
G. Same resampling for I_min, BROJA, RF, XGBoost and SVM.
H. Diagnostics du solveur, reprises, convergence et comparaison des estimateurs.
I. Comparaison avec Rpred.

IMPORTANT
---------
The earlier code:
    redundancy = min(I(T;A), I(T;B))
is not the Williams-Beer I_min; it corresponds to MMI redundancy.
This script computes the true I_min from the specific information
of each target state.

BROJA installation in Colab, before running:
    !pip install -q ecos
    !pip install -q git+https://github.com/Abzinger/BROJA_2PID.git@master
    %run pid_imin_broja_fixed_models_complete.py

k50 run:
    import os
    os.environ["RUN_DIR"] = (
        str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'runs' / '2026-07-26')
    )
    %run pid_imin_broja_fixed_models_complete.py
"""

import os
import gc
import json
import math
import time
import pickle
import hashlib
import warnings
import importlib.metadata
from pathlib import Path
from itertools import combinations
from collections import Counter
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

try:
    from broja2pid import BROJA_2PID
except Exception as import_error:
    raise RuntimeError(
        "BROJA-2PID is not installed.\n"
        "In a Colab cell, run:\n"
        "  !pip install -q ecos\n"
        "  !pip install -q "
        "git+https://github.com/Abzinger/BROJA_2PID.git@master\n"
        "Puis relancez ce script."
    ) from import_error


# =============================================================================
# 0. CONFIGURATION
# =============================================================================

DEFAULT_RUN_DIR = (
)

RUN_DIR = Path(os.environ.get("RUN_DIR", DEFAULT_RUN_DIR))
RESULTS_DIR = RUN_DIR / "results"
CKPT_DIR = RUN_DIR / "checkpoints"
EVAL_DIR = RESULTS_DIR / "model_evaluation"

OUT_DIR = EVAL_DIR / "pid_imin_broja_fixed_models"
OOF_UNIT_DIR = OUT_DIR / "oof_single_units"
BOOT_DIR = OUT_DIR / "bootstrap_checkpoints"

for directory in (OUT_DIR, OOF_UNIT_DIR, BOOT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get("SEED", "42"))

# Doit reproduire exactement le script evaluation initial.
MODEL_SEED_MODE = os.environ.get(
    "MODEL_SEED_MODE",
    "repeat",
).strip().lower()

VALID_SEED_MODES = {
    "repeat",
    "split",
    "repeat_fold",
    "constant",
}

if MODEL_SEED_MODE not in VALID_SEED_MODES:
    raise ValueError(
        f"MODEL_SEED_MODE={MODEL_SEED_MODE!r} invalide. "
        f"Allowed values:  {sorted(VALID_SEED_MODES)}"
    )

LAYERS = ["mutations", "cnv", "mrna", "rppa"]

SHORT = {
    "mutations": "Mutations",
    "cnv": "CNV",
    "mrna": "mRNA",
    "rppa": "RPPA",
}

CLASSIFIERS = ["RF", "XGB", "SVM"]
PAIRS = list(combinations(LAYERS, 2))

# Main bootstrap. Resuming allows running 100, then 250,
# 500 and finally 1,000 without losing earlier computations.
N_PATIENT_BOOTSTRAP = int(
    os.environ.get("PID_BOOTSTRAP_REPLICATES", "1000")
)

N_REPEAT_BOOTSTRAP = int(
    os.environ.get("REPEAT_BOOTSTRAP_REPLICATES", "10000")
)

BOOTSTRAP_SAVE_EVERY = int(
    os.environ.get("PID_BOOTSTRAP_SAVE_EVERY", "5")
)

BOOTSTRAP_CONVERGENCE_POINTS = sorted(set(
    point
    for point in [50, 100, 250, 500, 1000]
    if point <= N_PATIENT_BOOTSTRAP
))

METRIC_TOLERANCE = float(
    os.environ.get("METRIC_MATCH_TOLERANCE", "0.0001")
)

STRICT_METRIC_MATCH = (
    os.environ.get("STRICT_METRIC_MATCH", "1").strip() != "0"
)

BROJA_ABSTOL = float(
    os.environ.get("BROJA_ABSTOL", "1e-9")
)
BROJA_RELTOL = float(
    os.environ.get("BROJA_RELTOL", "1e-9")
)
BROJA_FEASTOL = float(
    os.environ.get("BROJA_FEASTOL", "1e-9")
)
BROJA_MAX_ITERS = int(
    os.environ.get("BROJA_MAX_ITERS", "1000")
)

# This threshold does not change the results: it only decides whether a
# solution counts as numerically valid in the summaries.
BROJA_MAX_NUMERICAL_ERROR = float(
    os.environ.get("BROJA_MAX_NUMERICAL_ERROR", "1e-6")
)

BROJA_MAX_DECOMPOSITION_RESIDUAL = float(
    os.environ.get("BROJA_MAX_DECOMPOSITION_RESIDUAL", "1e-5")
)

RUN_SELF_TESTS = (
    os.environ.get("RUN_PID_SELF_TESTS", "1").strip() != "0"
)

# Fixed hyperparameters: must match the script already run.
EVAL_CONFIG = {
    "random_forest": {
        "n_estimators": 500,
        "max_features": "sqrt",
        "min_samples_leaf": 2,
        "n_jobs": -1,
    },
    "xgboost": {
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.03,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0,
        "min_child_weight": 1.0,
        "n_jobs": -1,
    },
    "svm": {
        "C": 1.0,
        "kernel": "rbf",
        "gamma": "scale",
        "class_weight": "balanced",
        "probability": False,
    },
}


# =============================================================================
# 1. GENERAL UTILITIES
# =============================================================================

def log(message: str = "") -> None:
    print(message)


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(payload: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as handle:
        json.dump(
            payload,
            handle,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    os.replace(tmp, path)


def atomic_pickle(payload: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as handle:
        pickle.dump(
            payload,
            handle,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    os.replace(tmp, path)


def stable_seed(*parts: Any) -> int:
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def canonical_json_hash(payload: Any) -> str:
    text = json.dumps(
        payload,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def find_existing(candidates: List[Path]) -> Path:
    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "No file found among: \n"
        + "\n".join(str(path) for path in candidates)
    )


def detect_column(
    frame: pd.DataFrame,
    candidates: Iterable[str],
    required: bool = True,
) -> Optional[str]:
    lower_map = {
        str(column).lower(): column
        for column in frame.columns
    }

    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    if required:
        raise RuntimeError(
            f"No column found among {list(candidates)}.\n"
            f"Colonnes disponibles : {list(frame.columns)}"
        )

    return None


def normalize_classifier(value: Any) -> str:
    text = str(value).strip().upper()

    aliases = {
        "RANDOMFOREST": "RF",
        "RANDOM_FOREST": "RF",
        "RANDOM FOREST": "RF",
        "XGBOOST": "XGB",
        "SVC": "SVM",
    }

    return aliases.get(text, text)


def normalize_panel(value: Any) -> str:
    text = str(value).strip().lower()

    aliases = {
        "mutation": "mutations",
        "mut": "mutations",
        "muts": "mutations",
        "copy_number": "cnv",
        "copy-number": "cnv",
        "rna": "mrna",
        "protein": "rppa",
        "proteomics": "rppa",
        "baseline": "EMPTY",
        "empty": "EMPTY",
    }

    if text.upper() == "EMPTY":
        return "EMPTY"

    for separator in ("|", ",", ";", "__", "/", " "):
        text = text.replace(separator, "+")

    tokens = [
        aliases.get(token.strip(), token.strip())
        for token in text.split("+")
        if token.strip()
    ]

    if not tokens:
        return "EMPTY"

    order = {
        layer: index
        for index, layer in enumerate(LAYERS)
    }

    unknown = [
        token for token in tokens
        if token not in order
    ]

    if unknown:
        raise ValueError(
            f"Panel non reconnu : {value!r}; "
            f"tokens inconnus={unknown}"
        )

    tokens = sorted(
        set(tokens),
        key=lambda token: order[token],
    )

    return "+".join(tokens)


def model_seed(
    split_index: int,
    repeat_index: int,
    fold_index: int,
) -> int:
    if MODEL_SEED_MODE == "repeat":
        return int(SEED + repeat_index)

    if MODEL_SEED_MODE == "split":
        return int(SEED + split_index)

    if MODEL_SEED_MODE == "repeat_fold":
        return int(
            SEED + 1000 * repeat_index + fold_index
        )

    return int(SEED)


def bootstrap_mean_ci(
    values: np.ndarray,
    n_boot: int,
    seed: int,
) -> Tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)

    sample_indices = rng.integers(
        0,
        len(values),
        size=(n_boot, len(values)),
    )

    means = values[sample_indices].mean(axis=1)

    return (
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )


def exact_sign_flip_pvalue(
    differences: np.ndarray,
) -> float:
    differences = np.asarray(
        differences,
        dtype=float,
    )
    differences = differences[
        np.isfinite(differences)
    ]

    if len(differences) == 0:
        return np.nan

    observed = abs(float(differences.mean()))
    n_values = len(differences)

    null_values = np.empty(
        2 ** n_values,
        dtype=float,
    )

    for mask in range(2 ** n_values):
        signs = np.asarray([
            1.0 if ((mask >> bit) & 1) else -1.0
            for bit in range(n_values)
        ])

        null_values[mask] = abs(
            float(np.mean(signs * differences))
        )

    return float(
        np.mean(
            null_values >= observed - 1e-12
        )
    )


# =============================================================================
# 2. INFORMATION-THEORETIC UTILITIES - IN BITS
# =============================================================================

def discrete_mutual_information_bits(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    first = np.asarray(first)
    second = np.asarray(second)

    if len(first) != len(second):
        raise ValueError(
            "Variables must have the same length."
        )

    n_observations = len(first)

    if n_observations == 0:
        return np.nan

    joint_counts = Counter(
        zip(first.tolist(), second.tolist())
    )
    first_counts = Counter(first.tolist())
    second_counts = Counter(second.tolist())

    mutual_information = 0.0

    for (first_value, second_value), count in (
        joint_counts.items()
    ):
        p_joint = count / n_observations
        p_first = (
            first_counts[first_value]
            / n_observations
        )
        p_second = (
            second_counts[second_value]
            / n_observations
        )

        mutual_information += (
            p_joint
            * math.log2(
                p_joint / (p_first * p_second)
            )
        )

    return float(mutual_information)


def joint_discrete_code(
    first: np.ndarray,
    second: np.ndarray,
) -> np.ndarray:
    pairs = list(
        zip(
            np.asarray(first).tolist(),
            np.asarray(second).tolist(),
        )
    )

    mapping = {
        value: index
        for index, value in enumerate(
            sorted(set(pairs))
        )
    }

    return np.asarray(
        [mapping[value] for value in pairs],
        dtype=int,
    )


def specific_information_bits(
    target: np.ndarray,
    source: np.ndarray,
) -> Dict[Any, float]:
    """
    Williams–Beer specific information :

        I(T=t ; S)
        = sum_s p(s|t) log2[p(s|t)/p(s)]

    This quantity is computed separately for each target state t.
    """
    target = np.asarray(target)
    source = np.asarray(source)

    if len(target) != len(source):
        raise ValueError(
            "target and source must have the same length."
        )

    n_observations = len(target)

    target_counts = Counter(target.tolist())
    source_counts = Counter(source.tolist())
    joint_counts = Counter(
        zip(target.tolist(), source.tolist())
    )

    output: Dict[Any, float] = {}

    for target_value, target_count in target_counts.items():
        specific_information = 0.0

        for source_value, source_count in source_counts.items():
            joint_count = joint_counts.get(
                (target_value, source_value),
                0,
            )

            if joint_count == 0:
                continue

            p_source_given_target = (
                joint_count / target_count
            )
            p_source = (
                source_count / n_observations
            )

            specific_information += (
                p_source_given_target
                * math.log2(
                    p_source_given_target / p_source
                )
            )

        output[target_value] = float(
            specific_information
        )

    return output


def pid_williams_beer_imin(
    source_a: np.ndarray,
    source_b: np.ndarray,
    target: np.ndarray,
) -> Dict[str, float]:
    """
    True Williams-Beer I_min PID, not MMI.

        R_Imin = sum_t p(t) min[
            I(T=t; A),
            I(T=t; B)
        ]
    """
    source_a = np.asarray(source_a)
    source_b = np.asarray(source_b)
    target = np.asarray(target)

    if not (
        len(source_a)
        == len(source_b)
        == len(target)
    ):
        raise ValueError(
            "All three variables must have the same length."
        )

    n_observations = len(target)
    target_counts = Counter(target.tolist())

    specific_a = specific_information_bits(
        target,
        source_a,
    )
    specific_b = specific_information_bits(
        target,
        source_b,
    )

    redundancy = 0.0

    for target_value, count in target_counts.items():
        p_target = count / n_observations

        redundancy += (
            p_target
            * min(
                specific_a[target_value],
                specific_b[target_value],
            )
        )

    mi_a = discrete_mutual_information_bits(
        target,
        source_a,
    )
    mi_b = discrete_mutual_information_bits(
        target,
        source_b,
    )
    mi_joint = discrete_mutual_information_bits(
        target,
        joint_discrete_code(
            source_a,
            source_b,
        ),
    )

    unique_a = mi_a - redundancy
    unique_b = mi_b - redundancy

    synergy = (
        mi_joint
        - redundancy
        - unique_a
        - unique_b
    )

    # Theoretical values are non-negative.
    # On corrige uniquement le bruit d'arrondi microscopique.
    components = {
        "redundancy": redundancy,
        "unique_a": unique_a,
        "unique_b": unique_b,
        "synergy": synergy,
    }

    for key, value in list(components.items()):
        if -1e-12 < value < 0:
            components[key] = 0.0

    residual = (
        mi_joint
        - components["redundancy"]
        - components["unique_a"]
        - components["unique_b"]
        - components["synergy"]
    )

    return {
        "redundancy": float(
            components["redundancy"]
        ),
        "unique_a": float(
            components["unique_a"]
        ),
        "unique_b": float(
            components["unique_b"]
        ),
        "synergy": float(
            components["synergy"]
        ),
        "mi_a": float(mi_a),
        "mi_b": float(mi_b),
        "mi_joint": float(mi_joint),
        "decomposition_residual": float(
            residual
        ),
        "solver_status": "analytic",
        "valid": bool(
            np.isfinite([
                components["redundancy"],
                components["unique_a"],
                components["unique_b"],
                components["synergy"],
            ]).all()
        ),
    }


def empirical_pdf_target_source_source(
    target: np.ndarray,
    source_a: np.ndarray,
    source_b: np.ndarray,
) -> Dict[Tuple[int, int, int], float]:
    """
    BROJA attend un dictionnaire :
        (X, Y, Z) -> p(x,y,z)

    Ici :
        X = cible PAM50
        Y = prediction de la couche A
        Z = prediction de la couche B
    """
    target = np.asarray(target, dtype=int)
    source_a = np.asarray(source_a, dtype=int)
    source_b = np.asarray(source_b, dtype=int)

    if not (
        len(target)
        == len(source_a)
        == len(source_b)
    ):
        raise ValueError(
            "All three variables must have the same length."
        )

    counts = Counter(
        zip(
            target.tolist(),
            source_a.tolist(),
            source_b.tolist(),
        )
    )

    total = float(len(target))

    pdf = {
        tuple(map(int, key)): float(count / total)
        for key, count in counts.items()
        if count > 0
    }

    # Defensive renormalisation to satisfy the package tolerance.
    pdf_sum = float(sum(pdf.values()))

    pdf = {
        key: float(value / pdf_sum)
        for key, value in pdf.items()
    }

    return pdf


def pid_broja(
    source_a: np.ndarray,
    source_b: np.ndarray,
    target: np.ndarray,
) -> Dict[str, Any]:
    """
    BROJA-2PID true, via le solveur officiel ECOS.

    Mapping du package :
        SI  -> shared information -> redundancy
        UIY -> unique information of source A
        UIZ -> unique information of source B
        CI  -> complementary information -> synergy
    """
    started_at = time.time()

    try:
        pdf = empirical_pdf_target_source_source(
            target,
            source_a,
            source_b,
        )

        result = BROJA_2PID.pid(
            pdf,
            cone_solver="ECOS",
            output=0,
            abstol=BROJA_ABSTOL,
            reltol=BROJA_RELTOL,
            feastol=BROJA_FEASTOL,
            max_iters=BROJA_MAX_ITERS,
        )

        redundancy = float(result["SI"])
        unique_a = float(result["UIY"])
        unique_b = float(result["UIZ"])
        synergy = float(result["CI"])

        numerical_errors = tuple(
            float(value)
            for value in result.get(
                "Num_err",
                (np.nan, np.nan, np.nan),
            )
        )

        max_numerical_error = float(
            np.nanmax(
                np.abs(
                    np.asarray(
                        numerical_errors,
                        dtype=float,
                    )
                )
            )
        )

        mi_a = discrete_mutual_information_bits(
            target,
            source_a,
        )
        mi_b = discrete_mutual_information_bits(
            target,
            source_b,
        )
        mi_joint = discrete_mutual_information_bits(
            target,
            joint_discrete_code(
                source_a,
                source_b,
            ),
        )

        residual = (
            mi_joint
            - redundancy
            - unique_a
            - unique_b
            - synergy
        )

        finite = bool(
            np.isfinite([
                redundancy,
                unique_a,
                unique_b,
                synergy,
                mi_a,
                mi_b,
                mi_joint,
                residual,
                max_numerical_error,
            ]).all()
        )

        valid = bool(
            finite
            and max_numerical_error
            <= BROJA_MAX_NUMERICAL_ERROR
            and abs(residual)
            <= BROJA_MAX_DECOMPOSITION_RESIDUAL
        )

        return {
            "redundancy": redundancy,
            "unique_a": unique_a,
            "unique_b": unique_b,
            "synergy": synergy,
            "mi_a": float(mi_a),
            "mi_b": float(mi_b),
            "mi_joint": float(mi_joint),
            "decomposition_residual": float(
                residual
            ),
            "solver_status": (
                "valid"
                if valid
                else "numerical_warning"
            ),
            "solver_name": str(
                result.get("Solver", "ECOS")
            ),
            "primal_infeasibility": (
                numerical_errors[0]
            ),
            "dual_infeasibility": (
                numerical_errors[1]
            ),
            "duality_gap_proxy": (
                numerical_errors[2]
            ),
            "max_numerical_error": (
                max_numerical_error
            ),
            "elapsed_seconds": float(
                time.time() - started_at
            ),
            "support_size": int(len(pdf)),
            "valid": valid,
            "error_type": None,
            "error_message": None,
        }

    except Exception as error:
        return {
            "redundancy": np.nan,
            "unique_a": np.nan,
            "unique_b": np.nan,
            "synergy": np.nan,
            "mi_a": np.nan,
            "mi_b": np.nan,
            "mi_joint": np.nan,
            "decomposition_residual": np.nan,
            "solver_status": "failed",
            "solver_name": "ECOS",
            "primal_infeasibility": np.nan,
            "dual_infeasibility": np.nan,
            "duality_gap_proxy": np.nan,
            "max_numerical_error": np.nan,
            "elapsed_seconds": float(
                time.time() - started_at
            ),
            "support_size": np.nan,
            "valid": False,
            "error_type": type(error).__name__,
            "error_message": str(error),
        }


# =============================================================================
# 3. AUTOTESTS PID
# =============================================================================

def run_pid_self_tests() -> pd.DataFrame:
    """
    Trois portes classiques :

    1. REDUNDANT COPY :
       T=A=B -> redondance ~1 bit.

    2. XOR :
       T=A xor B -> synergie ~1 bit.

    3. TWO-BIT COPY :
       T=(A,B).
       BROJA : unique A ~1 et unique B ~1.
       I_min : redondance ~1 et synergie ~1
       (known identity pathology of I_min).
    """
    rows = []

    repetitions = 128

    # Redundant copy
    target = np.tile([0, 0, 1, 1], repetitions)
    source_a = target.copy()
    source_b = target.copy()

    test_cases = [
        (
            "redundant_copy",
            source_a,
            source_b,
            target,
        )
    ]

    # XOR
    source_a = np.tile([0, 0, 1, 1], repetitions)
    source_b = np.tile([0, 1, 0, 1], repetitions)
    target = source_a ^ source_b

    test_cases.append(
        ("xor", source_a, source_b, target)
    )

    # Target encodes two independent bits.
    source_a = np.tile([0, 0, 1, 1], repetitions)
    source_b = np.tile([0, 1, 0, 1], repetitions)
    target = 2 * source_a + source_b

    test_cases.append(
        ("two_bit_copy", source_a, source_b, target)
    )

    for test_name, a_values, b_values, target_values in test_cases:
        for estimator_name, function in [
            ("I_min", pid_williams_beer_imin),
            ("BROJA", pid_broja),
        ]:
            result = function(
                a_values,
                b_values,
                target_values,
            )

            rows.append({
                "test": test_name,
                "estimator": estimator_name,
                "redundancy": result["redundancy"],
                "unique_a": result["unique_a"],
                "unique_b": result["unique_b"],
                "synergy": result["synergy"],
                "mi_joint": result["mi_joint"],
                "residual": result[
                    "decomposition_residual"
                ],
                "valid": result["valid"],
                "solver_status": result[
                    "solver_status"
                ],
            })

    tests = pd.DataFrame(rows)

    # Tests BROJA minimaux.
    broja_copy = tests[
        (tests["test"] == "redundant_copy")
        & (tests["estimator"] == "BROJA")
    ].iloc[0]

    broja_xor = tests[
        (tests["test"] == "xor")
        & (tests["estimator"] == "BROJA")
    ].iloc[0]

    broja_two_bit = tests[
        (tests["test"] == "two_bit_copy")
        & (tests["estimator"] == "BROJA")
    ].iloc[0]

    assertions = [
        abs(broja_copy["redundancy"] - 1.0) < 1e-4,
        abs(broja_copy["synergy"]) < 1e-4,
        abs(broja_xor["synergy"] - 1.0) < 1e-4,
        abs(broja_two_bit["unique_a"] - 1.0) < 1e-4,
        abs(broja_two_bit["unique_b"] - 1.0) < 1e-4,
    ]

    if not all(assertions):
        raise RuntimeError(
            "BROJA self-tests failed. "
            "Check the ECOS and BROJA-2PID installation."
        )

    atomic_csv(
        tests,
        OUT_DIR / "pid_self_tests.csv",
    )

    return tests


if RUN_SELF_TESTS:
    log("Autotests I_min/BROJA...")
    self_tests = run_pid_self_tests()
    log("Self-tests passed.")


# =============================================================================
# 4. PROVENANCE ET CONFIGURATION
# =============================================================================

try:
    broja_version = importlib.metadata.version(
        "broja2pid"
    )
except Exception:
    broja_version = "unknown"

try:
    ecos_version = importlib.metadata.version("ecos")
except Exception:
    ecos_version = "unknown"

CONFIG = {
    "run_dir": str(RUN_DIR),
    "seed": SEED,
    "model_seed_mode": MODEL_SEED_MODE,
    "layers": LAYERS,
    "classifiers": CLASSIFIERS,
    "fixed_model_hyperparameters": EVAL_CONFIG,
    "metric_match_tolerance": METRIC_TOLERANCE,
    "strict_metric_match": STRICT_METRIC_MATCH,
    "repeat_bootstrap_replicates": (
        N_REPEAT_BOOTSTRAP
    ),
    "patient_cluster_bootstrap_replicates": (
        N_PATIENT_BOOTSTRAP
    ),
    "bootstrap_convergence_points": (
        BOOTSTRAP_CONVERGENCE_POINTS
    ),
    "broja": {
        "package_version": broja_version,
        "ecos_version": ecos_version,
        "abstol": BROJA_ABSTOL,
        "reltol": BROJA_RELTOL,
        "feastol": BROJA_FEASTOL,
        "max_iters": BROJA_MAX_ITERS,
        "max_numerical_error_for_validity": (
            BROJA_MAX_NUMERICAL_ERROR
        ),
        "max_decomposition_residual_for_validity": (
            BROJA_MAX_DECOMPOSITION_RESIDUAL
        ),
    },
    "estimators": {
        "I_min": (
            "Williams-Beer specific-information redundancy"
        ),
        "BROJA": (
            "BROJA-2PID exponential-cone estimator"
        ),
    },
    "bootstrap_estimand": (
        "pooled-repeat prediction-level PID with "
        "stratified patient-cluster resampling"
    ),
}

# The scientific identity of the checkpoint excludes only the requested number
# of replicates. This allows extending 100 -> 250 -> 500 -> 1000 without
# invalidating draws already computed. Any change to data, seeds,
# hyperparameters, estimateurs ou tolerances invalide toujours le checkpoint.
ANALYSIS_IDENTITY = dict(CONFIG)
ANALYSIS_IDENTITY.pop("repeat_bootstrap_replicates", None)
ANALYSIS_IDENTITY.pop("patient_cluster_bootstrap_replicates", None)
ANALYSIS_IDENTITY.pop("bootstrap_convergence_points", None)
CONFIG_HASH = canonical_json_hash(ANALYSIS_IDENTITY)
CONFIG["analysis_hash"] = CONFIG_HASH

atomic_json(
    CONFIG,
    OUT_DIR / "pid_config.json",
)


# =============================================================================
# 5. CHARGEMENT DU CACHE, COHORTE, SPLITS ET METRICS
# =============================================================================

cache_path = find_existing([
    CKPT_DIR / "ckpt_preprocessing_cache.pkl",
    CKPT_DIR / "preprocessing_cache.pkl",
])

cohort_path = find_existing([
    RESULTS_DIR / "cohort_manifest.csv",
    RESULTS_DIR / "cohort.csv",
])

splits_path = find_existing([
    RESULTS_DIR / "cv_splits.json",
])

fold_results_path = find_existing([
    EVAL_DIR / "panel_fold_results.csv",
    RESULTS_DIR / "panel_fold_results.csv",
])

with open(cache_path, "rb") as handle:
    cache_payload = pickle.load(handle)

if (
    isinstance(cache_payload, dict)
    and "data" in cache_payload
):
    cache_data = cache_payload["data"]
else:
    cache_data = cache_payload

if "representations" not in cache_data:
    raise RuntimeError(
        "The preprocessing checkpoint does not contain "
        "the 'representations' key."
    )

REPRESENTATIONS = cache_data["representations"]

cohort = pd.read_csv(cohort_path)

patient_column = detect_column(
    cohort,
    ["patient_id", "patient", "case_id"],
)

label_column = detect_column(
    cohort,
    ["label_encoded", "y", "target_encoded"],
)

patient_ids = (
    cohort[patient_column]
    .astype(str)
    .to_numpy()
)

y = pd.to_numeric(
    cohort[label_column],
    errors="raise",
).to_numpy(dtype=int)

N_PATIENTS = len(y)
N_CLASSES = len(np.unique(y))

with open(splits_path, encoding="utf-8") as handle:
    split_payload = json.load(handle)

split_payload = sorted(
    split_payload,
    key=lambda row: int(row["split"]),
)

SPLITS = [
    (
        np.asarray(
            row["train_indices"],
            dtype=int,
        ),
        np.asarray(
            row["test_indices"],
            dtype=int,
        ),
    )
    for row in split_payload
]

REPEAT_OF = [
    int(row["repeat"])
    for row in split_payload
]

FOLD_OF = [
    int(row["fold"])
    for row in split_payload
]

if len(SPLITS) != 50:
    raise RuntimeError(
        f"50 splits attendus, obtenu={len(SPLITS)}"
    )

for split_index in range(len(SPLITS)):
    for layer in LAYERS:
        if (
            split_index,
            layer,
        ) not in REPRESENTATIONS:
            raise RuntimeError(
                "Missing representation:  "
                f"split={split_index}, layer={layer}"
            )

raw_fold_results = pd.read_csv(
    fold_results_path
)

classifier_column = detect_column(
    raw_fold_results,
    ["classifier", "model", "classifier_name"],
)

split_column = detect_column(
    raw_fold_results,
    ["split", "outer_split", "split_index"],
)

repeat_column = detect_column(
    raw_fold_results,
    ["repeat", "repeat_index", "cv_repeat"],
)

fold_column = detect_column(
    raw_fold_results,
    ["fold", "fold_index", "cv_fold"],
)

panel_column = detect_column(
    raw_fold_results,
    ["panel", "panel_key", "combination"],
)

ba_column = detect_column(
    raw_fold_results,
    ["balanced_accuracy", "balanced_acc", "ba"],
)

f1_column = detect_column(
    raw_fold_results,
    ["macro_f1", "f1_macro", "macro-f1"],
)

accuracy_column = detect_column(
    raw_fold_results,
    ["accuracy", "acc"],
)

fold_results = pd.DataFrame({
    "classifier": (
        raw_fold_results[classifier_column]
        .map(normalize_classifier)
    ),
    "split": pd.to_numeric(
        raw_fold_results[split_column],
        errors="raise",
    ).astype(int),
    "repeat": pd.to_numeric(
        raw_fold_results[repeat_column],
        errors="raise",
    ).astype(int),
    "fold": pd.to_numeric(
        raw_fold_results[fold_column],
        errors="raise",
    ).astype(int),
    "panel": (
        raw_fold_results[panel_column]
        .map(normalize_panel)
    ),
    "balanced_accuracy": pd.to_numeric(
        raw_fold_results[ba_column],
        errors="raise",
    ),
    "macro_f1": pd.to_numeric(
        raw_fold_results[f1_column],
        errors="raise",
    ),
    "accuracy": pd.to_numeric(
        raw_fold_results[accuracy_column],
        errors="raise",
    ),
})

unknown_classifiers = (
    set(fold_results["classifier"])
    - set(CLASSIFIERS)
)

if unknown_classifiers:
    raise RuntimeError(
        "Classifieurs non reconnus : "
        f"{unknown_classifiers}"
    )

atomic_csv(
    fold_results,
    OUT_DIR / "normalized_panel_fold_results.csv",
)

log(f"RUN_DIR : {RUN_DIR}")
log(f"Cohorte : {N_PATIENTS} patientes")
log(f"Classes : {N_CLASSES}")
log(f"BROJA-2PID : {broja_version}")
log(f"ECOS : {ecos_version}")


# =============================================================================
# 6. RPRED FROM THE RECORDED METRICS
# =============================================================================

repeat_metrics = (
    fold_results[
        fold_results["panel"] != "EMPTY"
    ]
    .groupby(
        ["classifier", "repeat", "panel"],
        as_index=False,
    )[[
        "balanced_accuracy",
        "macro_f1",
        "accuracy",
    ]]
    .mean()
)

baseline_repeat = (
    fold_results[
        fold_results["panel"] == "EMPTY"
    ]
    .groupby(
        ["classifier", "repeat"],
        as_index=False,
    )["balanced_accuracy"]
    .mean()
    .rename(columns={
        "balanced_accuracy": (
            "baseline_balanced_accuracy"
        )
    })
)


def get_repeat_panel_scores(
    classifier: str,
    panel: str,
) -> pd.Series:
    return (
        repeat_metrics[
            (repeat_metrics["classifier"] == classifier)
            & (repeat_metrics["panel"] == panel)
        ]
        .set_index("repeat")[
            "balanced_accuracy"
        ]
    )


def rpred_value(
    pair_score: float,
    single_a_score: float,
    single_b_score: float,
    full_score: float,
    baseline_score: float,
) -> float:
    denominator = (
        full_score - baseline_score
    )

    if abs(denominator) < 1e-8:
        return np.nan

    return float(
        1.0
        - (
            pair_score
            - max(
                single_a_score,
                single_b_score,
            )
        )
        / denominator
    )


full_panel_key = "+".join(LAYERS)
rpred_repeat_rows = []

for classifier in CLASSIFIERS:
    full_scores = get_repeat_panel_scores(
        classifier,
        full_panel_key,
    )

    baseline_by_repeat = (
        baseline_repeat[
            baseline_repeat["classifier"]
            == classifier
        ]
        .set_index("repeat")[
            "baseline_balanced_accuracy"
        ]
    )

    for layer_a, layer_b in PAIRS:
        pair_key = "+".join(
            sorted(
                [layer_a, layer_b],
                key=LAYERS.index,
            )
        )

        pair_scores = get_repeat_panel_scores(
            classifier,
            pair_key,
        )

        single_a_scores = (
            get_repeat_panel_scores(
                classifier,
                layer_a,
            )
        )

        single_b_scores = (
            get_repeat_panel_scores(
                classifier,
                layer_b,
            )
        )

        shared_repeats = sorted(
            set(pair_scores.index)
            & set(single_a_scores.index)
            & set(single_b_scores.index)
            & set(full_scores.index)
        )

        for repeat_index in shared_repeats:
            if (
                repeat_index
                in baseline_by_repeat.index
            ):
                baseline_score = float(
                    baseline_by_repeat.loc[
                        repeat_index
                    ]
                )
            else:
                baseline_score = (
                    1.0 / N_CLASSES
                )

            rpred_repeat_rows.append({
                "classifier": classifier,
                "repeat": int(repeat_index),
                "pair": pair_key,
                "pair_label": (
                    f"{SHORT[layer_a]}+"
                    f"{SHORT[layer_b]}"
                ),
                "rpred": rpred_value(
                    pair_score=float(
                        pair_scores.loc[
                            repeat_index
                        ]
                    ),
                    single_a_score=float(
                        single_a_scores.loc[
                            repeat_index
                        ]
                    ),
                    single_b_score=float(
                        single_b_scores.loc[
                            repeat_index
                        ]
                    ),
                    full_score=float(
                        full_scores.loc[
                            repeat_index
                        ]
                    ),
                    baseline_score=baseline_score,
                ),
            })

rpred_repeat = pd.DataFrame(
    rpred_repeat_rows
)

atomic_csv(
    rpred_repeat,
    OUT_DIR / "rpred_repeat_level.csv",
)

rpred_summary_rows = []

for keys, group in rpred_repeat.groupby(
    ["classifier", "pair", "pair_label"]
):
    values = group["rpred"].to_numpy(
        dtype=float
    )
    values = values[np.isfinite(values)]

    ci_low, ci_high = bootstrap_mean_ci(
        values,
        n_boot=N_REPEAT_BOOTSTRAP,
        seed=stable_seed("rpred", *keys),
    )

    differences_from_one = (
        values - 1.0
    )

    rpred_summary_rows.append({
        "classifier": keys[0],
        "pair": keys[1],
        "pair_label": keys[2],
        "rpred_repeat_mean": float(
            values.mean()
        ),
        "rpred_repeat_sd": float(
            values.std(ddof=1)
        ),
        "ci95_low": ci_low,
        "ci95_high": ci_high,
        "ci_excludes_1": bool(
            ci_high < 1.0
            or ci_low > 1.0
        ),
        "interpretation": (
            "complementarity"
            if ci_high < 1.0
            else (
                "interference"
                if ci_low > 1.0
                else "compatible_with_redundancy"
            )
        ),
        "exact_sign_flip_p_vs_1": (
            exact_sign_flip_pvalue(
                differences_from_one
            )
        ),
        "n_valid_repeats": int(
            len(values)
        ),
    })

rpred_summary = pd.DataFrame(
    rpred_summary_rows
)

rpred_summary["q_bh"] = np.nan

for classifier, group in (
    rpred_summary.groupby("classifier")
):
    indices = group.index

    rpred_summary.loc[
        indices,
        "q_bh",
    ] = multipletests(
        group[
            "exact_sign_flip_p_vs_1"
        ].to_numpy(dtype=float),
        method="fdr_bh",
    )[1]

atomic_csv(
    rpred_summary,
    OUT_DIR / "rpred_repeat_summary.csv",
)


# =============================================================================
# 7. RECONSTRUCTION OOF DES QUATRE SINGLE-OMICS
# =============================================================================

MODEL_CONFIG_HASH = canonical_json_hash({
    "seed": SEED,
    "seed_mode": MODEL_SEED_MODE,
    "hyperparameters": EVAL_CONFIG,
})


def make_classifier(
    classifier: str,
    random_seed: int,
):
    if classifier == "RF":
        return RandomForestClassifier(
            random_state=random_seed,
            **EVAL_CONFIG["random_forest"],
        )

    if classifier == "XGB":
        return XGBClassifier(
            objective="multi:softprob",
            num_class=N_CLASSES,
            eval_metric="mlogloss",
            random_state=random_seed,
            verbosity=0,
            **EVAL_CONFIG["xgboost"],
        )

    if classifier == "SVM":
        return SVC(
            random_state=random_seed,
            **EVAL_CONFIG["svm"],
        )

    raise ValueError(classifier)


def oof_unit_path(
    split_index: int,
    classifier: str,
    layer: str,
) -> Path:
    return (
        OOF_UNIT_DIR
        / (
            f"split_{split_index:02d}"
            f"__{classifier}__{layer}.csv"
        )
    )


def validate_oof_unit(
    frame: pd.DataFrame,
    split_index: int,
    classifier: str,
    layer: str,
    expected_test_index: np.ndarray,
) -> bool:
    required_columns = {
        "split",
        "repeat",
        "fold",
        "classifier",
        "layer",
        "patient_index",
        "patient_id",
        "y_true",
        "y_pred",
    }

    if not required_columns.issubset(
        frame.columns
    ):
        return False

    if len(frame) != len(expected_test_index):
        return False

    if frame["patient_index"].duplicated().any():
        return False

    if set(
        frame["patient_index"].astype(int)
    ) != set(
        expected_test_index.astype(int)
    ):
        return False

    if set(
        frame["patient_id"].astype(str)
    ) != set(
        patient_ids[expected_test_index]
    ):
        return False

    if (
        "model_config_hash" in frame.columns
        and not (
            frame["model_config_hash"]
            .astype(str)
            .eq(MODEL_CONFIG_HASH)
            .all()
        )
    ):
        return False

    return bool(
        frame["split"].astype(int)
        .eq(split_index)
        .all()
        and frame["classifier"]
        .astype(str)
        .eq(classifier)
        .all()
        and frame["layer"]
        .astype(str)
        .eq(layer)
        .all()
    )


prediction_unit_frames = []
audit_rows = []

for split_index, (
    train_index,
    test_index,
) in enumerate(SPLITS):
    repeat_index = int(
        REPEAT_OF[split_index]
    )
    fold_index = int(
        FOLD_OF[split_index]
    )

    current_seed = model_seed(
        split_index,
        repeat_index,
        fold_index,
    )

    y_train = y[train_index]
    y_test = y[test_index]

    sample_weights = (
        compute_sample_weight(
            class_weight="balanced",
            y=y_train,
        )
    )

    for classifier in CLASSIFIERS:
        for layer in LAYERS:
            path = oof_unit_path(
                split_index,
                classifier,
                layer,
            )

            unit_frame = None

            if path.exists():
                candidate = pd.read_csv(path)

                if validate_oof_unit(
                    candidate,
                    split_index,
                    classifier,
                    layer,
                    test_index,
                ):
                    unit_frame = candidate

            if unit_frame is None:
                X_train, X_test = (
                    REPRESENTATIONS[
                        (split_index, layer)
                    ]
                )

                model = make_classifier(
                    classifier,
                    current_seed,
                )

                if classifier in {"RF", "XGB"}:
                    model.fit(
                        X_train,
                        y_train,
                        sample_weight=sample_weights,
                    )
                else:
                    model.fit(
                        X_train,
                        y_train,
                    )

                predictions = (
                    model.predict(X_test)
                    .astype(int)
                )

                unit_frame = pd.DataFrame({
                    "split": split_index,
                    "repeat": repeat_index,
                    "fold": fold_index,
                    "classifier": classifier,
                    "layer": layer,
                    "patient_index": test_index,
                    "patient_id": (
                        patient_ids[test_index]
                    ),
                    "y_true": y_test,
                    "y_pred": predictions,
                    "model_seed": current_seed,
                    "model_seed_mode": (
                        MODEL_SEED_MODE
                    ),
                    "model_config_hash": (
                        MODEL_CONFIG_HASH
                    ),
                })

                atomic_csv(
                    unit_frame,
                    path,
                )

            reconstructed_ba = float(
                balanced_accuracy_score(
                    unit_frame["y_true"],
                    unit_frame["y_pred"],
                )
            )

            reconstructed_f1 = float(
                f1_score(
                    unit_frame["y_true"],
                    unit_frame["y_pred"],
                    average="macro",
                    zero_division=0,
                )
            )

            reconstructed_accuracy = float(
                accuracy_score(
                    unit_frame["y_true"],
                    unit_frame["y_pred"],
                )
            )

            recorded = fold_results[
                (
                    fold_results["split"]
                    == split_index
                )
                & (
                    fold_results["classifier"]
                    == classifier
                )
                & (
                    fold_results["panel"]
                    == layer
                )
            ]

            if len(recorded) != 1:
                raise RuntimeError(
                    "Single-omics metric missing or "
                    "duplicated:  "
                    f"split={split_index}, "
                    f"classifier={classifier}, "
                    f"layer={layer}"
                )

            recorded_row = recorded.iloc[0]

            audit_rows.append({
                "split": split_index,
                "repeat": repeat_index,
                "fold": fold_index,
                "classifier": classifier,
                "layer": layer,
                "recorded_balanced_accuracy": (
                    float(
                        recorded_row[
                            "balanced_accuracy"
                        ]
                    )
                ),
                "reconstructed_balanced_accuracy": (
                    reconstructed_ba
                ),
                "absolute_difference_balanced_accuracy": (
                    abs(
                        reconstructed_ba
                        - float(
                            recorded_row[
                                "balanced_accuracy"
                            ]
                        )
                    )
                ),
                "recorded_macro_f1": float(
                    recorded_row["macro_f1"]
                ),
                "reconstructed_macro_f1": (
                    reconstructed_f1
                ),
                "absolute_difference_macro_f1": (
                    abs(
                        reconstructed_f1
                        - float(
                            recorded_row[
                                "macro_f1"
                            ]
                        )
                    )
                ),
                "recorded_accuracy": float(
                    recorded_row["accuracy"]
                ),
                "reconstructed_accuracy": (
                    reconstructed_accuracy
                ),
                "absolute_difference_accuracy": (
                    abs(
                        reconstructed_accuracy
                        - float(
                            recorded_row[
                                "accuracy"
                            ]
                        )
                    )
                ),
            })

            prediction_unit_frames.append(
                unit_frame
            )

    log(
        "OOF single-omics : "
        f"split {split_index + 1}/"
        f"{len(SPLITS)}"
    )

audit = pd.DataFrame(audit_rows)

atomic_csv(
    audit,
    OUT_DIR / "metric_reconstruction_audit.csv",
)

metric_difference_columns = [
    "absolute_difference_balanced_accuracy",
    "absolute_difference_macro_f1",
    "absolute_difference_accuracy",
]

max_metric_difference = float(
    audit[metric_difference_columns]
    .to_numpy(dtype=float)
    .max()
)

metric_reconstruction_passed = bool(
    max_metric_difference
    <= METRIC_TOLERANCE
)

if (
    STRICT_METRIC_MATCH
    and not metric_reconstruction_passed
):
    raise RuntimeError(
        "The reconstructed predictions do not "
        "reproduce the metrics "
        "recorded.\n"
        f"Maximum difference={max_metric_difference:.8g}; "
        f"tolerance={METRIC_TOLERANCE}.\n"
        "Check MODEL_SEED_MODE and the "
        "fixed hyperparameters before any PID computation."
    )

oof_singles = pd.concat(
    prediction_unit_frames,
    ignore_index=True,
)

if oof_singles.duplicated([
    "split",
    "classifier",
    "layer",
    "patient_index",
]).any():
    raise RuntimeError(
        "Duplicated out-of-fold predictions."
    )

expected_oof_rows = (
    sum(
        len(test_index)
        for _, test_index in SPLITS
    )
    * len(CLASSIFIERS)
    * len(LAYERS)
)

if len(oof_singles) != expected_oof_rows:
    raise RuntimeError(
        "Unexpected number of out-of-fold predictions:  "
        f"{len(oof_singles)}/"
        f"{expected_oof_rows}"
    )

atomic_csv(
    oof_singles,
    OUT_DIR / "oof_single_omics_predictions.csv",
)


# =============================================================================
# 8. ALIGNMENT OF OUT-OF-FOLD PREDICTIONS
# =============================================================================

oof_by: Dict[
    Tuple[str, int, str],
    np.ndarray,
] = {}

for keys, group in oof_singles.groupby(
    ["classifier", "repeat", "layer"]
):
    group = group.sort_values(
        "patient_index"
    )

    if len(group) != N_PATIENTS:
        raise RuntimeError(
            f"OOF incomplet : {keys}, "
            f"n={len(group)}/{N_PATIENTS}"
        )

    if not np.array_equal(
        group["patient_index"]
        .to_numpy(dtype=int),
        np.arange(N_PATIENTS),
    ):
        raise RuntimeError(
            "Out-of-fold predictions not aligned with the cohort:  "
            f"{keys}"
        )

    if not np.array_equal(
        group["y_true"]
        .to_numpy(dtype=int),
        y,
    ):
        raise RuntimeError(
            f"Inconsistent out-of-fold labels:  {keys}"
        )

    oof_by[keys] = (
        group["y_pred"]
        .to_numpy(dtype=int)
    )

repeats = sorted(
    oof_singles["repeat"]
    .unique()
    .astype(int)
    .tolist()
)

if len(repeats) != 10:
    raise RuntimeError(
        "10 repetitions expected,  "
        f"obtenu={len(repeats)}"
    )


def pooled_repeat_arrays(
    classifier: str,
    layer_a: str,
    layer_b: str,
    patient_indices: Optional[np.ndarray] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Estimand pooled-repeat :

    The prediction of a model trained under one CV repetition is treated
    chosen among the ten repetitions. Patients are the clusters of the
    resampling; all their repeated predictions travel together.
    """
    if patient_indices is None:
        patient_indices = np.arange(
            N_PATIENTS
        )

    source_a_long = np.concatenate([
        oof_by[
            (classifier, repeat_index, layer_a)
        ][patient_indices]
        for repeat_index in repeats
    ])

    source_b_long = np.concatenate([
        oof_by[
            (classifier, repeat_index, layer_b)
        ][patient_indices]
        for repeat_index in repeats
    ])

    target_long = np.concatenate([
        y[patient_indices]
        for _ in repeats
    ])

    return (
        source_a_long,
        source_b_long,
        target_long,
    )


# =============================================================================
# 9. PID PAR REPETITION : I_min ET BROJA
# =============================================================================

pid_repeat_rows = []

for classifier in CLASSIFIERS:
    for layer_a, layer_b in PAIRS:
        pair_key = f"{layer_a}+{layer_b}"
        pair_display = (
            f"{SHORT[layer_a]}+"
            f"{SHORT[layer_b]}"
        )

        for repeat_index in repeats:
            source_a = oof_by[
                (
                    classifier,
                    repeat_index,
                    layer_a,
                )
            ]

            source_b = oof_by[
                (
                    classifier,
                    repeat_index,
                    layer_b,
                )
            ]

            for estimator, function in [
                (
                    "I_min",
                    pid_williams_beer_imin,
                ),
                (
                    "BROJA",
                    pid_broja,
                ),
            ]:
                result = function(
                    source_a,
                    source_b,
                    y,
                )

                pid_repeat_rows.append({
                    "estimator": estimator,
                    "classifier": classifier,
                    "repeat": repeat_index,
                    "pair": pair_key,
                    "pair_label": pair_display,
                    **result,
                })

        log(
            "PID per repetition:  "
            f"{classifier} | {pair_display}"
        )

pid_repeat = pd.DataFrame(
    pid_repeat_rows
)

atomic_csv(
    pid_repeat,
    OUT_DIR / "pid_repeat_level.csv",
)


# =============================================================================
# 10. PER-REPETITION SUMMARY WITH BOOTSTRAP OVER THE 10 REPETITIONS
# =============================================================================

PID_COMPONENTS = [
    "redundancy",
    "unique_a",
    "unique_b",
    "synergy",
    "mi_a",
    "mi_b",
    "mi_joint",
]

pid_repeat_summary_rows = []

valid_repeat_data = pid_repeat[
    pid_repeat["valid"].astype(bool)
].copy()

for keys, group in valid_repeat_data.groupby(
    [
        "estimator",
        "classifier",
        "pair",
        "pair_label",
    ]
):
    output_row = {
        "estimator": keys[0],
        "classifier": keys[1],
        "pair": keys[2],
        "pair_label": keys[3],
        "n_valid_repeats": int(
            group["repeat"].nunique()
        ),
    }

    for component in PID_COMPONENTS:
        values = group[component].to_numpy(
            dtype=float
        )
        values = values[
            np.isfinite(values)
        ]

        ci_low, ci_high = bootstrap_mean_ci(
            values,
            n_boot=N_REPEAT_BOOTSTRAP,
            seed=stable_seed(
                "repeat_ci",
                *keys,
                component,
            ),
        )

        output_row[
            f"{component}_mean"
        ] = float(values.mean())

        output_row[
            f"{component}_sd"
        ] = float(
            values.std(ddof=1)
        )

        output_row[
            f"{component}_ci95_low"
        ] = ci_low

        output_row[
            f"{component}_ci95_high"
        ] = ci_high

    pid_repeat_summary_rows.append(
        output_row
    )

pid_repeat_summary = pd.DataFrame(
    pid_repeat_summary_rows
)

atomic_csv(
    pid_repeat_summary,
    OUT_DIR / "pid_repeat_summary.csv",
)


# =============================================================================
# 11. ESTIMATION POOLED-REPEAT PONCTUELLE
# =============================================================================

pid_pooled_rows = []

for classifier in CLASSIFIERS:
    for layer_a, layer_b in PAIRS:
        source_a, source_b, target = (
            pooled_repeat_arrays(
                classifier,
                layer_a,
                layer_b,
            )
        )

        for estimator, function in [
            (
                "I_min",
                pid_williams_beer_imin,
            ),
            (
                "BROJA",
                pid_broja,
            ),
        ]:
            result = function(
                source_a,
                source_b,
                target,
            )

            pid_pooled_rows.append({
                "estimator": estimator,
                "classifier": classifier,
                "pair": (
                    f"{layer_a}+{layer_b}"
                ),
                "pair_label": (
                    f"{SHORT[layer_a]}+"
                    f"{SHORT[layer_b]}"
                ),
                "n_patient_clusters": (
                    N_PATIENTS
                ),
                "n_long_observations": (
                    len(target)
                ),
                **result,
            })

pid_pooled = pd.DataFrame(
    pid_pooled_rows
)

atomic_csv(
    pid_pooled,
    OUT_DIR / "pid_pooled_repeat_point_estimates.csv",
)


# =============================================================================
# 12. STRATIFIED PATIENT-CLUSTER BOOTSTRAP
# =============================================================================

bootstrap_state_path = (
    BOOT_DIR
    / "imin_broja_patient_cluster_bootstrap.pkl"
)

draw_keys = [
    (
        f"{estimator}||{classifier}||"
        f"{layer_a}+{layer_b}||{component}"
    )
    for estimator in ["I_min", "BROJA"]
    for classifier in CLASSIFIERS
    for layer_a, layer_b in PAIRS
    for component in PID_COMPONENTS
]

diagnostic_keys = [
    (
        f"BROJA||{classifier}||"
        f"{layer_a}+{layer_b}||"
        f"{diagnostic}"
    )
    for classifier in CLASSIFIERS
    for layer_a, layer_b in PAIRS
    for diagnostic in [
        "max_numerical_error",
        "decomposition_residual",
        "elapsed_seconds",
        "valid",
    ]
]

if bootstrap_state_path.exists():
    with open(
        bootstrap_state_path,
        "rb",
    ) as handle:
        bootstrap_state = pickle.load(handle)

    if bootstrap_state.get("config_hash") != CONFIG_HASH:
        raise RuntimeError(
            "The bootstrap checkpoint belongs to a configuration "
            "that differs scientifically.\nMove or delete:  "
            f"{bootstrap_state_path}"
        )

    if int(bootstrap_state.get("done", 0)) > N_PATIENT_BOOTSTRAP:
        raise RuntimeError(
            "The checkpoint holds more replicates than the target "
            f"requested ({bootstrap_state['done']} > {N_PATIENT_BOOTSTRAP}). "
            "Augmentez PID_BOOTSTRAP_REPLICATES ou utilisez un nouveau dossier."
        )
else:
    bootstrap_state = {
        "analysis_hash": CONFIG_HASH,
        "done": 0,
        "draws": {
            key: []
            for key in draw_keys
        },
        "diagnostics": {
            key: []
            for key in diagnostic_keys
        },
        "errors": [],
        "convergence": [],
    }

class_indices = {
    int(class_value): np.flatnonzero(
        y == class_value
    )
    for class_value in np.unique(y)
}

for bootstrap_index in range(
    int(bootstrap_state["done"]),
    N_PATIENT_BOOTSTRAP,
):
    rng = np.random.default_rng(
        stable_seed(
            SEED,
            "patient_cluster_bootstrap",
            bootstrap_index,
        )
    )

    # Un unique resampling de patients, stratified par PAM50,
    # shared across all classifiers, all pairs and both PID estimators.
    sampled_patient_indices = np.concatenate([
        rng.choice(
            indices,
            size=len(indices),
            replace=True,
        )
        for indices in class_indices.values()
    ])

    rng.shuffle(sampled_patient_indices)

    for classifier in CLASSIFIERS:
        for layer_a, layer_b in PAIRS:
            pair_key = f"{layer_a}+{layer_b}"

            source_a, source_b, target = (
                pooled_repeat_arrays(
                    classifier,
                    layer_a,
                    layer_b,
                    sampled_patient_indices,
                )
            )

            # I_min
            imin_result = (
                pid_williams_beer_imin(
                    source_a,
                    source_b,
                    target,
                )
            )

            for component in PID_COMPONENTS:
                bootstrap_state["draws"][
                    (
                        f"I_min||{classifier}||"
                        f"{pair_key}||{component}"
                    )
                ].append(
                    float(
                        imin_result[component]
                    )
                )

            # BROJA
            broja_result = pid_broja(
                source_a,
                source_b,
                target,
            )

            for component in PID_COMPONENTS:
                bootstrap_state["draws"][
                    (
                        f"BROJA||{classifier}||"
                        f"{pair_key}||{component}"
                    )
                ].append(
                    float(
                        broja_result[component]
                    )
                    if np.isfinite(
                        broja_result[component]
                    )
                    else np.nan
                )

            for diagnostic in [
                "max_numerical_error",
                "decomposition_residual",
                "elapsed_seconds",
            ]:
                bootstrap_state[
                    "diagnostics"
                ][
                    (
                        f"BROJA||{classifier}||"
                        f"{pair_key}||{diagnostic}"
                    )
                ].append(
                    float(
                        broja_result[diagnostic]
                    )
                    if np.isfinite(
                        broja_result[diagnostic]
                    )
                    else np.nan
                )

            bootstrap_state[
                "diagnostics"
            ][
                (
                    f"BROJA||{classifier}||"
                    f"{pair_key}||valid"
                )
            ].append(
                bool(broja_result["valid"])
            )

            if not broja_result["valid"]:
                bootstrap_state[
                    "errors"
                ].append({
                    "bootstrap": bootstrap_index,
                    "classifier": classifier,
                    "pair": pair_key,
                    "solver_status": (
                        broja_result[
                            "solver_status"
                        ]
                    ),
                    "error_type": (
                        broja_result[
                            "error_type"
                        ]
                    ),
                    "error_message": (
                        broja_result[
                            "error_message"
                        ]
                    ),
                    "max_numerical_error": (
                        broja_result[
                            "max_numerical_error"
                        ]
                    ),
                    "decomposition_residual": (
                        broja_result[
                            "decomposition_residual"
                        ]
                    ),
                })

    bootstrap_state["done"] = (
        bootstrap_index + 1
    )

    if (
        bootstrap_state["done"]
        in BOOTSTRAP_CONVERGENCE_POINTS
    ):
        snapshot = {
            "n_boot": int(
                bootstrap_state["done"]
            ),
            "estimates": {},
        }

        for key, draws in (
            bootstrap_state["draws"].items()
        ):
            values = np.asarray(
                draws,
                dtype=float,
            )
            values = values[
                np.isfinite(values)
            ]

            if len(values) == 0:
                snapshot[
                    "estimates"
                ][key] = {
                    "n_valid": 0,
                    "mean": None,
                    "ci95_low": None,
                    "ci95_high": None,
                }
            else:
                snapshot[
                    "estimates"
                ][key] = {
                    "n_valid": int(
                        len(values)
                    ),
                    "mean": float(
                        values.mean()
                    ),
                    "ci95_low": float(
                        np.quantile(
                            values,
                            0.025,
                        )
                    ),
                    "ci95_high": float(
                        np.quantile(
                            values,
                            0.975,
                        )
                    ),
                }

        bootstrap_state[
            "convergence"
        ].append(snapshot)

    if (
        bootstrap_state["done"]
        % BOOTSTRAP_SAVE_EVERY
        == 0
        or bootstrap_state["done"]
        == N_PATIENT_BOOTSTRAP
    ):
        atomic_pickle(
            bootstrap_state,
            bootstrap_state_path,
        )

        log(
            "PID patient-cluster bootstrap : "
            f"{bootstrap_state['done']}/"
            f"{N_PATIENT_BOOTSTRAP}"
        )


# =============================================================================
# 13. SUMMARY DU BOOTSTRAP PATIENT
# =============================================================================

bootstrap_summary_rows = []

for estimator in ["I_min", "BROJA"]:
    for classifier in CLASSIFIERS:
        for layer_a, layer_b in PAIRS:
            pair_key = (
                f"{layer_a}+{layer_b}"
            )

            row = {
                "estimator": estimator,
                "classifier": classifier,
                "pair": pair_key,
                "pair_label": (
                    f"{SHORT[layer_a]}+"
                    f"{SHORT[layer_b]}"
                ),
                "n_bootstrap_requested": (
                    N_PATIENT_BOOTSTRAP
                ),
            }

            component_valid_counts = []

            for component in PID_COMPONENTS:
                values = np.asarray(
                    bootstrap_state[
                        "draws"
                    ][
                        (
                            f"{estimator}||"
                            f"{classifier}||"
                            f"{pair_key}||"
                            f"{component}"
                        )
                    ],
                    dtype=float,
                )

                values = values[
                    np.isfinite(values)
                ]

                component_valid_counts.append(
                    len(values)
                )

                if len(values) == 0:
                    row[
                        f"{component}_mean"
                    ] = np.nan
                    row[
                        f"{component}_sd"
                    ] = np.nan
                    row[
                        f"{component}_ci95_low"
                    ] = np.nan
                    row[
                        f"{component}_ci95_high"
                    ] = np.nan
                else:
                    row[
                        f"{component}_mean"
                    ] = float(
                        values.mean()
                    )
                    row[
                        f"{component}_sd"
                    ] = float(
                        values.std(ddof=1)
                    )
                    row[
                        f"{component}_ci95_low"
                    ] = float(
                        np.quantile(
                            values,
                            0.025,
                        )
                    )
                    row[
                        f"{component}_ci95_high"
                    ] = float(
                        np.quantile(
                            values,
                            0.975,
                        )
                    )

            row["n_valid_bootstrap"] = int(
                min(component_valid_counts)
            )

            row["n_failed_bootstrap"] = int(
                N_PATIENT_BOOTSTRAP
                - row["n_valid_bootstrap"]
            )

            if estimator == "BROJA":
                validity = np.asarray(
                    bootstrap_state[
                        "diagnostics"
                    ][
                        (
                            f"BROJA||{classifier}||"
                            f"{pair_key}||valid"
                        )
                    ],
                    dtype=bool,
                )

                numerical_errors = np.asarray(
                    bootstrap_state[
                        "diagnostics"
                    ][
                        (
                            f"BROJA||{classifier}||"
                            f"{pair_key}||"
                            "max_numerical_error"
                        )
                    ],
                    dtype=float,
                )

                elapsed_seconds = np.asarray(
                    bootstrap_state[
                        "diagnostics"
                    ][
                        (
                            f"BROJA||{classifier}||"
                            f"{pair_key}||"
                            "elapsed_seconds"
                        )
                    ],
                    dtype=float,
                )

                row["solver_valid_rate"] = float(
                    validity.mean()
                )

                row[
                    "median_max_numerical_error"
                ] = float(
                    np.nanmedian(
                        numerical_errors
                    )
                )

                row[
                    "median_solver_seconds"
                ] = float(
                    np.nanmedian(
                        elapsed_seconds
                    )
                )

            bootstrap_summary_rows.append(
                row
            )

pid_bootstrap_summary = pd.DataFrame(
    bootstrap_summary_rows
)

atomic_csv(
    pid_bootstrap_summary,
    OUT_DIR / "pid_patient_cluster_bootstrap_summary.csv",
)

failure_frame = pd.DataFrame(
    bootstrap_state["errors"]
)

atomic_csv(
    failure_frame,
    OUT_DIR / "broja_bootstrap_solver_failures.csv",
)

atomic_json(
    bootstrap_state["convergence"],
    OUT_DIR / "pid_bootstrap_convergence.json",
)


# =============================================================================
# 14. PAIRED COMPARISON BROJA - I_min
# =============================================================================

estimator_difference_rows = []

for classifier in CLASSIFIERS:
    for layer_a, layer_b in PAIRS:
        pair_key = f"{layer_a}+{layer_b}"

        for component in [
            "redundancy",
            "unique_a",
            "unique_b",
            "synergy",
        ]:
            imin_values = np.asarray(
                bootstrap_state["draws"][
                    (
                        f"I_min||{classifier}||"
                        f"{pair_key}||{component}"
                    )
                ],
                dtype=float,
            )

            broja_values = np.asarray(
                bootstrap_state["draws"][
                    (
                        f"BROJA||{classifier}||"
                        f"{pair_key}||{component}"
                    )
                ],
                dtype=float,
            )

            valid = (
                np.isfinite(imin_values)
                & np.isfinite(broja_values)
            )

            differences = (
                broja_values[valid]
                - imin_values[valid]
            )

            if len(differences) == 0:
                difference_mean = np.nan
                ci_low = np.nan
                ci_high = np.nan
            else:
                difference_mean = float(
                    differences.mean()
                )
                ci_low = float(
                    np.quantile(
                        differences,
                        0.025,
                    )
                )
                ci_high = float(
                    np.quantile(
                        differences,
                        0.975,
                    )
                )

            estimator_difference_rows.append({
                "classifier": classifier,
                "pair": pair_key,
                "pair_label": (
                    f"{SHORT[layer_a]}+"
                    f"{SHORT[layer_b]}"
                ),
                "component": component,
                "mean_difference_broja_minus_imin": (
                    difference_mean
                ),
                "ci95_low": ci_low,
                "ci95_high": ci_high,
                "ci_excludes_zero": bool(
                    np.isfinite(ci_low)
                    and np.isfinite(ci_high)
                    and (
                        ci_high < 0
                        or ci_low > 0
                    )
                ),
                "n_paired_bootstrap": int(
                    len(differences)
                ),
            })

estimator_differences = pd.DataFrame(
    estimator_difference_rows
)

atomic_csv(
    estimator_differences,
    OUT_DIR / "broja_minus_imin_paired_bootstrap.csv",
)


# =============================================================================
# 15. RANKING STABILITY BETWEEN ESTIMATORS
# =============================================================================

ranking_correlation_rows = []

for classifier in CLASSIFIERS:
    classifier_data = (
        pid_bootstrap_summary[
            pid_bootstrap_summary[
                "classifier"
            ]
            == classifier
        ]
    )

    for component in [
        "redundancy",
        "unique_a",
        "unique_b",
        "synergy",
    ]:
        pivot = classifier_data.pivot(
            index="pair",
            columns="estimator",
            values=f"{component}_mean",
        ).dropna()

        rho, p_value = spearmanr(
            pivot["I_min"],
            pivot["BROJA"],
        )

        ranking_correlation_rows.append({
            "classifier": classifier,
            "component": component,
            "spearman_rho_imin_vs_broja": (
                float(rho)
            ),
            "p_value": float(p_value),
            "n_pairs": int(len(pivot)),
        })

estimator_rank_correlations = pd.DataFrame(
    ranking_correlation_rows
)

atomic_csv(
    estimator_rank_correlations,
    OUT_DIR / "imin_broja_rank_correlations.csv",
)


# =============================================================================
# 16. COMPARAISON PID / Rpred
# =============================================================================

pid_rpred_rows = []

for estimator in ["I_min", "BROJA"]:
    estimator_data = (
        pid_bootstrap_summary[
            pid_bootstrap_summary[
                "estimator"
            ]
            == estimator
        ]
    )

    merged = rpred_summary.merge(
        estimator_data[
            [
                "classifier",
                "pair",
                "redundancy_mean",
                "synergy_mean",
                "unique_a_mean",
                "unique_b_mean",
            ]
        ],
        on=["classifier", "pair"],
        how="inner",
        validate="one_to_one",
    )

    atomic_csv(
        merged,
        OUT_DIR
        / (
            f"rpred_{estimator.lower()}"
            "_pairwise_table.csv"
        ),
    )

    for classifier, group in merged.groupby(
        "classifier"
    ):
        for component in [
            "redundancy",
            "synergy",
            "unique_a",
            "unique_b",
        ]:
            rho, p_value = spearmanr(
                group["rpred_repeat_mean"],
                group[f"{component}_mean"],
            )

            pid_rpred_rows.append({
                "estimator": estimator,
                "classifier": classifier,
                "pid_component": component,
                "spearman_rpred_vs_pid": float(
                    rho
                ),
                "p_value": float(p_value),
                "n_pairs": int(len(group)),
            })

pid_rpred_correlations = pd.DataFrame(
    pid_rpred_rows
)

atomic_csv(
    pid_rpred_correlations,
    OUT_DIR / "rpred_pid_rank_correlations.csv",
)


# =============================================================================
# 17. RAPPORT AUTOMATIQUE
# =============================================================================

def best_pair_for_component(
    estimator: str,
    component: str,
    ascending: bool = False,
) -> Dict[str, Any]:
    subset = (
        pid_bootstrap_summary[
            pid_bootstrap_summary[
                "estimator"
            ]
            == estimator
        ]
        .groupby(
            ["pair", "pair_label"],
            as_index=False,
        )[f"{component}_mean"]
        .mean()
        .sort_values(
            f"{component}_mean",
            ascending=ascending,
        )
    )

    row = subset.iloc[0]

    return {
        "pair": str(row["pair"]),
        "pair_label": str(
            row["pair_label"]
        ),
        f"mean_{component}_across_classifiers": (
            float(
                row[f"{component}_mean"]
            )
        ),
    }


broja_validity = (
    pid_bootstrap_summary[
        pid_bootstrap_summary["estimator"]
        == "BROJA"
    ]
)

report = {
    "run_dir": str(RUN_DIR),
    "analysis_hash": CONFIG_HASH,
    "metric_reconstruction": {
        "passed": (
            metric_reconstruction_passed
        ),
        "maximum_absolute_difference": (
            max_metric_difference
        ),
        "tolerance": METRIC_TOLERANCE,
    },
    "pid_estimators": {
        "I_min": {
            "name": (
                "Williams-Beer I_min"
            ),
            "definition": (
                "sum_t p(t) min("
                "specific_information_t_A, "
                "specific_information_t_B)"
            ),
            "warning": (
                "This is not min(I(T;A),I(T;B)); "
                "the latter is MMI."
            ),
        },
        "BROJA": {
            "name": "BROJA-2PID",
            "package_version": (
                broja_version
            ),
            "solver": "ECOS",
            "ecos_version": ecos_version,
            "mean_solver_valid_rate": float(
                broja_validity[
                    "solver_valid_rate"
                ].mean()
            ),
        },
    },
    "bootstrap": {
        "method": (
            "stratified patient-cluster bootstrap "
            "with all ten repeated-CV predictions "
            "kept together for each sampled patient"
        ),
        "requested_replicates": (
            N_PATIENT_BOOTSTRAP
        ),
        "same_indices_across_estimators_classifiers_pairs": (
            True
        ),
    },
    "primary_answers": {
        "I_min_most_redundant_pair": (
            best_pair_for_component(
                "I_min",
                "redundancy",
            )
        ),
        "I_min_most_synergistic_pair": (
            best_pair_for_component(
                "I_min",
                "synergy",
            )
        ),
        "BROJA_most_redundant_pair": (
            best_pair_for_component(
                "BROJA",
                "redundancy",
            )
        ),
        "BROJA_most_synergistic_pair": (
            best_pair_for_component(
                "BROJA",
                "synergy",
            )
        ),
    },
    "interpretation_notes": [
        "Both decompositions are prediction-level PIDs computed from out-of-fold categorical predictions.",
        "Neither decomposition directly measures information in the raw omics matrices.",
        "Williams-Beer I_min and BROJA answer related but non-identical redundancy questions.",
        "Estimator disagreement is reported rather than hidden.",
        "BROJA numerical diagnostics and failed bootstrap solves are retained.",
        "The pooled-repeat bootstrap treats patients as clusters and carries all ten repeated-CV predictions together.",
        "Rpred, I_min and BROJA quantify different objects; rank agreement is an empirical result, not an identity.",
    ],
}

atomic_json(
    report,
    OUT_DIR / "report.json",
)

log("")
log("=" * 80)
log("CALCUL I_min + BROJA COMPLETE")
log(f"Results : {OUT_DIR}")
log(
    "Fixed-model audit:  "
    f"{'OK' if metric_reconstruction_passed else 'FAILED'}"
)
log(
    "Mean rate of valid BROJA solutions:  "
    f"{report['pid_estimators']['BROJA']['mean_solver_valid_rate']:.3f}"
)
log("=" * 80)

del prediction_unit_frames
gc.collect()


## 4. Consolidation of reported analyses


**Consolidation.** Merges the performance, complementarity, contribution and decomposition outputs into a single report.

In [ ]:

"""Consolide les sorties performance, Rpred, LOO, Shapley et PID."""
from __future__ import annotations
import json
import os
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    if not os.path.exists(str(DATA_DIR)):
        drive.mount("/content/drive")
    DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
except Exception:
    DATA_DIR = Path("./TCGA_BRCA_data")

REGIME_PATHS = {
    "k50": Path(os.environ.get("K50_RUN_DIR", str(DATA_DIR / "runs" / "2026-07-26"))),
}
requested = [x.strip() for x in os.environ.get("REGIMES_TO_ANALYZE", "k50").split(",") if x.strip()]
OUT_DIR = DATA_DIR / "runs" / "full_scientific_pipeline"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def read_json(path: Path) -> Any:
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def atomic_json(obj: Any, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)

perf_dir = DATA_DIR / "runs" / "comparison_k50_fixed_models"
rpred_dir = DATA_DIR / "runs" / "comparison_rpred_fixed_models"
perf_report = read_json(perf_dir / "report.json")
rpred_report = read_json(rpred_dir / "report.json")

loo_path = perf_dir / "contribution_leave_one_omics_out.csv"
shap_path = perf_dir / "contribution_shapley_summary.csv"
loo = pd.read_csv(loo_path) if loo_path.exists() else pd.DataFrame()
shap = pd.read_csv(shap_path) if shap_path.exists() else pd.DataFrame()

regime_reports = {}
for regime in requested:
    pid_dir = REGIME_PATHS[regime] / "results" / "model_evaluation" / "pid_imin_broja_fixed_models"
    pid_report = read_json(pid_dir / "report.json")

    loo_answer = None
    if len(loo):
        g = loo[loo["regime"] == regime]
        if len(g):
            agg = (g.groupby(["layer", "layer_label"], as_index=False)
                     ["mean_loo_contribution"].mean()
                     .sort_values("mean_loo_contribution", ascending=False))
            row = agg.iloc[0]
            loo_answer = {
                "layer": str(row["layer"]),
                "layer_label": str(row["layer_label"]),
                "mean_loo_contribution_across_classifiers": float(row["mean_loo_contribution"]),
            }

    shap_answer = None
    if len(shap):
        g = shap[shap["regime"] == regime]
        if len(g):
            agg = (g.groupby(["layer", "layer_label"], as_index=False)
                     ["mean_performance_shapley"].mean()
                     .sort_values("mean_performance_shapley", ascending=False))
            row = agg.iloc[0]
            shap_answer = {
                "layer": str(row["layer"]),
                "layer_label": str(row["layer_label"]),
                "mean_performance_shapley_across_classifiers": float(row["mean_performance_shapley"]),
            }

    regime_reports[regime] = {
        "performance_and_shapley": None if perf_report is None else perf_report.get("regime_results", {}).get(regime),
        "rpred": None if rpred_report is None else rpred_report.get("primary_answers", {}).get(regime),
        "most_dominant_layer_by_loo": loo_answer,
        "most_dominant_layer_by_shapley": shap_answer,
        "pid_imin_broja": pid_report,
    }

report = {
    "analysis": "fixed-hyperparameter reported analysis",
    "regimes": requested,
    "primary_answers_by_regime": regime_reports,
    "methodological_status": {
        "preprocessing_rerun": False,
        "nested_cv": False,
        "model_hyperparameters": "fixed a priori",
        "performance_inference_unit": "repeated-CV repetition (n=10)",
        "pid_bootstrap_unit": "stratified patient cluster preserving all 10 repeated-CV predictions",
        "final_publication_note": "The reported study uses the fixed-k=50, fixed-hyperparameter analysis; nested cross-validation is not part of the reported analysis.",
    },
}
atomic_json(report, OUT_DIR / "report_consolidated.json")
print(f"Consolidated report:  {OUT_DIR / 'report_consolidated.json'}")


**Output listing.** Lists the files written by the consolidation step.

In [ ]:
from pathlib import Path
pid_dir = Path(str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'runs' / '2026-07-26' / 'results' / 'model_evaluation' / 'pid_imin_broja_fixed_models'))
for f in sorted(pid_dir.glob("*.csv")):
    print(f.name, f"({f.stat().st_size/1024:.1f} KB)")

## 5. Gene-level biological plausibility


**Working directory.** Prepares the output location for the attribution stage.

In [ ]:
import shutil
shutil.rmtree("/content/drive", ignore_errors=True)

**Data directory.** Points the attribution stage at the input data.

In [ ]:
# Data directory. Set TCGA_BRCA_DATA_DIR to point at the local copy of the
# public input data; the Google Colab mount below is optional and is skipped
# when the environment variable is already defined.
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
if not DATA_DIR.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path(os.environ["TCGA_BRCA_DATA_DIR"])
    except Exception:
        raise SystemExit(
            "Set TCGA_BRCA_DATA_DIR to the directory containing the input data."
        )

**Level 6.** Computes gene-level attribution on the minimal sufficient panel for each classifier across all 50 splits, back-projecting component-level SHAP values or permutation importance into gene space, and measures overlap with the canonical PAM50 genes across top-ranked thresholds.

In [ ]:
"""
===============================================================================
- BIOLOGICAL INTERPRETATION: gene-level SHAP (minimal mRNA panel)
STANDALONE VERSION - variance filter + top-5000 MAD included
===============================================================================

Charge tout from le disque (cohort_manifest.csv, cv_splits.json,
preprocessing_report.json, mrna_hugo_mapped_primary_only.parquet), sans
depend on in-memory context.

Reproduit exactement le preprocessing mRNA du pipeline principal :
filtre de variance -> top-5000 MAD -> standardisation -> PCA, toutes les
steps fitted sur le train uniquement, split par split.

Audit de consistency strict : chaque model reconstruit doit reproduire
panel_fold_results.csv to within 1e-4, otherwise the script stops.

Launch (paste into a new cell of the Pipeline notebook):
    import os
    os.environ["RUN_DIR"] = str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'runs' / '2026-07-26')
    %run shap_gene_level_minimal_panel.py
"""

import os
import json
import time
import pickle
import hashlib
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False

if not HAS_SHAP:
    raise ImportError("!pip install shap --quiet, puis relancez cette cellule.")

# =============================================================================
# 0. PORTABLE DATA ACCESS
# =============================================================================

from google.colab import drive
if not os.path.exists(str(DATA_DIR)):
    drive.mount("/content/drive")

# =============================================================================
# 1. CHEMINS
# =============================================================================

DEFAULT_RUN_DIR = str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'runs' / '2026-07-26')
RUN_DIR = Path(os.environ.get("RUN_DIR", DEFAULT_RUN_DIR))
DATA_DIR = RUN_DIR.parent.parent  # .../TCGA_BRCA_data
RESULTS_DIR = RUN_DIR / "results"
EVAL_DIR = RESULTS_DIR / "model_evaluation"
CKPT_DIR = RUN_DIR / "checkpoints"
OUT_DIR = EVAL_DIR / "shap_gene_level_minimal_panel"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get("SEED", "42"))
PANEL = "mrna"
CLASSIFIERS = ["RF", "XGB", "SVM"]
METRIC_TOLERANCE = 1e-4
STRICT_METRIC_MATCH = True

MIN_VARIANCE = 1e-6
FILTER_TOP_K = 5000

PAM50_GENES = [
    "ACTR3B", "ANLN", "BAG1", "BCL2", "BIRC5", "BLVRA", "CCNB1",
    "CCNE1", "CDC20", "CDC6", "CDH3", "CENPF", "CEP55", "CXXC5",
    "EGFR", "ERBB2", "ESR1", "EXO1", "FGFR4", "FOXA1", "FOXC1",
    "GPR160", "GRB7", "KIF2C", "KRT14", "KRT17", "KRT5", "MAPT",
    "MDM2", "MELK", "MIA", "MKI67", "MLPH", "MMP11", "MYBL2",
    "MYC", "NAT1", "NDC80", "NUF2", "ORC6", "PGR", "PHGDH",
    "PTTG1", "RRM2", "SFRP1", "SLC39A6", "TMEM45B", "TYMS",
    "UBE2C", "UBE2T",
]

EVAL_CONFIG = {
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 2, "n_jobs": -1},
    "xgboost": {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.85,
               "colsample_bytree": 0.85, "reg_lambda": 1.0, "reg_alpha": 0.0, "min_child_weight": 1.0, "n_jobs": -1},
    "svm": {"C": 1.0, "kernel": "rbf", "gamma": "scale", "class_weight": "balanced", "probability": False},
}


def find_existing(candidates: List[Path]) -> Path:
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f"None of these paths exists: {candidates}")


def detect_column(df: pd.DataFrame, candidates: List[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"No column among {candidates} found in {df.columns.tolist()}")


def stable_seed(*parts) -> int:
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def atomic_csv(df: pd.DataFrame, path: Path) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


print(f"RUN_DIR : {RUN_DIR}")
print(f"Analysed panel:  {PANEL}")

# =============================================================================
# 2. CHARGEMENT COHORTE + SPLITS
# =============================================================================

cohort_path = find_existing([RESULTS_DIR / "cohort_manifest.csv", RESULTS_DIR / "cohort.csv"])
splits_path = find_existing([RESULTS_DIR / "cv_splits.json"])
fold_results_path = find_existing([EVAL_DIR / "panel_fold_results.csv", RESULTS_DIR / "panel_fold_results.csv"])
report_path = find_existing([RESULTS_DIR / "preprocessing_report.json"])

cohort = pd.read_csv(cohort_path)
patient_column = detect_column(cohort, ["patient_id", "patient", "case_id"])
label_column = detect_column(cohort, ["label_encoded", "y", "target_encoded"])
patient_ids = cohort[patient_column].astype(str).to_numpy()
y = pd.to_numeric(cohort[label_column], errors="raise").to_numpy(dtype=int)
N_PATIENTS = len(y)
N_CLASSES = len(np.unique(y))

with open(splits_path, encoding="utf-8") as f:
    split_payload = json.load(f)
split_payload = sorted(split_payload, key=lambda row: int(row["split"]))
SPLITS = [(np.asarray(r["train_indices"], dtype=int), np.asarray(r["test_indices"], dtype=int))
          for r in split_payload]
REPEAT_OF = [int(r["repeat"]) for r in split_payload]

with open(report_path, encoding="utf-8") as f:
    preprocessing_report = json.load(f)
n_components = int(preprocessing_report["config"]["dim_k"])

print(f"Cohorte : {N_PATIENTS} patients, {N_CLASSES} classes, {len(SPLITS)} splits, dim_k={n_components}")

# =============================================================================
# 3. RAW mRNA LOADING - reindexed exactly to the cohort_manifest.csv order
# =============================================================================

mrna_path = find_existing([DATA_DIR / "mrna_hugo_mapped_primary_only.parquet"])
mrna_raw = pd.read_parquet(mrna_path)
mrna_raw.index = mrna_raw.index.astype(str)

missing_patients = set(patient_ids) - set(mrna_raw.index)
if missing_patients:
    raise RuntimeError(f"{len(missing_patients)} patients de la cohorte absents du parquet mRNA.")

mrna_raw = mrna_raw.loc[patient_ids]  # strict reindexing, same order as y/SPLITS
if not np.array_equal(mrna_raw.index.to_numpy(), patient_ids):
    raise RuntimeError("patient_id misalignment between cohort_manifest.csv and the reindexed mRNA.")

print(f"mRNA brut : {mrna_raw.shape} (patients x genes)")

# =============================================================================
# 4. REFERENCE: recorded metrics, for the audit
# =============================================================================

fold_results = pd.read_csv(fold_results_path)
official = fold_results[fold_results["panel"] == PANEL].copy()
official = official.set_index(["classifier", "split"])["balanced_accuracy"]


def make_classifier(name: str, seed: int):
    if name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if name == "XGB":
        return XGBClassifier(objective="multi:softprob", num_class=N_CLASSES, eval_metric="mlogloss",
                             random_state=seed, verbosity=0, **EVAL_CONFIG["xgboost"])
    if name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    raise ValueError(name)


def score_mad(df_train: pd.DataFrame) -> pd.Series:
    """Median absolute deviation, computed on training rows only."""
    medians = df_train.median(axis=0)
    return (df_train - medians).abs().median(axis=0)


def fit_mrna_reducer(train_index: np.ndarray, test_index: np.ndarray, seed: int):
    """Reproduces EXACTLY the mRNA preprocessing of the main pipeline :
    filtre de variance -> top-5000 MAD -> standardisation -> PCA,
    toutes les steps fitted sur le train uniquement."""
    train_df = mrna_raw.iloc[train_index]
    test_df = mrna_raw.iloc[test_index]

    # 1. Filtre de variance quasi-nulle (train uniquement)
    variances = train_df.var(axis=0)
    keep_columns = variances[variances > MIN_VARIANCE].index
    train_df = train_df.loc[:, keep_columns]
    test_df = test_df.loc[:, keep_columns]

    # 2. Top-5000 MAD (train uniquement)
    if train_df.shape[1] > FILTER_TOP_K:
        mad_scores = score_mad(train_df)
        selected_columns = mad_scores.nlargest(FILTER_TOP_K).index
        train_df = train_df.loc[:, selected_columns]
        test_df = test_df.loc[:, selected_columns]

    gene_names_selected = list(train_df.columns)

    # 3. Standardisation
    scaler = StandardScaler().fit(train_df.to_numpy(dtype=np.float64))
    X_train = scaler.transform(train_df.to_numpy(dtype=np.float64))
    X_test = scaler.transform(test_df.to_numpy(dtype=np.float64))

    # 4. PCA
    reducer = PCA(n_components=n_components, random_state=seed).fit(X_train)
    Z_train = reducer.transform(X_train).astype(np.float32)
    Z_test = reducer.transform(X_test).astype(np.float32)

    return Z_train, Z_test, reducer, gene_names_selected


# =============================================================================
# 5. RECONSTRUCTION SHAP GENE-NIVEAU -- REPRISE FINE
# =============================================================================

shap_path = OUT_DIR / "gene_shap_by_split.csv"
audit_path = OUT_DIR / "metric_reconstruction_audit.csv"

if shap_path.exists() and audit_path.exists():
    shap_rows = pd.read_csv(shap_path).to_dict("records")
    audit_rows = pd.read_csv(audit_path).to_dict("records")
    done_splits = set(pd.read_csv(shap_path)["split"].unique().tolist())
    print(f"Resuming:  {len(done_splits)}/{len(SPLITS)} splits already processed")
else:
    shap_rows, audit_rows, done_splits = [], [], set()

t0 = time.time()
for split_index, (train_index, test_index) in enumerate(SPLITS):
    if split_index in done_splits:
        continue

    repeat_index = REPEAT_OF[split_index]
    seed = int(SEED + repeat_index)
    y_train, y_test = y[train_index], y[test_index]
    sample_weight = compute_sample_weight("balanced", y_train)

    Z_train, Z_test, reducer, gene_names_split = fit_mrna_reducer(train_index, test_index, seed)
    loadings = reducer.components_  # (n_components, n_genes_selected_ce_split)

    for clf_name in CLASSIFIERS:
        model = make_classifier(clf_name, seed)
        if clf_name in {"RF", "XGB"}:
            model.fit(Z_train, y_train, sample_weight=sample_weight)
        else:
            model.fit(Z_train, y_train)

        y_pred = model.predict(Z_test)
        reconstructed_ba = float(balanced_accuracy_score(y_test, y_pred))
        official_ba = float(official.loc[(clf_name, split_index)])
        diff = abs(reconstructed_ba - official_ba)
        audit_rows.append({
            "split": split_index, "classifier": clf_name,
            "official_ba": official_ba, "reconstructed_ba": reconstructed_ba,
            "abs_diff": diff, "within_tolerance": diff <= METRIC_TOLERANCE,
        })
        if STRICT_METRIC_MATCH and diff > METRIC_TOLERANCE:
            atomic_csv(pd.DataFrame(audit_rows), audit_path)
            raise RuntimeError(
                f"Audit failed:  split={split_index}, classifier={clf_name}, "
                f"difference={diff:.6f} > tolerance={METRIC_TOLERANCE}."
            )

        if clf_name in {"RF", "XGB"}:
            explainer = shap.TreeExplainer(model)
            raw_shap = explainer.shap_values(Z_test)
            if isinstance(raw_shap, list):
                comp_importance = np.mean([np.abs(s).mean(axis=0) for s in raw_shap], axis=0)
                comp_signed = np.mean([s.mean(axis=0) for s in raw_shap], axis=0)
            else:
                comp_importance = (np.abs(raw_shap).mean(axis=(0, 2)) if raw_shap.ndim == 3
                                   else np.abs(raw_shap).mean(axis=0))
                comp_signed = (raw_shap.mean(axis=(0, 2)) if raw_shap.ndim == 3
                              else raw_shap.mean(axis=0))
            method = "TreeExplainer_SHAP"
        else:
            perm = permutation_importance(model, Z_test, y_test, n_repeats=10,
                                          random_state=seed, scoring="balanced_accuracy", n_jobs=-1)
            comp_importance = np.abs(perm.importances_mean)
            comp_signed = perm.importances_mean
            method = "permutation_importance"

        gene_importance = comp_importance @ np.abs(loadings)
        gene_signed = comp_signed @ loadings

        for gene, imp, signed in zip(gene_names_split, gene_importance, gene_signed):
            shap_rows.append({
                "split": split_index, "repeat": repeat_index, "classifier": clf_name,
                "method": method, "gene": gene,
                "importance_abs": float(imp), "importance_signed": float(signed),
            })

    if (split_index + 1) % 5 == 0 or split_index == len(SPLITS) - 1:
        atomic_csv(pd.DataFrame(shap_rows), shap_path)
        atomic_csv(pd.DataFrame(audit_rows), audit_path)
        print(f"  split {split_index + 1}/{len(SPLITS)} saved ({time.time() - t0:.0f}s cette session)")

shap_df = pd.DataFrame(shap_rows)
audit_df = pd.DataFrame(audit_rows)
print(f"\nAudit : {audit_df['within_tolerance'].mean() * 100:.1f}% dans la tolerance "
      f"(difference max = {audit_df['abs_diff'].max():.6f})")

# =============================================================================
# 6. AGGREGATION + CHEVAUCHEMENT PAM50
# =============================================================================

def boot_ci(values: np.ndarray, n_boot: int = 2000, seed: int = 0):
    rng = np.random.default_rng(seed)
    draws = rng.choice(values, size=(n_boot, len(values)), replace=True).mean(axis=1)
    return float(np.quantile(draws, 0.025)), float(np.quantile(draws, 0.975))


gene_summary_rows = []
for (clf_name, gene), g in shap_df.groupby(["classifier", "gene"]):
    v_abs = g["importance_abs"].to_numpy(float)
    lo, hi = boot_ci(v_abs, seed=stable_seed(clf_name, gene) % (2**31))
    gene_summary_rows.append({
        "classifier": clf_name, "gene": gene,
        "n_folds_present": len(g),
        "mean_importance_abs": float(v_abs.mean()), "ci95_low": lo, "ci95_high": hi,
        "mean_importance_signed": float(g["importance_signed"].mean()),
        "is_pam50": gene in PAM50_GENES,
    })
gene_summary = pd.DataFrame(gene_summary_rows)
atomic_csv(gene_summary, OUT_DIR / "gene_importance_summary.csv")

top_rows = []
for clf_name in CLASSIFIERS:
    top = gene_summary[gene_summary.classifier == clf_name].sort_values(
        "mean_importance_abs", ascending=False).head(30)
    top["rank"] = range(1, len(top) + 1)
    top_rows.append(top)
atomic_csv(pd.concat(top_rows, ignore_index=True), OUT_DIR / "top30_genes_by_classifier.csv")

pam50_overlap_rows = []
for clf_name in CLASSIFIERS:
    for top_n in (10, 20, 30, 50, 100):
        top_genes = set(gene_summary[gene_summary.classifier == clf_name]
                        .sort_values("mean_importance_abs", ascending=False).head(top_n)["gene"])
        overlap = top_genes & set(PAM50_GENES)
        pam50_overlap_rows.append({
            "classifier": clf_name, "top_n": top_n,
            "n_pam50_in_top_n": len(overlap),
            "fraction_of_pam50_covered": len(overlap) / len(PAM50_GENES),
            "pam50_genes_found": sorted(overlap),
        })
pam50_overlap = pd.DataFrame(pam50_overlap_rows)
atomic_csv(pam50_overlap, OUT_DIR / "pam50_overlap_by_top_n.csv")

print("\nChevauchement PAM50 (top-30) :")
print(pam50_overlap[pam50_overlap.top_n == 30][["classifier", "n_pam50_in_top_n", "fraction_of_pam50_covered"]]
      .to_string(index=False))

report = {
    "run_dir": str(RUN_DIR), "panel": PANEL,
    "n_genes_total_observed": int(shap_df["gene"].nunique()),
    "min_variance_threshold": MIN_VARIANCE,
    "mad_top_k": FILTER_TOP_K,
    "methods_by_classifier": {"RF": "TreeExplainer_SHAP", "XGB": "TreeExplainer_SHAP", "SVM": "permutation_importance"},
    "metric_reconstruction_audit": {
        "passed": bool(audit_df["within_tolerance"].all()),
        "max_abs_diff": float(audit_df["abs_diff"].max()), "tolerance": METRIC_TOLERANCE,
    },
    "pam50_overlap_top30": pam50_overlap[pam50_overlap.top_n == 30].to_dict("records"),
    "caveats": [
        "Gene-level values are linear projections from PCA-component-level "
        "SHAP/permutation importance via components_; exact only under a "
        "linear component-to-prediction relationship.",
        "The MAD-selected top-5000 gene set differs slightly across CV "
        "splits (train-only fitting); genes absent from a given split are "
        "simply not counted in it, and the bootstrap mean is computed over "
        "the folds where the gene was retained.",
        "SVM uses permutation importance, not true SHAP.",
        "No pathway enrichment performed here; only direct PAM50 overlap.",
    ],
}
with open(OUT_DIR / "report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"\nComplete. Results : {OUT_DIR}")

## 6. External validation and reproducibility


**External cohort.** Loads the METABRIC expression matrix and subtype annotation.

In [ ]:
"""
===============================================================================
 - METABRIC DOWNLOAD (cBioPortal datahub, git+LFS route)
===============================================================================
"""

import subprocess
from pathlib import Path

from google.colab import drive
import os
if not os.path.exists(str(DATA_DIR)):
    drive.mount("/content/drive")

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
CLONE_DIR = Path("/content/datahub_metabric_tmp")

print("Cloning the cBioPortal/datahub repository (depth 1)...")
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/cBioPortal/datahub.git", str(CLONE_DIR)],
               check=True)

subprocess.run(["git", "-C", str(CLONE_DIR), "lfs", "install", "--local", "--skip-smudge"],
               check=True)

print("Fetching the brca_metabric directory (may take a few minutes)...")
result = subprocess.run(
    ["git", "-C", str(CLONE_DIR), "-c", "lfs.fetchexclude=",
     "lfs", "pull", "-I", "public/brca_metabric"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout[-1500:])
print("STDERR:", result.stderr[-1500:])

src = CLONE_DIR / "public" / "brca_metabric"
print(f"\nFiles fetched into {src} :")
for f in sorted(src.glob("*")):
    size = f.stat().st_size
    flag = "unresolved LFS pointer" if size < 2000 and f.suffix == ".txt" else ""
    print(f"  {f.name:<55} {size:>12,} octets  {flag}")

dest = DATA_DIR / "brca_metabric_raw"
import shutil
shutil.copytree(src, dest, dirs_exist_ok=True)
print(f"\nCopied to the data directory:  {dest}")

shutil.rmtree(CLONE_DIR, ignore_errors=True)
print("Temporary directory cleaned.")

**Cohort check.** Reports the METABRIC class distribution and the genes shared with the discovery cohort.

In [ ]:
import pandas as pd

METABRIC_DIR = Path(str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / 'brca_metabric_raw'))

print("=== Colonnes de data_clinical_patient.txt ===")
clin = pd.read_csv(METABRIC_DIR / "data_clinical_patient.txt", sep="\t", comment="#", low_memory=False)
print(clin.columns.tolist())
print()

# Recherche automatique des colonnes candidates PAM50/sous-type
candidates = [c for c in clin.columns if any(k in c.upper() for k in ["SUBTYPE", "PAM50", "CLAUDIN"])]
print("Candidate columns for the molecular subtype: ", candidates)
for c in candidates:
    print(f"\n--- {c} ---")
    print(clin[c].value_counts(dropna=False))

print("\n=== Preview of data_mrna_illumina_microarray.txt (5 first rows/columns) ===")
mrna_preview = pd.read_csv(METABRIC_DIR / "data_mrna_illumina_microarray.txt",
                           sep="\t", nrows=5)
print(f"Shape (preview):  {mrna_preview.shape}")
print("Colonnes (first) :", mrna_preview.columns[:8].tolist())
print(mrna_preview.iloc[:5, :6])

### External importance reproducibility


**Level 7, first question.** Re-executes the pipeline independently within METABRIC and compares gene-level importance rankings with TCGA-BRCA across several truncation thresholds.

In [ ]:
"""
===============================================================================
— VALIDATION EXTERNE METABRIC
Phase 1: preprocessing + Approach A (gene-importance correlation)
===============================================================================
"""

import os, json, time, hashlib, warnings
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from scipy.stats import spearmanr
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")

from google.colab import drive
if not os.path.exists(str(DATA_DIR)):
    drive.mount("/content/drive")

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
METABRIC_DIR = DATA_DIR / "brca_metabric_raw"
TCGA_RUN_DIR = DATA_DIR / "runs" / "2026-07-26"
OUT_DIR = TCGA_RUN_DIR / "results" / "model_evaluation" / "metabric_external_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_SUBTYPES = ["LumA", "LumB", "Her2", "Basal", "Normal"]
MIN_VARIANCE = 1e-6
FILTER_TOP_K = 5000
N_COMPONENTS = 50  # dim_k of the main k50 regime

EVAL_CONFIG = {
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 2, "n_jobs": -1},
    "xgboost": {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.85,
               "colsample_bytree": 0.85, "reg_lambda": 1.0, "reg_alpha": 0.0, "min_child_weight": 1.0, "n_jobs": -1},
    "svm": {"C": 1.0, "kernel": "rbf", "gamma": "scale", "class_weight": "balanced", "probability": False},
}
CLASSIFIERS = ["RF", "XGB", "SVM"]


def atomic_csv(df, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def stable_seed(*parts):
    return int(hashlib.sha256("||".join(map(str, parts)).encode()).hexdigest()[:8], 16)


# =============================================================================
# 1. CHARGEMENT ET NETTOYAGE CLINIQUE
# =============================================================================

clin = pd.read_csv(METABRIC_DIR / "data_clinical_patient.txt", sep="\t", comment="#", low_memory=False)
labels_all = clin.set_index("PATIENT_ID")["CLAUDIN_SUBTYPE"].dropna()
n_before_filter = len(labels_all)
labels_all = labels_all[labels_all.isin(VALID_SUBTYPES)]
print(f"CLAUDIN_SUBTYPE : {n_before_filter} labels bruts -> {len(labels_all)} "
      f"after restriction aux 5 classes compatibles PAM50")

# =============================================================================
# 2. CHARGEMENT mRNA -- Hugo_Symbol en index, fusion des doublons
# =============================================================================

mrna_full = pd.read_csv(METABRIC_DIR / "data_mrna_illumina_microarray.txt", sep="\t", low_memory=False)
mrna_full = mrna_full.drop(columns=["Entrez_Gene_Id"], errors="ignore")
mrna_full = mrna_full.dropna(subset=["Hugo_Symbol"])
mrna_full = mrna_full.groupby("Hugo_Symbol").mean(numeric_only=True)  # fusion doublons de symboles
mrna_full = mrna_full.T  # patients en rows, genes en colonnes
mrna_full.index.name = "PATIENT_ID"

print(f"mRNA METABRIC : {mrna_full.shape} (patients x genes)")

# =============================================================================
# 3. COHORTE FINALE METABRIC
# =============================================================================

common_patients = sorted(set(labels_all.index) & set(mrna_full.index))
print(f"Cohorte finale METABRIC (labels + mRNA disponibles) : {len(common_patients)} patients")

labels_metabric = labels_all.loc[common_patients]
mrna_metabric = mrna_full.loc[common_patients]

le_metabric = LabelEncoder()
y_metabric = le_metabric.fit_transform(labels_metabric.to_numpy())
print("Classes METABRIC :", dict(zip(le_metabric.classes_, np.bincount(y_metabric).tolist())))

pd.DataFrame({"patient_id": common_patients, "claudin_subtype": labels_metabric.values,
             "label_encoded": y_metabric}).to_csv(OUT_DIR / "metabric_cohort_manifest.csv", index=False)

# =============================================================================
# 4. GENES SHARED BETWEEN TCGA AND METABRIC
# =============================================================================

tcga_genes = set(pd.read_parquet(DATA_DIR / "mrna_hugo_mapped_primary_only.parquet").columns)
metabric_genes = set(mrna_metabric.columns)
shared_genes = sorted(tcga_genes & metabric_genes)
print(f"Genes shared TCGA/METABRIC : {len(shared_genes)} "
      f"(TCGA={len(tcga_genes)}, METABRIC={len(metabric_genes)})")

mrna_metabric_shared = mrna_metabric[shared_genes]

print(f"NaN dans mrna_metabric_shared : {mrna_metabric_shared.isna().sum().sum()}")
print(f"Proportion de patients avec au moins un NaN : "
      f"{mrna_metabric_shared.isna().any(axis=1).mean():.4f}")


# =============================================================================
# 5. APPROACH A - TRAINING ON METABRIC, GENE-LEVEL SHAP, REPEATED CV
# =============================================================================

def score_mad(df_train):
    medians = df_train.median(axis=0)
    return (df_train - medians).abs().median(axis=0)


def make_classifier(name, seed, n_classes):
    if name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if name == "XGB":
        return XGBClassifier(objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
                             random_state=seed, verbosity=0, **EVAL_CONFIG["xgboost"])
    if name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    raise ValueError(name)


N_CLASSES_METABRIC = len(np.unique(y_metabric))
if np.bincount(y_metabric).min() < 5:
    raise ValueError("A METABRIC class has fewer than 5 patients - reduce cv_folds.")

splitter = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)
SPLITS_MB = list(splitter.split(np.zeros(len(y_metabric)), y_metabric))
REPEAT_OF_MB = [i // 5 for i in range(len(SPLITS_MB))]

shap_path_mb = OUT_DIR / "metabric_gene_shap_by_split.csv"
if shap_path_mb.exists():
    shap_rows_mb = pd.read_csv(shap_path_mb).to_dict("records")
    done_splits_mb = set(pd.read_csv(shap_path_mb)["split"].unique().tolist())
    print(f"Resuming:  {len(done_splits_mb)}/{len(SPLITS_MB)} splits already processed")
else:
    shap_rows_mb, done_splits_mb = [], set()

t0 = time.time()
for split_index, (train_index, test_index) in enumerate(SPLITS_MB):
    if split_index in done_splits_mb:
        continue
    repeat_index = REPEAT_OF_MB[split_index]
    seed = int(SEED + repeat_index)
    y_train, y_test = y_metabric[train_index], y_metabric[test_index]
    sample_weight = compute_sample_weight("balanced", y_train)

    train_df = mrna_metabric_shared.iloc[train_index]
    test_df = mrna_metabric_shared.iloc[test_index]

    # -- Filtre de variance (train uniquement) --
    variances = train_df.var(axis=0)
    keep = variances[variances > MIN_VARIANCE].index
    train_df, test_df = train_df[keep], test_df[keep]

    # -- Top-5000 MAD (train uniquement) --
    if train_df.shape[1] > FILTER_TOP_K:
        mad_scores = score_mad(train_df)
        selected = mad_scores.nlargest(FILTER_TOP_K).index
        train_df, test_df = train_df[selected], test_df[selected]
    gene_names_split = list(train_df.columns)

    # -- Median imputation (training rows only) --
    # Les puces Illumina METABRIC contiennent des valeurs manquantes,
    # unlike TCGA RNA-seq - a step absent from v1 and corrected here.
    train_medians = train_df.median(axis=0)
    train_df = train_df.fillna(train_medians)
    test_df = test_df.fillna(train_medians)
    # Safety net: a gene entirely NaN on the training rows -> NaN median
    # -> it is removed cleanly rather than leaving a residual NaN.
    still_nan = train_df.columns[train_df.isna().any()]
    if len(still_nan) > 0:
        train_df = train_df.drop(columns=still_nan)
        test_df = test_df.drop(columns=still_nan)
        gene_names_split = list(train_df.columns)

    # -- Standardisation --
    scaler = StandardScaler().fit(train_df.to_numpy(dtype=np.float64))
    X_train = scaler.transform(train_df.to_numpy(dtype=np.float64))
    X_test = scaler.transform(test_df.to_numpy(dtype=np.float64))

    # -- PCA --
    reducer = PCA(n_components=N_COMPONENTS, random_state=seed).fit(X_train)
    Z_train = reducer.transform(X_train).astype(np.float32)
    Z_test = reducer.transform(X_test).astype(np.float32)
    loadings = reducer.components_

    for clf_name in CLASSIFIERS:
        model = make_classifier(clf_name, seed, N_CLASSES_METABRIC)
        if clf_name in {"RF", "XGB"}:
            model.fit(Z_train, y_train, sample_weight=sample_weight)
        else:
            model.fit(Z_train, y_train)

        if clf_name in {"RF", "XGB"}:
            explainer = shap.TreeExplainer(model)
            raw_shap = explainer.shap_values(Z_test)
            if isinstance(raw_shap, list):
                comp_importance = np.mean([np.abs(s).mean(axis=0) for s in raw_shap], axis=0)
            else:
                comp_importance = (np.abs(raw_shap).mean(axis=(0, 2)) if raw_shap.ndim == 3
                                   else np.abs(raw_shap).mean(axis=0))
        else:
            perm = permutation_importance(model, Z_test, y_test, n_repeats=10,
                                          random_state=seed, scoring="balanced_accuracy", n_jobs=-1)
            comp_importance = np.abs(perm.importances_mean)

        gene_importance = comp_importance @ np.abs(loadings)
        for gene, imp in zip(gene_names_split, gene_importance):
            shap_rows_mb.append({"split": split_index, "repeat": repeat_index,
                                 "classifier": clf_name, "gene": gene, "importance_abs": float(imp)})

    if (split_index + 1) % 5 == 0 or split_index == len(SPLITS_MB) - 1:
        atomic_csv(pd.DataFrame(shap_rows_mb), shap_path_mb)
        print(f"  split {split_index + 1}/{len(SPLITS_MB)} saved ({time.time()-t0:.0f}s)")

shap_df_mb = pd.DataFrame(shap_rows_mb)

gene_summary_mb = (shap_df_mb.groupby(["classifier", "gene"], as_index=False)["importance_abs"]
                   .mean().rename(columns={"importance_abs": "mean_importance_abs_metabric"}))
atomic_csv(gene_summary_mb, OUT_DIR / "metabric_gene_importance_summary.csv")

# =============================================================================
# 6. TCGA vs METABRIC CORRELATION, ACROSS SEVERAL TOP-N
# =============================================================================

tcga_gene_summary = pd.read_csv(
    TCGA_RUN_DIR / "results" / "model_evaluation" / "shap_gene_level_minimal_panel" / "gene_importance_summary.csv"
).rename(columns={"mean_importance_abs": "mean_importance_abs_tcga"})

corr_rows = []
for clf_name in CLASSIFIERS:
    tcga_clf = tcga_gene_summary[tcga_gene_summary.classifier == clf_name].set_index("gene")["mean_importance_abs_tcga"]
    mb_clf = gene_summary_mb[gene_summary_mb.classifier == clf_name].set_index("gene")["mean_importance_abs_metabric"]

    shared = sorted(set(tcga_clf.index) & set(mb_clf.index))
    for top_n in (100, 150, 200, 300, 500, len(shared)):
        top_genes_tcga = tcga_clf.reindex(shared).nlargest(min(top_n, len(shared))).index
        genes_for_corr = [g for g in shared if g in top_genes_tcga]
        if len(genes_for_corr) < 10:
            continue
        rho, p = spearmanr(tcga_clf.loc[genes_for_corr], mb_clf.loc[genes_for_corr])
        corr_rows.append({"classifier": clf_name, "top_n": top_n, "n_genes_shared": len(shared),
                          "n_genes_used": len(genes_for_corr), "spearman_rho": float(rho), "p_value": float(p)})

corr_df = pd.DataFrame(corr_rows)
atomic_csv(corr_df, OUT_DIR / "tcga_metabric_gene_importance_correlation.csv")
print("\nTCGA vs METABRIC gene-importance correlation: ")
print(corr_df.to_string(index=False))

print(f"\nComplete. Results : {OUT_DIR}")

### Direct transfer to METABRIC


**Level 7, second question.** Fits one model per classifier on the full discovery cohort and applies it to METABRIC without retraining, under two expression-processing arms: unharmonised transfer, and transfer after within-cohort rank normalisation.

In [ ]:
"""
===============================================================================
— VALIDATION EXTERNE METABRIC
Phase 2: Approach B - direct transfer (trained on TCGA, tested on METABRIC)
===============================================================================
"""

import os, json, time, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

from google.colab import drive
if not os.path.exists(str(DATA_DIR)):
    drive.mount("/content/drive")

DATA_DIR = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
METABRIC_DIR = DATA_DIR / "brca_metabric_raw"
TCGA_RUN_DIR = DATA_DIR / "runs" / "2026-07-26"
OUT_DIR = TCGA_RUN_DIR / "results" / "model_evaluation" / "metabric_external_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VALID_SUBTYPES = ["LumA", "LumB", "Her2", "Basal", "Normal"]
MIN_VARIANCE = 1e-6
FILTER_TOP_K = 5000
N_COMPONENTS = 50
N_BOOTSTRAP = 5000
CLASSIFIERS = ["RF", "XGB", "SVM"]

EVAL_CONFIG = {
    "random_forest": {"n_estimators": 500, "max_features": "sqrt", "min_samples_leaf": 2, "n_jobs": -1},
    "xgboost": {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03, "subsample": 0.85,
               "colsample_bytree": 0.85, "reg_lambda": 1.0, "reg_alpha": 0.0, "min_child_weight": 1.0, "n_jobs": -1},
    "svm": {"C": 1.0, "kernel": "rbf", "gamma": "scale", "class_weight": "balanced", "probability": False},
}


def atomic_csv(df, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(obj, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def stable_int_seed(*parts):
    """Convert a string into a deterministic integer seed compatible with np.random.default_rng."""
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def make_classifier(name, seed, n_classes):
    if name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if name == "XGB":
        return XGBClassifier(objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
                             random_state=seed, verbosity=0, **EVAL_CONFIG["xgboost"])
    if name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    raise ValueError(name)


def score_mad(df_train):
    medians = df_train.median(axis=0)
    return (df_train - medians).abs().median(axis=0)


# =============================================================================
# 1. TCGA LOADING (complete cohort, mRNA, labels)
# =============================================================================

tcga_cohort = pd.read_csv(TCGA_RUN_DIR / "results" / "cohort_manifest.csv")
tcga_mrna_full = pd.read_parquet(DATA_DIR / "mrna_hugo_mapped_primary_only.parquet")
tcga_mrna_full.index = tcga_mrna_full.index.astype(str)
tcga_mrna_full = tcga_mrna_full.loc[tcga_cohort["patient_id"].astype(str)]

le_tcga = LabelEncoder()
y_tcga = le_tcga.fit_transform(tcga_cohort["pam50"].to_numpy())
print(f"TCGA : {len(y_tcga)} patients, classes = {list(le_tcga.classes_)}")

# =============================================================================
# 2. METABRIC LOADING (complete cohort, mRNA, labels)
# =============================================================================

clin = pd.read_csv(METABRIC_DIR / "data_clinical_patient.txt", sep="\t", comment="#", low_memory=False)
labels_all = clin.set_index("PATIENT_ID")["CLAUDIN_SUBTYPE"].dropna()
labels_all = labels_all[labels_all.isin(VALID_SUBTYPES)]

mrna_full_mb = pd.read_csv(METABRIC_DIR / "data_mrna_illumina_microarray.txt", sep="\t", low_memory=False)
mrna_full_mb = mrna_full_mb.drop(columns=["Entrez_Gene_Id"], errors="ignore").dropna(subset=["Hugo_Symbol"])
mrna_full_mb = mrna_full_mb.groupby("Hugo_Symbol").mean(numeric_only=True).T
mrna_full_mb.index.name = "PATIENT_ID"

common_mb = sorted(set(labels_all.index) & set(mrna_full_mb.index))
labels_mb = labels_all.loc[common_mb]
mrna_mb = mrna_full_mb.loc[common_mb]

le_mb = LabelEncoder()
y_mb = le_mb.fit_transform(labels_mb.to_numpy())
assert list(le_mb.classes_) == list(le_tcga.classes_), \
    f"Class order differs:  TCGA={list(le_tcga.classes_)} vs METABRIC={list(le_mb.classes_)}"
print(f"METABRIC : {len(y_mb)} patients, classes = {list(le_mb.classes_)}")

# =============================================================================
# 3. GENE SELECTION - restricted to shared genes from the start
# =============================================================================

shared_genes = sorted(set(tcga_mrna_full.columns) & set(mrna_mb.columns))
print(f"Genes shared (univers de start pour la selection) : {len(shared_genes)}")

tcga_shared = tcga_mrna_full[shared_genes]
mb_shared = mrna_mb[shared_genes]

variances = tcga_shared.var(axis=0)
keep = variances[variances > MIN_VARIANCE].index
tcga_selected = tcga_shared[keep]

if tcga_selected.shape[1] > FILTER_TOP_K:
    mad_scores = score_mad(tcga_selected)
    selected_genes = mad_scores.nlargest(FILTER_TOP_K).index.tolist()
else:
    selected_genes = list(tcga_selected.columns)

print(f"Selected genes (variance + top-{FILTER_TOP_K} MAD, on TCGA):  {len(selected_genes)}")
print("Tous present dans METABRIC par construction :",
      set(selected_genes).issubset(set(mb_shared.columns)))

tcga_final = tcga_shared[selected_genes]
mb_final = mb_shared[selected_genes]

# Imputation (TCGA median, applied to both sides for consistency)
tcga_medians = tcga_final.median(axis=0)
tcga_final = tcga_final.fillna(tcga_medians)
mb_final = mb_final.fillna(tcga_medians)
print(f"Residual NaN - TCGA: {tcga_final.isna().sum().sum()}, METABRIC: {mb_final.isna().sum().sum()}")

# =============================================================================
# 4. AJUSTEMENT UNIQUE SUR TCGA COMPLET -- standardisation + PCA
# =============================================================================

scaler = StandardScaler().fit(tcga_final.to_numpy(dtype=np.float64))
X_tcga = scaler.transform(tcga_final.to_numpy(dtype=np.float64))
X_mb = scaler.transform(mb_final.to_numpy(dtype=np.float64))

reducer = PCA(n_components=N_COMPONENTS, random_state=SEED).fit(X_tcga)
Z_tcga = reducer.transform(X_tcga).astype(np.float32)
Z_mb = reducer.transform(X_mb).astype(np.float32)
print(f"Variance retained by PCA (fitted on the full TCGA cohort):  "
      f"{reducer.explained_variance_ratio_.sum()*100:.1f}%")

n_classes = len(le_tcga.classes_)
sample_weight_tcga = compute_sample_weight("balanced", y_tcga)

# =============================================================================
# 5. SINGLE FIT (full TCGA) + EXTERNAL PREDICTION (METABRIC)
# =============================================================================

results_rows = []
predictions_mb = {}

for clf_name in CLASSIFIERS:
    model = make_classifier(clf_name, SEED, n_classes)
    if clf_name in {"RF", "XGB"}:
        model.fit(Z_tcga, y_tcga, sample_weight=sample_weight_tcga)
    else:
        model.fit(Z_tcga, y_tcga)

    y_pred_mb = model.predict(Z_mb)
    predictions_mb[clf_name] = y_pred_mb

    ba_point = float(balanced_accuracy_score(y_mb, y_pred_mb))

    # Bootstrap sur la cohorte METABRIC (pas de retraining, juste resampling)
    rng = np.random.default_rng(stable_int_seed("metabric_boot", clf_name))
    class_indices = {c: np.flatnonzero(y_mb == c) for c in np.unique(y_mb)}
    boot_scores = []
    for _ in range(N_BOOTSTRAP):
        idx = np.concatenate([rng.choice(ix, len(ix), replace=True) for ix in class_indices.values()])
        boot_scores.append(balanced_accuracy_score(y_mb[idx], y_pred_mb[idx]))
    boot_scores = np.asarray(boot_scores)
    ci_lo, ci_hi = np.quantile(boot_scores, [0.025, 0.975])

    results_rows.append({
        "classifier": clf_name,
        "metabric_balanced_accuracy": ba_point,
        "ci95_low": float(ci_lo), "ci95_high": float(ci_hi),
        "n_metabric_patients": len(y_mb),
    })
    print(f"{clf_name:<8} METABRIC BA={ba_point:.4f}  IC95%=[{ci_lo:.4f},{ci_hi:.4f}]")

results_df = pd.DataFrame(results_rows)
atomic_csv(results_df, OUT_DIR / "metabric_direct_transfer_performance.csv")

# =============================================================================
# 6. DIAGNOSTIC PAR CLASSE (recall par sous-type PAM50)
# =============================================================================

per_class_rows = []
for clf_name in CLASSIFIERS:
    report = classification_report(y_mb, predictions_mb[clf_name],
                                    target_names=le_tcga.classes_, output_dict=True)
    for cls in le_tcga.classes_:
        per_class_rows.append({
            "classifier": clf_name, "class": cls,
            "recall": report[cls]["recall"], "precision": report[cls]["precision"],
            "f1": report[cls]["f1-score"], "support": report[cls]["support"],
        })
per_class_df = pd.DataFrame(per_class_rows)
atomic_csv(per_class_df, OUT_DIR / "metabric_direct_transfer_per_class.csv")
print("\nRecall par classe :")
print(per_class_df.pivot(index="class", columns="classifier", values="recall").round(3))

# =============================================================================
# 7. COMPARISON WITH INTERNAL TCGA PERFORMANCE (reference)
# =============================================================================

tcga_internal = pd.read_csv(TCGA_RUN_DIR / "results" / "model_evaluation" / "panel_fold_results.csv")
tcga_internal_mrna = (tcga_internal[tcga_internal.panel == "mrna"]
                      .groupby("classifier")["balanced_accuracy"].mean())

comparison_rows = []
for clf_name in CLASSIFIERS:
    comparison_rows.append({
        "classifier": clf_name,
        "tcga_internal_cv_ba": float(tcga_internal_mrna.get(clf_name, np.nan)),
        "metabric_external_ba": float(results_df[results_df.classifier == clf_name]["metabric_balanced_accuracy"].iloc[0]),
    })
comparison_df = pd.DataFrame(comparison_rows)
comparison_df["gap"] = comparison_df["tcga_internal_cv_ba"] - comparison_df["metabric_external_ba"]
atomic_csv(comparison_df, OUT_DIR / "metabric_vs_tcga_internal_comparison.csv")
print("\nComparaison performance interne (TCGA, CV) vs externe (METABRIC, transfert direct) :")
print(comparison_df.to_string(index=False))

# =============================================================================
# 8. RAPPORT FINAL
# =============================================================================

report = {
    "approach": "direct_transfer_B",
    "panel": "mrna",
    "n_shared_genes_pool": len(shared_genes),
    "n_genes_selected_for_training": len(selected_genes),
    "n_pca_components": N_COMPONENTS,
    "pca_variance_retained": float(reducer.explained_variance_ratio_.sum()),
    "n_tcga_train": len(y_tcga),
    "n_metabric_test": len(y_mb),
    "performance": results_df.to_dict("records"),
    "comparison_to_internal_cv": comparison_df.to_dict("records"),
    "caveats": [
        "Single model fit on the full TCGA cohort (no cross-validation on the "
        "training side); uncertainty is quantified via bootstrap resampling "
        "of the METABRIC test cohort only.",
        "Gene selection (variance filter, top-5000 MAD) was restricted to "
        "genes already shared with METABRIC, to guarantee exact transferability "
        "of the fitted scaler and PCA without imputing missing genes.",
        "CLAUDIN_SUBTYPE is used as the METABRIC PAM50 proxy label, which "
        "differs slightly from a canonical PAM50 centroid assignment.",
    ],
}
atomic_json(report, OUT_DIR / "metabric_direct_transfer_report.json")

print(f"\nComplete. Results : {OUT_DIR}")

**Transfer outputs.** Writes the two-arm transfer performance, the per-class recalls, and the comparison with internal cross-validated performance.

In [ ]:
"""
===============================================================================
EXTERNAL VALIDATION -- METABRIC
Phase 2bis : Approche B -- transfert direct, DEUX BRAS

  Raw arm: direct transfer using the reported mRNA representation without cross-platform harmonisation. Scaler et PCA ajustes sur des
                    valeurs log2(TPM+1) RNA-seq, appliques sans harmonisation a
                    des intensites microarray.
  Bras "ranknorm" : identique en tout point, sauf que les expressions sont
                    normalisees par rang au sein de chaque cohorte, gene par
                    gene, AVANT l'ajustement du scaler et de la PCA.

The raw arm is a reproducibility control for the direct-transfer result reported in the manuscript and should reproduce the recorded reference values (RF 0.207, XGB 0.254, SVM 0.200) a l'arrondi pres. S'il ne les
reproduit pas, les entrees ou la selection de genes ont change et le bras
"ranknorm" n'est pas interpretable.

This script overwrites no existing file: every output carries the
suffixe _two_arms.
===============================================================================
"""

import os, json, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

from google.colab import drive
if not os.path.exists(str(DATA_DIR)):
    drive.mount("/content/drive")

DATA_DIR     = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
METABRIC_DIR = DATA_DIR / "brca_metabric_raw"
TCGA_RUN_DIR = DATA_DIR / "runs" / "2026-07-26"
OUT_DIR      = TCGA_RUN_DIR / "results" / "model_evaluation" / "metabric_external_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED           = 42
VALID_SUBTYPES = ["LumA", "LumB", "Her2", "Basal", "Normal"]
MIN_VARIANCE   = 1e-6
FILTER_TOP_K   = 5000
N_COMPONENTS   = 50
N_BOOTSTRAP    = 5000
CLASSIFIERS    = ["RF", "XGB", "SVM"]
ARMS           = ["raw", "ranknorm"]

EVAL_CONFIG = {
    "random_forest": {"n_estimators": 500, "max_features": "sqrt",
                      "min_samples_leaf": 2, "n_jobs": -1},
    "xgboost": {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03,
                "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 1.0,
                "reg_alpha": 0.0, "min_child_weight": 1.0, "n_jobs": -1},
    "svm": {"C": 1.0, "kernel": "rbf", "gamma": "scale",
            "class_weight": "balanced", "probability": False},
}


def atomic_csv(df, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(obj, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def stable_int_seed(*parts):
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def make_classifier(name, seed, n_classes):
    if name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if name == "XGB":
        return XGBClassifier(objective="multi:softprob", num_class=n_classes,
                             eval_metric="mlogloss", random_state=seed,
                             verbosity=0, **EVAL_CONFIG["xgboost"])
    if name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    raise ValueError(name)


def score_mad(df_train):
    medians = df_train.median(axis=0)
    return (df_train - medians).abs().median(axis=0)


def rank_normalise(df):
    """Normalisation par rang, colonne par colonne, au sein de la cohorte.

    Les rangs bruts dependent de l'effectif (1..677 pour TCGA, 1..1756 pour
    METABRIC). La division par n+1 rend la transformation independante de la
    taille de la cohorte, ce qui est la condition pour qu'un scaler ajuste sur
    TCGA s'applique legitimement a METABRIC. Les ex aequo recoivent leur rang
    moyen.
    """
    return df.rank(axis=0, method="average") / (len(df) + 1.0)


# =============================================================================
# 1. CHARGEMENT TCGA
# =============================================================================

tcga_cohort = pd.read_csv(TCGA_RUN_DIR / "results" / "cohort_manifest.csv")
tcga_mrna_full = pd.read_parquet(DATA_DIR / "mrna_hugo_mapped_primary_only.parquet")
tcga_mrna_full.index = tcga_mrna_full.index.astype(str)
tcga_mrna_full = tcga_mrna_full.loc[tcga_cohort["patient_id"].astype(str)]

le_tcga = LabelEncoder()
y_tcga = le_tcga.fit_transform(tcga_cohort["pam50"].to_numpy())
print(f"TCGA : {len(y_tcga)} patients, classes = {list(le_tcga.classes_)}")

# =============================================================================
# 2. CHARGEMENT METABRIC
# =============================================================================

clin = pd.read_csv(METABRIC_DIR / "data_clinical_patient.txt", sep="\t",
                   comment="#", low_memory=False)
labels_all = clin.set_index("PATIENT_ID")["CLAUDIN_SUBTYPE"].dropna()
labels_all = labels_all[labels_all.isin(VALID_SUBTYPES)]

mrna_full_mb = pd.read_csv(METABRIC_DIR / "data_mrna_illumina_microarray.txt",
                           sep="\t", low_memory=False)
mrna_full_mb = (mrna_full_mb.drop(columns=["Entrez_Gene_Id"], errors="ignore")
                            .dropna(subset=["Hugo_Symbol"]))
mrna_full_mb = mrna_full_mb.groupby("Hugo_Symbol").mean(numeric_only=True).T
mrna_full_mb.index.name = "PATIENT_ID"

common_mb = sorted(set(labels_all.index) & set(mrna_full_mb.index))
labels_mb = labels_all.loc[common_mb]
mrna_mb = mrna_full_mb.loc[common_mb]

le_mb = LabelEncoder()
y_mb = le_mb.fit_transform(labels_mb.to_numpy())
assert list(le_mb.classes_) == list(le_tcga.classes_), \
    f"Ordre des classes different : TCGA={list(le_tcga.classes_)} vs METABRIC={list(le_mb.classes_)}"
print(f"METABRIC : {len(y_mb)} patients, classes = {list(le_mb.classes_)}")

# =============================================================================
# 3. SELECTION DE GENES -- restreinte aux genes partages des le depart
# =============================================================================

shared_genes = sorted(set(tcga_mrna_full.columns) & set(mrna_mb.columns))
print(f"Genes partages : {len(shared_genes)}")

tcga_shared = tcga_mrna_full[shared_genes]
mb_shared = mrna_mb[shared_genes]

variances = tcga_shared.var(axis=0)
keep = variances[variances > MIN_VARIANCE].index
tcga_selected = tcga_shared[keep]

if tcga_selected.shape[1] > FILTER_TOP_K:
    mad_scores = score_mad(tcga_selected)
    selected_genes = mad_scores.nlargest(FILTER_TOP_K).index.tolist()
else:
    selected_genes = list(tcga_selected.columns)

print(f"Genes selectionnes (variance + top-{FILTER_TOP_K} MAD, on TCGA):  "
      f"{len(selected_genes)}")

# ordre de colonnes identique des deux cotes, impose explicitement
tcga_final = tcga_shared[selected_genes]
mb_final = mb_shared[selected_genes]

# imputation par les medianes TCGA, appliquee aux deux cotes
tcga_medians = tcga_final.median(axis=0)
tcga_final = tcga_final.fillna(tcga_medians)
mb_final = mb_final.fillna(tcga_medians)
print(f"NaN residuels -- TCGA: {tcga_final.isna().sum().sum()}, "
      f"METABRIC: {mb_final.isna().sum().sum()}")

# =============================================================================
# 4-6. BOUCLE SUR LES DEUX BRAS
# =============================================================================

n_classes = len(le_tcga.classes_)
sample_weight_tcga = compute_sample_weight("balanced", y_tcga)
class_indices = {c: np.flatnonzero(y_mb == c) for c in np.unique(y_mb)}

results_rows, per_class_rows, arm_meta = [], [], {}
predictions_mb = {}

for arm in ARMS:
    print(f"\n{'='*70}\nBRAS : {arm}\n{'='*70}")

    if arm == "raw":
        A_tcga, A_mb = tcga_final, mb_final
    else:
        # transformation ajustee INDEPENDAMMENT dans chaque cohorte : c'est le
        # principe meme de la correction d'echelle
        A_tcga, A_mb = rank_normalise(tcga_final), rank_normalise(mb_final)

    scaler = StandardScaler().fit(A_tcga.to_numpy(dtype=np.float64))
    X_tcga = scaler.transform(A_tcga.to_numpy(dtype=np.float64))
    X_mb = scaler.transform(A_mb.to_numpy(dtype=np.float64))

    reducer = PCA(n_components=N_COMPONENTS, random_state=SEED).fit(X_tcga)
    Z_tcga = reducer.transform(X_tcga).astype(np.float32)
    Z_mb = reducer.transform(X_mb).astype(np.float32)
    var_ret = float(reducer.explained_variance_ratio_.sum())
    arm_meta[arm] = {"pca_variance_retained": var_ret}
    print(f"Variance retenue par la PCA : {var_ret*100:.1f}%")

    for clf_name in CLASSIFIERS:
        model = make_classifier(clf_name, SEED, n_classes)
        if clf_name in {"RF", "XGB"}:
            model.fit(Z_tcga, y_tcga, sample_weight=sample_weight_tcga)
        else:
            model.fit(Z_tcga, y_tcga)

        y_pred_mb = model.predict(Z_mb)
        predictions_mb[(arm, clf_name)] = y_pred_mb
        ba_point = float(balanced_accuracy_score(y_mb, y_pred_mb))
        n_pred_classes = int(len(np.unique(y_pred_mb)))

        # bootstrap stratifie par classe sur la cohorte METABRIC uniquement
        rng = np.random.default_rng(stable_int_seed("metabric_boot", arm, clf_name))
        boot = np.empty(N_BOOTSTRAP)
        for b in range(N_BOOTSTRAP):
            idx = np.concatenate([rng.choice(ix, len(ix), replace=True)
                                  for ix in class_indices.values()])
            boot[b] = balanced_accuracy_score(y_mb[idx], y_pred_mb[idx])
        ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])

        results_rows.append({
            "arm": arm, "classifier": clf_name,
            "metabric_balanced_accuracy": ba_point,
            "ci95_low": float(ci_lo), "ci95_high": float(ci_hi),
            "n_metabric_patients": len(y_mb),
            "n_distinct_predicted_classes": n_pred_classes,
        })
        print(f"{clf_name:<5} BA={ba_point:.4f}  IC95%=[{ci_lo:.4f},{ci_hi:.4f}]"
              f"  classes predites={n_pred_classes}")

        rep = classification_report(y_mb, y_pred_mb,
                                    target_names=le_tcga.classes_,
                                    output_dict=True, zero_division=0)
        for cls in le_tcga.classes_:
            per_class_rows.append({
                "arm": arm, "classifier": clf_name, "class": cls,
                "recall": rep[cls]["recall"], "precision": rep[cls]["precision"],
                "f1": rep[cls]["f1-score"], "support": rep[cls]["support"],
            })

results_df = pd.DataFrame(results_rows)
per_class_df = pd.DataFrame(per_class_rows)
atomic_csv(results_df, OUT_DIR / "metabric_direct_transfer_performance_two_arms.csv")
atomic_csv(per_class_df, OUT_DIR / "metabric_direct_transfer_per_class_two_arms.csv")

# =============================================================================
# 7. CONTROLE DE REPRODUCTION DU BRAS RAW
# =============================================================================

PUBLISHED = {"RF": 0.2065, "XGB": 0.2543, "SVM": 0.2000}
print(f"\n{'='*70}\nCONTROLE : le bras raw reproduit-il les valeurs publiees ?\n{'='*70}")
raw = results_df[results_df.arm == "raw"].set_index("classifier")
ctrl_ok = True
for clf_name, expected in PUBLISHED.items():
    got = float(raw.loc[clf_name, "metabric_balanced_accuracy"])
    delta = abs(got - expected)
    flag = "OK" if delta < 0.005 else "ECART"
    if delta >= 0.005:
        ctrl_ok = False
    print(f"{clf_name:<5} publie={expected:.4f}  obtenu={got:.4f}  "
          f"ecart={delta:.4f}  {flag}")
if not ctrl_ok:
    print("\nATTENTION : le bras raw ne reproduit pas le run publie. Les entrees "
          "or the gene selection differ; the rank-normalised arm is not "
          "interpretable en l'etat.")

# =============================================================================
# 8. EFFET DE LA NORMALISATION PAR RANG
# =============================================================================

piv = results_df.pivot(index="classifier", columns="arm",
                       values="metabric_balanced_accuracy")
piv["delta"] = piv["ranknorm"] - piv["raw"]
print(f"\n{'='*70}\nEFFET DE LA NORMALISATION PAR RANG\n{'='*70}")
print(piv.round(4).to_string())

print("\nRecall par classe, bras ranknorm :")
print(per_class_df[per_class_df.arm == "ranknorm"]
      .pivot(index="class", columns="classifier", values="recall").round(3))

# =============================================================================
# 9. COMPARAISON A LA PERFORMANCE INTERNE TCGA
# =============================================================================

tcga_internal = pd.read_csv(TCGA_RUN_DIR / "results" / "model_evaluation" /
                            "panel_fold_results.csv")
tcga_internal_mrna = (tcga_internal[tcga_internal.panel == "mrna"]
                      .groupby("classifier")["balanced_accuracy"].mean())

comparison_rows = []
for arm in ARMS:
    sub = results_df[results_df.arm == arm].set_index("classifier")
    for clf_name in CLASSIFIERS:
        internal = float(tcga_internal_mrna.get(clf_name, np.nan))
        external = float(sub.loc[clf_name, "metabric_balanced_accuracy"])
        comparison_rows.append({
            "arm": arm, "classifier": clf_name,
            "tcga_internal_cv_ba": internal,
            "metabric_external_ba": external,
            "gap": internal - external,
        })
comparison_df = pd.DataFrame(comparison_rows)
atomic_csv(comparison_df,
           OUT_DIR / "metabric_vs_tcga_internal_comparison_two_arms.csv")
print("\nInterne (TCGA, CV) vs externe (METABRIC), par bras :")
print(comparison_df.round(4).to_string(index=False))

# =============================================================================
# 10. RAPPORT
# =============================================================================

report = {
    "approach": "direct_transfer_B_two_arms",
    "panel": "mrna",
    "arms": arm_meta,
    "raw_arm_reproduces_published": bool(ctrl_ok),
    "published_reference": PUBLISHED,
    "n_shared_genes_pool": len(shared_genes),
    "n_genes_selected_for_training": len(selected_genes),
    "n_pca_components": N_COMPONENTS,
    "n_tcga_train": int(len(y_tcga)),
    "n_metabric_test": int(len(y_mb)),
    "classes": list(le_tcga.classes_),
    "n_bootstrap": N_BOOTSTRAP,
    "performance": results_df.to_dict("records"),
    "ranknorm_minus_raw": piv["delta"].to_dict(),
    "comparison_to_internal_cv": comparison_df.to_dict("records"),
    "caveats": [
        "Single model fit on the full TCGA cohort (no cross-validation on the "
        "training side); uncertainty is quantified via class-stratified bootstrap "
        "resampling of the METABRIC test cohort only.",
        "Gene selection (variance filter, top-5000 MAD) was restricted to genes "
        "already shared with METABRIC, to guarantee exact transferability of the "
        "fitted scaler and PCA without imputing missing genes.",
        "CLAUDIN_SUBTYPE is used as the METABRIC PAM50 proxy label, which differs "
        "from a canonical PAM50 centroid assignment.",
        "In the ranknorm arm the rank transformation is fitted independently within "
        "each cohort. It is unsupervised, but it uses the test cohort's own "
        "distribution, so that arm is not a strictly blind transfer.",
    ],
}
atomic_json(report, OUT_DIR / "metabric_direct_transfer_report_two_arms.json")

print(f"\nTermine. Sorties : {OUT_DIR}")


# =============================================================================
# TEXTE A AJOUTER APRES EXECUTION
# =============================================================================
# Methods, fin de la section 2.8 :
#
#   As a sensitivity analysis, the transfer protocol was repeated with expression
#   values rank-normalised within each cohort, gene by gene, before the
#   standardiser and principal components were fitted, placing the two cohorts on
#   a common scale. The transformation is unsupervised but estimated within each
#   cohort, so this arm is not a strictly blind transfer.
#
# Results, section 3.8, apres le paragraphe de transfert direct, AU CHOIX :
#
#   (si la performance remonte)
#   Rank-normalising expression within each cohort before projection raised
#   balanced accuracy to X, Y and Z for Random Forest, XGBoost and SVM
#   respectively, indicating that the collapse observed under unharmonised
#   transfer is largely attributable to the distributional gap between platforms
#   rather than to an absence of transferable signal in the mRNA panel.
#
#   (si elle reste au hasard)
#   Rank-normalising expression within each cohort before projection left
#   balanced accuracy near chance (X, Y and Z), indicating that a scale
#   correction alone does not recover cross-cohort performance.
# =============================================================================

**Output listing.** Lists the files written by the external-validation stage.

In [ ]:
# ==========================================================
# 1. Monter Google Drive
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

# Resolve the result directory from TCGA_BRCA_DATA_DIR.
RESULT_DIR = str(Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data")) / "runs" / "2026-07-26" / "results" / "model_evaluation" / "pid_imin_broja_fixed_models")

**Transfer outputs, repeated run.** Identical to the preceding cell; retained for re-execution convenience.

In [ ]:
"""
===============================================================================
EXTERNAL VALIDATION -- METABRIC
Phase 2bis : Approche B -- transfert direct, DEUX BRAS

  Raw arm: direct transfer using the reported mRNA representation without cross-platform harmonisation. Scaler et PCA ajustes sur des
                    valeurs log2(TPM+1) RNA-seq, appliques sans harmonisation a
                    des intensites microarray.
  Bras "ranknorm" : identique en tout point, sauf que les expressions sont
                    normalisees par rang au sein de chaque cohorte, gene par
                    gene, AVANT l'ajustement du scaler et de la PCA.

The raw arm is a reproducibility control for the direct-transfer result reported in the manuscript and should reproduce the recorded reference values (RF 0.207, XGB 0.254, SVM 0.200) a l'arrondi pres. S'il ne les
reproduit pas, les entrees ou la selection de genes ont change et le bras
"ranknorm" n'est pas interpretable.

This script overwrites no existing file: every output carries the
suffixe _two_arms.
===============================================================================
"""

import os, json, hashlib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

from google.colab import drive
if not os.path.exists(str(DATA_DIR)):
    drive.mount("/content/drive")

DATA_DIR     = Path(os.environ.get("TCGA_BRCA_DATA_DIR", "./TCGA_BRCA_data"))
METABRIC_DIR = DATA_DIR / "brca_metabric_raw"
TCGA_RUN_DIR = DATA_DIR / "runs" / "2026-07-26"
OUT_DIR      = TCGA_RUN_DIR / "results" / "model_evaluation" / "metabric_external_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED           = 42
VALID_SUBTYPES = ["LumA", "LumB", "Her2", "Basal", "Normal"]
MIN_VARIANCE   = 1e-6
FILTER_TOP_K   = 5000
N_COMPONENTS   = 50
N_BOOTSTRAP    = 5000
CLASSIFIERS    = ["RF", "XGB", "SVM"]
ARMS           = ["raw", "ranknorm"]

EVAL_CONFIG = {
    "random_forest": {"n_estimators": 500, "max_features": "sqrt",
                      "min_samples_leaf": 2, "n_jobs": -1},
    "xgboost": {"n_estimators": 500, "max_depth": 4, "learning_rate": 0.03,
                "subsample": 0.85, "colsample_bytree": 0.85, "reg_lambda": 1.0,
                "reg_alpha": 0.0, "min_child_weight": 1.0, "n_jobs": -1},
    "svm": {"C": 1.0, "kernel": "rbf", "gamma": "scale",
            "class_weight": "balanced", "probability": False},
}


def atomic_csv(df, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def atomic_json(obj, path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def stable_int_seed(*parts):
    payload = "||".join(map(str, parts)).encode("utf-8")
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)


def make_classifier(name, seed, n_classes):
    if name == "RF":
        return RandomForestClassifier(random_state=seed, **EVAL_CONFIG["random_forest"])
    if name == "XGB":
        return XGBClassifier(objective="multi:softprob", num_class=n_classes,
                             eval_metric="mlogloss", random_state=seed,
                             verbosity=0, **EVAL_CONFIG["xgboost"])
    if name == "SVM":
        return SVC(random_state=seed, **EVAL_CONFIG["svm"])
    raise ValueError(name)


def score_mad(df_train):
    medians = df_train.median(axis=0)
    return (df_train - medians).abs().median(axis=0)


def rank_normalise(df):
    """Normalisation par rang, colonne par colonne, au sein de la cohorte.

    Les rangs bruts dependent de l'effectif (1..677 pour TCGA, 1..1756 pour
    METABRIC). La division par n+1 rend la transformation independante de la
    taille de la cohorte, ce qui est la condition pour qu'un scaler ajuste sur
    TCGA s'applique legitimement a METABRIC. Les ex aequo recoivent leur rang
    moyen.
    """
    return df.rank(axis=0, method="average") / (len(df) + 1.0)


# =============================================================================
# 1. CHARGEMENT TCGA
# =============================================================================

tcga_cohort = pd.read_csv(TCGA_RUN_DIR / "results" / "cohort_manifest.csv")
tcga_mrna_full = pd.read_parquet(DATA_DIR / "mrna_hugo_mapped_primary_only.parquet")
tcga_mrna_full.index = tcga_mrna_full.index.astype(str)
tcga_mrna_full = tcga_mrna_full.loc[tcga_cohort["patient_id"].astype(str)]

le_tcga = LabelEncoder()
y_tcga = le_tcga.fit_transform(tcga_cohort["pam50"].to_numpy())
print(f"TCGA : {len(y_tcga)} patients, classes = {list(le_tcga.classes_)}")

# =============================================================================
# 2. CHARGEMENT METABRIC
# =============================================================================

clin = pd.read_csv(METABRIC_DIR / "data_clinical_patient.txt", sep="\t",
                   comment="#", low_memory=False)
labels_all = clin.set_index("PATIENT_ID")["CLAUDIN_SUBTYPE"].dropna()
labels_all = labels_all[labels_all.isin(VALID_SUBTYPES)]

mrna_full_mb = pd.read_csv(METABRIC_DIR / "data_mrna_illumina_microarray.txt",
                           sep="\t", low_memory=False)
mrna_full_mb = (mrna_full_mb.drop(columns=["Entrez_Gene_Id"], errors="ignore")
                            .dropna(subset=["Hugo_Symbol"]))
mrna_full_mb = mrna_full_mb.groupby("Hugo_Symbol").mean(numeric_only=True).T
mrna_full_mb.index.name = "PATIENT_ID"

common_mb = sorted(set(labels_all.index) & set(mrna_full_mb.index))
labels_mb = labels_all.loc[common_mb]
mrna_mb = mrna_full_mb.loc[common_mb]

le_mb = LabelEncoder()
y_mb = le_mb.fit_transform(labels_mb.to_numpy())
assert list(le_mb.classes_) == list(le_tcga.classes_), \
    f"Ordre des classes different : TCGA={list(le_tcga.classes_)} vs METABRIC={list(le_mb.classes_)}"
print(f"METABRIC : {len(y_mb)} patients, classes = {list(le_mb.classes_)}")

# =============================================================================
# 3. SELECTION DE GENES -- restreinte aux genes partages des le depart
# =============================================================================

shared_genes = sorted(set(tcga_mrna_full.columns) & set(mrna_mb.columns))
print(f"Genes partages : {len(shared_genes)}")

tcga_shared = tcga_mrna_full[shared_genes]
mb_shared = mrna_mb[shared_genes]

variances = tcga_shared.var(axis=0)
keep = variances[variances > MIN_VARIANCE].index
tcga_selected = tcga_shared[keep]

if tcga_selected.shape[1] > FILTER_TOP_K:
    mad_scores = score_mad(tcga_selected)
    selected_genes = mad_scores.nlargest(FILTER_TOP_K).index.tolist()
else:
    selected_genes = list(tcga_selected.columns)

print(f"Genes selectionnes (variance + top-{FILTER_TOP_K} MAD, on TCGA):  "
      f"{len(selected_genes)}")

# ordre de colonnes identique des deux cotes, impose explicitement
tcga_final = tcga_shared[selected_genes]
mb_final = mb_shared[selected_genes]

# imputation par les medianes TCGA, appliquee aux deux cotes
tcga_medians = tcga_final.median(axis=0)
tcga_final = tcga_final.fillna(tcga_medians)
mb_final = mb_final.fillna(tcga_medians)
print(f"NaN residuels -- TCGA: {tcga_final.isna().sum().sum()}, "
      f"METABRIC: {mb_final.isna().sum().sum()}")

# =============================================================================
# 4-6. BOUCLE SUR LES DEUX BRAS
# =============================================================================

n_classes = len(le_tcga.classes_)
sample_weight_tcga = compute_sample_weight("balanced", y_tcga)
class_indices = {c: np.flatnonzero(y_mb == c) for c in np.unique(y_mb)}

results_rows, per_class_rows, arm_meta = [], [], {}
predictions_mb = {}

for arm in ARMS:
    print(f"\n{'='*70}\nBRAS : {arm}\n{'='*70}")

    if arm == "raw":
        A_tcga, A_mb = tcga_final, mb_final
    else:
        # transformation ajustee INDEPENDAMMENT dans chaque cohorte : c'est le
        # principe meme de la correction d'echelle
        A_tcga, A_mb = rank_normalise(tcga_final), rank_normalise(mb_final)

    scaler = StandardScaler().fit(A_tcga.to_numpy(dtype=np.float64))
    X_tcga = scaler.transform(A_tcga.to_numpy(dtype=np.float64))
    X_mb = scaler.transform(A_mb.to_numpy(dtype=np.float64))

    reducer = PCA(n_components=N_COMPONENTS, random_state=SEED).fit(X_tcga)
    Z_tcga = reducer.transform(X_tcga).astype(np.float32)
    Z_mb = reducer.transform(X_mb).astype(np.float32)
    var_ret = float(reducer.explained_variance_ratio_.sum())
    arm_meta[arm] = {"pca_variance_retained": var_ret}
    print(f"Variance retenue par la PCA : {var_ret*100:.1f}%")

    for clf_name in CLASSIFIERS:
        model = make_classifier(clf_name, SEED, n_classes)
        if clf_name in {"RF", "XGB"}:
            model.fit(Z_tcga, y_tcga, sample_weight=sample_weight_tcga)
        else:
            model.fit(Z_tcga, y_tcga)

        y_pred_mb = model.predict(Z_mb)
        predictions_mb[(arm, clf_name)] = y_pred_mb
        ba_point = float(balanced_accuracy_score(y_mb, y_pred_mb))
        n_pred_classes = int(len(np.unique(y_pred_mb)))

        # bootstrap stratifie par classe sur la cohorte METABRIC uniquement
        rng = np.random.default_rng(stable_int_seed("metabric_boot", arm, clf_name))
        boot = np.empty(N_BOOTSTRAP)
        for b in range(N_BOOTSTRAP):
            idx = np.concatenate([rng.choice(ix, len(ix), replace=True)
                                  for ix in class_indices.values()])
            boot[b] = balanced_accuracy_score(y_mb[idx], y_pred_mb[idx])
        ci_lo, ci_hi = np.quantile(boot, [0.025, 0.975])

        results_rows.append({
            "arm": arm, "classifier": clf_name,
            "metabric_balanced_accuracy": ba_point,
            "ci95_low": float(ci_lo), "ci95_high": float(ci_hi),
            "n_metabric_patients": len(y_mb),
            "n_distinct_predicted_classes": n_pred_classes,
        })
        print(f"{clf_name:<5} BA={ba_point:.4f}  IC95%=[{ci_lo:.4f},{ci_hi:.4f}]"
              f"  classes predites={n_pred_classes}")

        rep = classification_report(y_mb, y_pred_mb,
                                    target_names=le_tcga.classes_,
                                    output_dict=True, zero_division=0)
        for cls in le_tcga.classes_:
            per_class_rows.append({
                "arm": arm, "classifier": clf_name, "class": cls,
                "recall": rep[cls]["recall"], "precision": rep[cls]["precision"],
                "f1": rep[cls]["f1-score"], "support": rep[cls]["support"],
            })

results_df = pd.DataFrame(results_rows)
per_class_df = pd.DataFrame(per_class_rows)
atomic_csv(results_df, OUT_DIR / "metabric_direct_transfer_performance_two_arms.csv")
atomic_csv(per_class_df, OUT_DIR / "metabric_direct_transfer_per_class_two_arms.csv")

# =============================================================================
# 7. CONTROLE DE REPRODUCTION DU BRAS RAW
# =============================================================================

PUBLISHED = {"RF": 0.2065, "XGB": 0.2543, "SVM": 0.2000}
print(f"\n{'='*70}\nCONTROLE : le bras raw reproduit-il les valeurs publiees ?\n{'='*70}")
raw = results_df[results_df.arm == "raw"].set_index("classifier")
ctrl_ok = True
for clf_name, expected in PUBLISHED.items():
    got = float(raw.loc[clf_name, "metabric_balanced_accuracy"])
    delta = abs(got - expected)
    flag = "OK" if delta < 0.005 else "ECART"
    if delta >= 0.005:
        ctrl_ok = False
    print(f"{clf_name:<5} publie={expected:.4f}  obtenu={got:.4f}  "
          f"ecart={delta:.4f}  {flag}")
if not ctrl_ok:
    print("\nATTENTION : le bras raw ne reproduit pas le run publie. Les entrees "
          "or the gene selection differ; the rank-normalised arm is not "
          "interpretable en l'etat.")

# =============================================================================
# 8. EFFET DE LA NORMALISATION PAR RANG
# =============================================================================

piv = results_df.pivot(index="classifier", columns="arm",
                       values="metabric_balanced_accuracy")
piv["delta"] = piv["ranknorm"] - piv["raw"]
print(f"\n{'='*70}\nEFFET DE LA NORMALISATION PAR RANG\n{'='*70}")
print(piv.round(4).to_string())

print("\nRecall par classe, bras ranknorm :")
print(per_class_df[per_class_df.arm == "ranknorm"]
      .pivot(index="class", columns="classifier", values="recall").round(3))

# =============================================================================
# 9. COMPARAISON A LA PERFORMANCE INTERNE TCGA
# =============================================================================

tcga_internal = pd.read_csv(TCGA_RUN_DIR / "results" / "model_evaluation" /
                            "panel_fold_results.csv")
tcga_internal_mrna = (tcga_internal[tcga_internal.panel == "mrna"]
                      .groupby("classifier")["balanced_accuracy"].mean())

comparison_rows = []
for arm in ARMS:
    sub = results_df[results_df.arm == arm].set_index("classifier")
    for clf_name in CLASSIFIERS:
        internal = float(tcga_internal_mrna.get(clf_name, np.nan))
        external = float(sub.loc[clf_name, "metabric_balanced_accuracy"])
        comparison_rows.append({
            "arm": arm, "classifier": clf_name,
            "tcga_internal_cv_ba": internal,
            "metabric_external_ba": external,
            "gap": internal - external,
        })
comparison_df = pd.DataFrame(comparison_rows)
atomic_csv(comparison_df,
           OUT_DIR / "metabric_vs_tcga_internal_comparison_two_arms.csv")
print("\nInterne (TCGA, CV) vs externe (METABRIC), par bras :")
print(comparison_df.round(4).to_string(index=False))

# =============================================================================
# 10. RAPPORT
# =============================================================================

report = {
    "approach": "direct_transfer_B_two_arms",
    "panel": "mrna",
    "arms": arm_meta,
    "raw_arm_reproduces_published": bool(ctrl_ok),
    "published_reference": PUBLISHED,
    "n_shared_genes_pool": len(shared_genes),
    "n_genes_selected_for_training": len(selected_genes),
    "n_pca_components": N_COMPONENTS,
    "n_tcga_train": int(len(y_tcga)),
    "n_metabric_test": int(len(y_mb)),
    "classes": list(le_tcga.classes_),
    "n_bootstrap": N_BOOTSTRAP,
    "performance": results_df.to_dict("records"),
    "ranknorm_minus_raw": piv["delta"].to_dict(),
    "comparison_to_internal_cv": comparison_df.to_dict("records"),
    "caveats": [
        "Single model fit on the full TCGA cohort (no cross-validation on the "
        "training side); uncertainty is quantified via class-stratified bootstrap "
        "resampling of the METABRIC test cohort only.",
        "Gene selection (variance filter, top-5000 MAD) was restricted to genes "
        "already shared with METABRIC, to guarantee exact transferability of the "
        "fitted scaler and PCA without imputing missing genes.",
        "CLAUDIN_SUBTYPE is used as the METABRIC PAM50 proxy label, which differs "
        "from a canonical PAM50 centroid assignment.",
        "In the ranknorm arm the rank transformation is fitted independently within "
        "each cohort. It is unsupervised, but it uses the test cohort's own "
        "distribution, so that arm is not a strictly blind transfer.",
    ],
}
atomic_json(report, OUT_DIR / "metabric_direct_transfer_report_two_arms.json")

print(f"\nTermine. Sorties : {OUT_DIR}")


# =============================================================================
# TEXTE A AJOUTER APRES EXECUTION
# =============================================================================
# Methods, fin de la section 2.8 :
#
#   As a sensitivity analysis, the transfer protocol was repeated with expression
#   values rank-normalised within each cohort, gene by gene, before the
#   standardiser and principal components were fitted, placing the two cohorts on
#   a common scale. The transformation is unsupervised but estimated within each
#   cohort, so this arm is not a strictly blind transfer.
#
# Results, section 3.8, apres le paragraphe de transfert direct, AU CHOIX :
#
#   (si la performance remonte)
#   Rank-normalising expression within each cohort before projection raised
#   balanced accuracy to X, Y and Z for Random Forest, XGBoost and SVM
#   respectively, indicating that the collapse observed under unharmonised
#   transfer is largely attributable to the distributional gap between platforms
#   rather than to an absence of transferable signal in the mRNA panel.
#
#   (si elle reste au hasard)
#   Rank-normalising expression within each cohort before projection left
#   balanced accuracy near chance (X, Y and Z), indicating that a scale
#   correction alone does not recover cross-cohort performance.
# =============================================================================